### Sponsored Search Auction — experimental business extension

This section implements an **offline Sponsored Search / Auction prototype** with
three vendors. It is intentionally separated from organic retrieval and is not
allowed to change the scientific Phase-2/3/4 results.

The technical experiment can be executed and measured now, but it is **not
claimed as a mentor-approved new-problem bonus unless the mentor explicitly
confirms that this matches the intended advertising/Auction requirement**.
After approval, set `MENTOR_APPROVED_AUCTION=1` and rerun the notebook.

The simulation uses transparent proxy CTR/revenue assumptions; these are not
production advertising outcomes.


In [ ]:
# Runtime compatibility guard: this project uses PyTorch, not TensorFlow.
# Set these BEFORE importing sentence-transformers / transformers.
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TORCH"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

from importlib.metadata import version as _pkg_version

def _version_tuple(v):
    nums = []
    for part in str(v).split(".")[:3]:
        m = __import__("re").match(r"(\d+)", part)
        nums.append(int(m.group(1)) if m else 0)
    return tuple(nums + [0] * (3 - len(nums)))

_numpy_version_before_import = _pkg_version("numpy")
if _version_tuple(_numpy_version_before_import) >= (2, 3, 0):
    raise RuntimeError(
        "Incompatible NumPy detected: "
        + _numpy_version_before_import
        + ". This project requires NumPy < 2.3; use NumPy 1.26.4 on Python 3.11. "
        + "Run: python -m pip install --force-reinstall numpy==1.26.4 "
        + "then restart the kernel and Run All again."
    )

# If needed:
# pip install pandas numpy pyarrow scikit-learn plotly hazm persiantools \
#             sentence-transformers torch huggingface-hub scipy joblib requests openai

import os, re, math, json, time, ast, logging, shutil, unicodedata, types
from pathlib import Path
from collections import Counter
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

logging.basicConfig(level=logging.WARNING)

# ---- knobs -------------------------------------------------------------
SAMPLE_SIZE = 100_000        # reproducible sample from the whole comments CSV; None = load all into memory
RUN_MODE = "project"         # official course Metis API; grounded fallback remains enabled
JUDGE_MODE = "project"       # Metis LLM-as-a-Judge for advanced evaluation
RUN_FREE_LLM_SMOKE_TEST = False
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

# ---- paths -------------------------------------------------------------
def _find_raw_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "data" / "raw",
        Path.cwd().parent,
        Path.cwd().parent / "data" / "raw",
    ]
    for d in candidates:
        if (d / "digikala-products.csv").exists() and (d / "digikala-comments.csv").exists():
            return d

    d = Path.cwd() / "data" / "raw"
    d.mkdir(parents=True, exist_ok=True)
    return d


RAW = _find_raw_dir()

# Use a fresh run directory instead of deleting the previous artifact tree.
# This is safer on Windows, where VS Code, antivirus software, or a preview
# window can temporarily lock parquet/json/index files from the previous run.
ARTIFACT_ROOT = Path.cwd() / "nb_artifacts"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = time.strftime("%Y%m%d_%H%M%S") + f"_{os.getpid()}_{time.time_ns() % 1_000_000_000:09d}"
BASE = ARTIFACT_ROOT / f"run_{RUN_ID}"
BASE.mkdir(parents=True, exist_ok=False)

try:
    (ARTIFACT_ROOT / "latest_run.txt").write_text(
        str(BASE.resolve()), encoding="utf-8"
    )
except OSError:
    pass

try:
    import torch
    EMBEDDING_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    EMBEDDING_DEVICE = "cpu"

config = types.SimpleNamespace(
    HF_REPO_ID="RadeAI/Digikala_comments_products",
    HF_REVISION="89c3133b169c8d3793db8834f56f32fee33d9db0",
    PRODUCTS_CSV=RAW / "digikala-products.csv",
    COMMENTS_CSV=RAW / "digikala-comments.csv",
    PRODUCTS_CLEAN=BASE / "products_clean.parquet",
    COMMENTS_CLEAN=BASE / "comments_clean.parquet",
    PHASE1_REPORT=BASE / "phase1_report.json",
    PRODUCT_INDEX_DIR=BASE / "index",
    PROCESSED_DIR=BASE,
    FIGURES_DIR=BASE,
    METRICS_DIR=BASE,
    MODELS_DIR=BASE,
    RAW_DIR=RAW,
    CHUNK_SIZE=200_000,
    COMMENTS_SAMPLE_SIZE=SAMPLE_SIZE,
    RANDOM_SEED=RANDOM_SEED,
    RECOMMENDATION_CLASSES=("recommended", "not_recommended", "no_idea"),
    TOMAN_TO_RIAL=10,
    EMBEDDING_MODEL="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    EMBEDDING_DEVICE=EMBEDDING_DEVICE,
    TOP_K=8,
    RRF_CANDIDATE_POOL=200,
    REVIEW_CANDIDATE_POOL=80,
    PRODUCT_RRF_DENSE_WEIGHT=0.65,
    PRODUCT_RRF_SPARSE_WEIGHT=1.35,
    REVIEW_NEGATIVE_INTENT_WEIGHT=0.32,
    REVIEW_POSITIVE_INTENT_WEIGHT=0.28,
    RUN_MODE=RUN_MODE,
    LOCAL_BACKEND="transformers",
    HF_LLM_MODEL="Qwen/Qwen2.5-1.5B-Instruct",
    OLLAMA_BASE_URL="http://localhost:11434",
    OLLAMA_MODEL="qwen2.5:7b-instruct",
    PROVIDERS={
        "metis": {
            # Official course/project API.
            # Keep the secret outside the notebook as METIS_API_KEY.
            "base_url": os.environ.get(
                "METIS_BASE_URL",
                "https://api.metisai.ir/openai/v1",
            ),
            "model": os.environ.get("METIS_MODEL", "gpt-4o-mini"),
            # Transparent estimate used only for engineering/accounting.
            # Actual Metis credit debit is provider-side and can differ.
            "price_per_m": (
                float(os.environ.get("METIS_INPUT_PRICE_PER_M", "0.15")),
                float(os.environ.get("METIS_OUTPUT_PRICE_PER_M", "0.60")),
            ),
            "price_note": (
                "Estimated token cost using configured gpt-4o-mini rates "
                "($0.15/M input, $0.60/M output by default). "
                "This is an engineering estimate, not a Metis invoice."
            ),
            "billed": True,
            "api_key_env": ("METIS_API_KEY",),
        },
        "openrouter": {
            "base_url": "https://openrouter.ai/api/v1",
            "model": "meta-llama/llama-3.3-70b-instruct:free",
            "api_key_env": "OPENROUTER_API_KEY",
            "billed": False,
        },
        "paid": {
            "base_url": os.environ.get("PAID_BASE_URL", "https://api.openai.com/v1"),
            "model": "gpt-4o-mini",
            "api_key_env": "PAID_API_KEY",
            "price_per_m": (0.15, 0.60),
            "billed": True,
        },
    },
    FREE_PROVIDER="metis",
    PROJECT_PROVIDER="metis",
    HOSTED_PROVIDER_HINT=os.environ.get("COURSE_API_PROVIDER", "").strip().lower(),
    HOSTED_PROVIDER_ORDER=("metis", "paid"),
    LLM_MAX_NEW_TOKENS=768,
    LLM_TEMPERATURE=0.2,
    API_MAX_ATTEMPTS=2,
    API_RETRY_BASE_S=1.5,
    API_CONNECT_TIMEOUT_S=30,
    API_READ_TIMEOUT_S=120,
    PROJECT_CREDIT_LIMIT_USD=5.0,
    API_SAFETY_STOP_USD=4.50,
    MAX_HOSTED_CHAT_ATTEMPTS=40,  # main generation + judge; bounded by $4.50 safety stop
    BUDGET_USD=4.50,
    BUDGET_LOG=BASE / "budget_log.jsonl",
    PAID_PRICE_PER_M=(0.15, 0.60),
    JUDGE_MODE=JUDGE_MODE,
)

config.PRODUCT_INDEX_DIR.mkdir(parents=True, exist_ok=True)
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
config.METRICS_DIR.mkdir(parents=True, exist_ok=True)

print("raw dir:", RAW)
print("artifact dir:", BASE)
print("config ready | SAMPLE_SIZE =", SAMPLE_SIZE, "| RUN_MODE =", RUN_MODE, "| embedding device =", EMBEDDING_DEVICE)
print(
    "runtime | numpy =",
    np.__version__,
    "| TensorFlow disabled for Transformers =",
    os.environ.get("USE_TF") == "0",
)


### Runtime compatibility note

This project uses **PyTorch** for SentenceTransformers and LoRA. TensorFlow is
not part of the project pipeline and is explicitly disabled for Transformers.

The validated Python-3.11 stack uses **NumPy 1.26.4**. The notebook now fails
immediately with a clear message if NumPy 2.3+ is detected, instead of
producing a later `_ARRAY_API` / TensorFlow binary-compatibility traceback.

Optional bonus-package installation is constrained so it cannot silently
upgrade the validated NumPy/SciPy/scikit-learn stack.


## 1 · Phase 1 — Data cleaning & EDA
### 1.1 Persian text utilities

In [ ]:
import re
import unicodedata

# hazm gives a better normalizer, but it's a heavy optional dep; degrade gracefully.
try:
    from hazm import Normalizer as _HazmNormalizer
    _hazm = _HazmNormalizer()
except Exception:                                   # pragma: no cover
    _hazm = None

# Persian/Arabic digits -> ASCII, and unify the Arabic ی/ک variants.
_DIGIT_MAP = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")
_CHAR_MAP = str.maketrans({"ي": "ی", "ى": "ی", "ك": "ک"})

_URL_RE = re.compile(r"https?://\S+|www\.\S+")
_EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F1E6-\U0001F1FF]", re.UNICODE)
_INVISIBLE_RE = re.compile("[​-‏‪-‮⁦-⁩﻿]")
_WS_RE = re.compile(r"\s+")
_TOKEN_RE = re.compile(r"[^\W_]+", re.UNICODE)


def normalize(text: object, *, fold_digits: bool = True, drop_emoji: bool = False) -> str:
    """Normalize a Persian string. Returns '' for NaN/None."""
    if text is None or (isinstance(text, float) and text != text):
        return ""
    s = unicodedata.normalize("NFKC", str(text)).translate(_CHAR_MAP)
    s = _URL_RE.sub(" ", s)
    if drop_emoji:
        s = _EMOJI_RE.sub(" ", s)
    s = _hazm.normalize(s) if _hazm is not None else s
    if fold_digits:
        s = s.translate(_DIGIT_MAP)
    s = _INVISIBLE_RE.sub(" ", s)
    return _WS_RE.sub(" ", s).strip()


def tokenize(text: object) -> list[str]:
    return _TOKEN_RE.findall(normalize(text))


def tokenize_norm(text: object) -> list[str]:
    """Fast tokenizer for text that is ALREADY normalized (skips the hazm pass).
    Use for corpus building where inputs are the stored *_norm columns."""
    if text is None or (isinstance(text, float) and text != text):
        return []
    return _TOKEN_RE.findall(str(text))


def is_meaningful(text: object, min_chars: int = 2) -> bool:
    return len(normalize(text).replace("‌", "").strip()) >= min_chars

# ---- price constraint parser (used by the router) -----------------------

_NUM = r"(?:\d{1,3}(?:[,٬،]\d{3})+|\d+)(?:[./٫]\d+)?"
_UNIT = r"(?:هزار|میلیون|میلیارد|k|K)?"
_CUR = r"(?:تومان|تومن|تومنی|تومانی|ریال)?"

_UPPER_WORDS = (
    "زیر",
    "کمتر",
    "حداکثر",
    "سقف",
    "تا",
    "ارزان‌تر",
    "ارزانتر",
    "پایین‌تر",
    "پایینتر"
)

_LOWER_WORDS = (
    "بالای",
    "بیشتر",
    "بالاتر",
    "حداقل",
    "کف",
    "گران‌تر",
    "گرانتر"
)

_DIRECTIONS = _UPPER_WORDS + _LOWER_WORDS

_PRICE_RE = re.compile(
    rf"(?:(?P<kw>{'|'.join(map(re.escape, _DIRECTIONS))})\s*)?"
    rf"(?P<num>{_NUM})\s*"
    rf"(?P<unit>{_UNIT})\s*"
    rf"(?P<currency>{_CUR})"
)


def _unit_mult(unit):

    return {
        "هزار": 1e3,
        "k": 1e3,
        "K": 1e3,
        "میلیون": 1e6,
        "میلیارد": 1e9
    }.get(unit, 1.0)


def extract_price_constraint(text, toman_to_rial=10):

    t = normalize(text)

    # Hazm may convert English comma to Persian comma.
    # Also tolerate spaces around thousands separators.
    t = re.sub(
        r"(?<=\d)\s*([,٬،])\s*(?=\d)",
        r"\1",
        t
    )

    out = {}

    for m in _PRICE_RE.finditer(t):

        kw = (m.group("kw") or "").strip()

        if not kw:
            continue

        num_text = m.group("num")

        num_text = (
            num_text
            .replace(",", "")
            .replace("٬", "")
            .replace("،", "")
            .replace("٫", ".")
            .replace("/", ".")
        )

        val = float(num_text)

        val *= _unit_mult(m.group("unit"))

        currency = (m.group("currency") or "").strip()

        if currency != "ریال":
            val *= toman_to_rial

        if kw in _UPPER_WORDS:

            out["price_max"] = min(
                out.get("price_max", val),
                val
            )

        elif kw in _LOWER_WORDS:

            out["price_min"] = max(
                out.get("price_min", val),
                val
            )

    return out


price_parser_checks = [
    ("تا 1000000 ریال", "price_max", 1_000_000),
    ("زیر 500 هزار تومان", "price_max", 5_000_000),
    ("بالای 2 میلیون", "price_min", 20_000_000),
    ("زیر 500,000 تومان", "price_max", 5_000_000),
    ("زیر ۵۰۰٬۰۰۰ تومان", "price_max", 5_000_000),
    ("زیر ۵۰۰،۰۰۰ تومان", "price_max", 5_000_000)
]


for query, key, expected in price_parser_checks:

    result = extract_price_constraint(query)
    actual = result.get(key)

    if actual != expected:

        raise RuntimeError(
            f"price parser check failed: "
            f"{query!r} -> {actual}, expected {expected}"
        )


print("price parser checks: OK")


def format_toman(value: object) -> str:
    """Rials -> a human 'Toman' string, or 'نامشخص' when missing."""
    if value is None or (isinstance(value, float) and value != value):
        return "نامشخص"
    try:
        return f"{int(float(value)) // 10:,}"
    except (TypeError, ValueError):
        return "نامشخص"

pt = types.SimpleNamespace(normalize=normalize, tokenize=tokenize, tokenize_norm=tokenize_norm,
    is_meaningful=is_meaningful, extract_price_constraint=extract_price_constraint,
    format_toman=format_toman, _DIGIT_MAP=_DIGIT_MAP)

### 1.2 Data IO — download + streaming/sampling

In [ ]:
import logging
import os
import shutil

import numpy as np
import pandas as pd


log = logging.getLogger("digikala.dataio")

# The HF "Xet" transfer backend can stall on the large comments file.
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")


def download_raw() -> None:
    """Fetch both CSVs from the pinned HF revision if they are missing."""
    from huggingface_hub import hf_hub_download

    targets = {
        "digikala-products.csv": config.PRODUCTS_CSV,
        "digikala-comments.csv": config.COMMENTS_CSV,
    }

    for filename, dest in targets.items():
        dest.parent.mkdir(parents=True, exist_ok=True)

        if dest.exists() and dest.stat().st_size > 0:
            log.info("%s already present, skipping", dest.name)
            continue

        log.info("downloading %s @ %s", filename, config.HF_REVISION[:8])
        cached = hf_hub_download(
            repo_id=config.HF_REPO_ID,
            repo_type="dataset",
            filename=filename,
            revision=config.HF_REVISION,
        )
        shutil.copy2(cached, dest)
        log.info("saved %s (%.0f MB)", dest.name, dest.stat().st_size / 1024**2)


def load_products() -> pd.DataFrame:
    return pd.read_csv(config.PRODUCTS_CSV, low_memory=False)


def iter_comment_chunks(chunksize: int | None = None):
    chunksize = chunksize or config.CHUNK_SIZE
    yield from pd.read_csv(config.COMMENTS_CSV, chunksize=chunksize, low_memory=False)


def priority_sample(path, k: int, seed: int) -> pd.DataFrame:
    """Exact uniform k-row sample in one pass using random priorities.

    Each row gets an i.i.d. random key and we keep the k smallest keys seen so far.
    This avoids the bias of reading the first k rows and keeps memory bounded by
    roughly one CSV chunk plus the sample.
    """
    if k <= 0:
        raise ValueError("sample size must be positive")

    rng = np.random.default_rng(seed)
    best = None
    source_rows_scanned = 0

    for chunk in pd.read_csv(path, chunksize=config.CHUNK_SIZE, low_memory=False):
        source_rows_scanned += len(chunk)
        part = chunk.copy()
        part["_sample_priority"] = rng.random(len(part))

        if best is None:
            best = part.nsmallest(min(k, len(part)), "_sample_priority")
        else:
            best = pd.concat([best, part], ignore_index=True)
            best = best.nsmallest(min(k, len(best)), "_sample_priority")

    if best is None:
        return pd.DataFrame()

    result = (
        best.drop(columns="_sample_priority")
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )
    result.attrs["source_rows_scanned"] = int(source_rows_scanned)
    return result


# compatibility name used elsewhere
reservoir_sample = priority_sample


def load_comments(sample_size: int | None = "default") -> pd.DataFrame:
    """Read a reproducible notebook sample; `None` truly means all rows."""
    if sample_size == "default":
        sample_size = config.COMMENTS_SAMPLE_SIZE

    if sample_size is None:
        log.warning("Loading the full comments CSV into memory; this can require substantial RAM.")
        result = pd.read_csv(config.COMMENTS_CSV, low_memory=False)
        result.attrs["source_rows_scanned"] = int(len(result))
        return result

    return priority_sample(config.COMMENTS_CSV, int(sample_size), config.RANDOM_SEED)


dataio = types.SimpleNamespace(
    download_raw=download_raw,
    load_products=load_products,
    iter_comment_chunks=iter_comment_chunks,
    load_comments=load_comments,
    reservoir_sample=reservoir_sample,
    priority_sample=priority_sample,
)


### 1.3 Cleaning — documented `_norm`/`_clean` schema

In [ ]:
import ast
import json
import logging
import re

import numpy as np
import pandas as pd
try:
    from persiantools.jdatetime import JalaliDate
except Exception:
    JalaliDate = None


log = logging.getLogger("digikala.phase1")

_JALALI_MONTHS = {
    "فروردین": 1, "اردیبهشت": 2, "خرداد": 3, "تیر": 4, "مرداد": 5, "شهریور": 6,
    "مهر": 7, "آبان": 8, "آذر": 9, "دی": 10, "بهمن": 11, "اسفند": 12,
}
_TRUE = {"1", "true", "yes", "بله", "t", "y"}
_FALSE = {"0", "false", "no", "خیر", "f", "n"}
_GENERIC = "نامشخص"


# ---- small typed converters --------------------------------------------
def _to_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")


def _to_bool(s: pd.Series) -> pd.Series:
    def conv(v):
        if pd.isna(v):
            return pd.NA
        t = str(v).strip().lower()
        return True if t in _TRUE else False if t in _FALSE else pd.NA
    return s.map(conv).astype("boolean")


def _parse_list_field(v) -> str:
    """advantages/disadvantages arrive as list-literal strings, e.g.
    "['جنسش خوبه\\r', 'خوش رنگه']" — pull the items out and join them."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    s = str(v).replace("\\r", " ").replace("\r", " ")
    try:
        parsed = ast.literal_eval(s)
        items = [str(x) for x in parsed] if isinstance(parsed, (list, tuple)) else [str(parsed)]
    except Exception:
        items = re.split(r"',\s*'", s.strip("[]"))
    items = [pt.normalize(it.strip(" '\"")) for it in items]
    return " ، ".join(it for it in items if it)


def _dedup(df: pd.DataFrame, report: dict, id_col: str) -> pd.DataFrame:
    n = len(df)
    df = df.drop_duplicates()
    report["dropped_exact_duplicates"] = n - len(df)
    if id_col in df.columns:
        n = len(df)
        df = df.drop_duplicates(subset=id_col, keep="first")
        report["dropped_duplicate_ids"] = n - len(df)
    return df


# ---- products -----------------------------------------------------------
def clean_products(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    report: dict = {"input_rows": len(df)}
    df = _dedup(df.copy(), report, id_col="id")
    out = pd.DataFrame()

    out["product_id"] = _to_num(df.get("id")).astype("Int64")
    out["title_fa"] = df.get("title_fa", "").map(pt.normalize)
    out["title_norm"] = out["title_fa"]

    for src, dst in [("Brand", "brand_norm"), ("Category1", "category1_norm"),
                     ("Category2", "category2_norm"), ("sub_category", "sub_category_norm"),
                     ("Seller", "seller_norm")]:
        out[dst] = df.get(src, pd.Series(index=df.index)).fillna(_GENERIC).map(pt.normalize)

    out["price_clean"] = _to_num(df.get("Price"))
    out["min_price_last_month"] = _to_num(df.get("min_price_last_month"))
    out["product_rate_clean"] = _to_num(df.get("Rate"))          # 0..100 scale in this dataset
    out["rate_count"] = _to_num(df.get("Rate_cnt")).fillna(0).astype("Int64")
    out["is_fake"] = _to_bool(df.get("Is_Fake", pd.Series(index=df.index)))

    # price 0 / missing means "not for sale", not a real price — flag it, keep the row
    report["zero_or_missing_price"] = int((out["price_clean"].isna() | (out["price_clean"] == 0)).sum())
    out["price_available"] = out["price_clean"].notna() & (out["price_clean"] > 0)

    # embedding text: title + categories + brand, skipping the generic placeholder
    def _ptext(r):
        parts = [r["title_norm"], r["category1_norm"], r["category2_norm"],
                 r["sub_category_norm"], r["brand_norm"]]
        return " ".join(p for p in parts if p and p != _GENERIC)
    out["product_text_norm"] = out.apply(_ptext, axis=1)

    out = out.dropna(subset=["product_id"]).reset_index(drop=True)
    report["output_rows"] = len(out)
    return out, report


# ---- comments -----------------------------------------------------------
def clean_comments(df: pd.DataFrame, valid_product_ids: set | None = None) -> tuple[pd.DataFrame, dict]:
    """Clean one comments dataframe (a chunk or the whole notebook sample)."""
    report: dict = {"input_rows": len(df)}
    df = _dedup(df.copy(), report, id_col="id")
    out = pd.DataFrame()

    out["comment_id"] = _to_num(df.get("id")).astype("Int64")
    out["product_id"] = _to_num(df.get("product_id")).astype("Int64")
    out["title_norm"] = df.get("title", "").map(pt.normalize)
    out["body_norm"] = df.get("body", "").map(pt.normalize)
    out["advantages_norm"] = df.get("advantages", pd.Series(index=df.index)).map(_parse_list_field)
    out["disadvantages_norm"] = df.get("disadvantages", pd.Series(index=df.index)).map(_parse_list_field)

    # combined text used by retrieval and the classifier
    out["comment_text_norm"] = (
        out[["title_norm", "body_norm", "advantages_norm", "disadvantages_norm"]]
        .agg(" ".join, axis=1).map(pt.normalize))
    out["has_text"] = out["comment_text_norm"].map(pt.is_meaningful)

    out["rate_clean"] = _to_num(df.get("rate"))
    out["likes"] = _to_num(df.get("likes")).fillna(0).astype("Int64")
    out["dislikes"] = _to_num(df.get("dislikes")).fillna(0).astype("Int64")
    out["is_buyer"] = _to_bool(df.get("is_buyer", pd.Series(index=df.index)))
    out["true_to_size_rate"] = _to_num(df.get("true_to_size_rate"))

    if "created_at" in df.columns:
        out["created_at_norm"] = df["created_at"].map(pt.normalize)
        date_parts = out["created_at_norm"].map(_jalali_parts)
        out["created_year_jalali"] = date_parts.map(
            lambda x: x[0] if x is not None else pd.NA
        ).astype("Int64")
        out["date_format_valid"] = date_parts.notna()
        out["created_at"] = _parse_datetime(df["created_at"])

    # recommendation label: validate against the 3 allowed classes
    rs = df.get("recommendation_status", pd.Series(index=df.index)).astype("string").str.strip().str.lower()
    out["recommendation_status"] = rs.where(rs.isin(config.RECOMMENDATION_CLASSES))
    out["recommendation_valid"] = out["recommendation_status"].isin(config.RECOMMENDATION_CLASSES)

    # does the comment point at a real product? (needed for RAG grounding + joins)
    if valid_product_ids is not None:
        out["has_product_match"] = out["product_id"].isin(valid_product_ids)
    else:
        out["has_product_match"] = out["product_id"].notna()

    report["rows_without_text"] = int((~out["has_text"]).sum())
    report["invalid_recommendation_labels"] = int((~out["recommendation_valid"]).sum())
    report["rows_without_product_match"] = int((~out["has_product_match"]).sum())
    out = out.dropna(subset=["comment_id"]).reset_index(drop=True)
    report["output_rows"] = len(out)
    return out, report


def _jalali_parts(v):
    """Return (year, month, day) for a valid Jalali date string, without requiring persiantools."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None

    key = pt.normalize(v)
    parts = key.split()

    if len(parts) != 3 or parts[1] not in _JALALI_MONTHS:
        return None

    try:
        day = int(parts[0])
        year = int(parts[2])
        month = _JALALI_MONTHS[parts[1]]
    except (TypeError, ValueError):
        return None

    if not (1 <= day <= 31 and 1300 <= year <= 1600):
        return None

    return year, month, day


def _parse_datetime(s: pd.Series) -> pd.Series:
    """created_at is a Jalali string like "23 شهریور 1402"; convert to Gregorian.
    Cached per distinct string since there are relatively few unique dates."""
    cache: dict[str, pd.Timestamp] = {}

    def parse_one(v):
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return pd.NaT
        key = str(v).strip()
        if key in cache:
            return cache[key]
        result = pd.NaT
        parts = _jalali_parts(key)

        if JalaliDate is not None and parts is not None:
            try:
                year, month, day = parts
                result = pd.Timestamp(JalaliDate(year, month, day).to_gregorian())
            except Exception:
                result = pd.NaT
        cache[key] = result
        return result
    return s.map(parse_one)


# ---- driver -------------------------------------------------------------
def _merge_reports(total: dict, chunk: dict) -> None:
    for k, v in chunk.items():
        if isinstance(v, (int, float)):
            total[k] = total.get(k, 0) + v


def build(full: bool = True) -> dict:
    """Clean products (in memory) then stream-clean comments to Parquet.

    full=True streams the entire comments CSV chunk-by-chunk (memory-safe full-data mode).
    full=False cleans only the notebook sample from config.COMMENTS_SAMPLE_SIZE.
    """
    import pyarrow as pa
    import pyarrow.parquet as pq

    dataio.download_raw()

    log.info("cleaning products")
    products, p_rep = clean_products(dataio.load_products())
    products.to_parquet(config.PRODUCTS_CLEAN, index=False)
    valid_ids = set(int(x) for x in products["product_id"].dropna())
    log.info("saved %s (%d rows)", config.PRODUCTS_CLEAN.name, len(products))

    c_total: dict = {}
    writer = None
    if full:
        log.info("streaming + cleaning comments in chunks of %d", config.CHUNK_SIZE)
        for i, chunk in enumerate(dataio.iter_comment_chunks()):
            cleaned, rep = clean_comments(chunk, valid_ids)
            _merge_reports(c_total, rep)
            table = pa.Table.from_pandas(cleaned, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(config.COMMENTS_CLEAN, table.schema)
            writer.write_table(table)
            if i % 5 == 0:
                log.info("  chunk %d done (%d rows cumulative)", i, c_total.get("output_rows", 0))
        if writer is not None:
            writer.close()
    else:
        comments, c_total = clean_comments(dataio.load_comments(), valid_ids)
        comments.to_parquet(config.COMMENTS_CLEAN, index=False)
    log.info("saved %s (%d rows)", config.COMMENTS_CLEAN.name, c_total.get("output_rows", 0))

    report = {"products": p_rep, "comments": c_total, "full": full}
    config.PHASE1_REPORT.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    return report

### 1.4 Run Phase 1

For the notebook we take a **uniform, seed-fixed sample from the full comments file**, then keep only products referenced by those comments. This makes the later QA, comparison and managerial analyses fully joined while remaining practical on a laptop.

`SAMPLE_SIZE=None` now genuinely means loading all comments into memory. For full cleaning without loading all comments at once, the `build(full=True)` function above streams the comments CSV chunk by chunk.


In [ ]:
# 1) reproducible random sample across the entire comments CSV
if not config.COMMENTS_CSV.exists() or not config.PRODUCTS_CSV.exists():
    dataio.download_raw()

com_raw = dataio.load_comments(SAMPLE_SIZE)
raw_comment_rows_scanned = int(com_raw.attrs.get("source_rows_scanned", len(com_raw)))

if "product_id" not in com_raw.columns:
    raise KeyError("The comments CSV must contain a 'product_id' column.")

wanted = set(
    pd.to_numeric(com_raw["product_id"], errors="coerce")
    .dropna()
    .astype(int)
)

# 2) load only referenced products, chunked to avoid holding the full catalogue twice
keep = []
raw_product_rows_scanned = 0

for ch in pd.read_csv(config.PRODUCTS_CSV, chunksize=200_000, low_memory=False):
    raw_product_rows_scanned += len(ch)
    ids = pd.to_numeric(ch["id"], errors="coerce")
    part = ch[ids.isin(wanted)]

    if len(part):
        keep.append(part)

if not keep:
    raise RuntimeError("No matching products were found for the sampled comments.")

prod_raw = pd.concat(keep, ignore_index=True)

products_df, p_rep = clean_products(prod_raw)
valid_ids = set(int(x) for x in products_df["product_id"].dropna())
comments_df, c_rep = clean_comments(com_raw, valid_ids)

# Persist the exact cleaned notebook data so later phases and a fresh helper call
# use the same rows and schema.
products_df.to_parquet(config.PRODUCTS_CLEAN, index=False)
comments_df.to_parquet(config.COMMENTS_CLEAN, index=False)

phase1_report = {
    "sampling": {
        "method": "uniform_random_priority_sample" if SAMPLE_SIZE is not None else "all_rows",
        "sample_size_requested": SAMPLE_SIZE,
        "random_seed": config.RANDOM_SEED,
        "raw_comment_rows_scanned": raw_comment_rows_scanned,
        "raw_product_rows_scanned": int(raw_product_rows_scanned),
        "sampled_comments": int(len(com_raw)),
        "matched_products": int(len(products_df)),
        "jalali_to_gregorian_available": bool(JalaliDate is not None),
    },
    "products": p_rep,
    "comments": c_rep,
}

config.PHASE1_REPORT.write_text(
    json.dumps(phase1_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("products:", len(products_df), "| comments:", len(comments_df))
print("sampling:", phase1_report["sampling"])
print("product report:", p_rep)
print("comment report:", c_rep)


### 1.5 EDA (Plotly)

In [ ]:
import json
import logging

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


log = logging.getLogger("digikala.eda")
_TEMPLATE = "plotly_white"


# ---- headline numbers ---------------------------------------------------
def summary_stats(products: pd.DataFrame, comments: pd.DataFrame) -> dict:
    return {
        "n_products": int(len(products)),
        "n_comments": int(len(comments)),
        "n_brands": int(products["brand_norm"].nunique()),
        "n_categories": int(products["category1_norm"].nunique()),
        "pct_products_priced": round(100 * products["price_available"].mean(), 1),
        "pct_comments_with_text": round(100 * comments["has_text"].mean(), 1),
        "pct_comments_labeled": round(100 * comments["recommendation_valid"].mean(), 1),
        "median_price_toman": (None if products["price_clean"].dropna().empty
                               else int(products.loc[products["price_available"], "price_clean"].median() // 10)),
        "avg_comments_per_product": round(
            len(comments) / max(1, comments["product_id"].nunique()), 1),
    }


# ---- figures ------------------------------------------------------------
def fig_recommendation_balance(comments: pd.DataFrame) -> go.Figure:
    vc = comments.loc[comments["recommendation_valid"], "recommendation_status"].value_counts()
    fig = px.bar(x=vc.index, y=vc.values, template=_TEMPLATE,
                 labels={"x": "recommendation_status", "y": "count"},
                 title="Phase 3 target — recommendation_status balance", text=vc.values)
    fig.update_traces(marker_color=["#2ca02c", "#d62728", "#7f7f7f"][:len(vc)])
    return fig


def fig_top_categories(products: pd.DataFrame, n: int = 15) -> go.Figure:
    vc = products["category1_norm"].value_counts().head(n)[::-1]
    return px.bar(x=vc.values, y=vc.index, orientation="h", template=_TEMPLATE,
                  labels={"x": "products", "y": "category"},
                  title=f"Top {n} level-1 categories")


def fig_top_brands(products: pd.DataFrame, n: int = 15) -> go.Figure:
    vc = products.loc[products["brand_norm"] != "نامشخص", "brand_norm"].value_counts().head(n)[::-1]
    return px.bar(x=vc.values, y=vc.index, orientation="h", template=_TEMPLATE,
                  labels={"x": "products", "y": "brand"}, title=f"Top {n} brands")


def fig_price_distribution(products: pd.DataFrame) -> go.Figure:
    priced = products.loc[products["price_available"], "price_clean"] / 10  # Rials -> Toman
    priced = priced[priced > 0]
    fig = px.histogram(np.log10(priced), nbins=60, template=_TEMPLATE,
                       title="Price distribution (log10 Toman)",
                       labels={"value": "log10(price, Toman)"})
    fig.update_layout(showlegend=False)
    return fig


def fig_rating_distribution(products: pd.DataFrame) -> go.Figure:
    rate = products["product_rate_clean"].dropna()
    return px.histogram(rate, nbins=50, template=_TEMPLATE,
                        title="Product rating distribution (0–100)",
                        labels={"value": "product_rate_clean"})


def fig_comments_per_product(comments: pd.DataFrame) -> go.Figure:
    counts = comments.groupby("product_id").size()
    counts = counts[counts > 0]
    fig = px.histogram(np.log10(counts), nbins=50, template=_TEMPLATE,
                       title="Reviews per product (log10)",
                       labels={"value": "log10(reviews per product)"})
    fig.update_layout(showlegend=False)
    return fig


def fig_text_length(comments: pd.DataFrame) -> go.Figure:
    lens = comments.loc[comments["has_text"], "comment_text_norm"].str.split().map(len)
    lens = lens[lens <= lens.quantile(0.99)]        # trim the long tail for readability
    return px.histogram(lens, nbins=60, template=_TEMPLATE,
                        title="Review length (words, 99th pct clipped)",
                        labels={"value": "words per review"})


def fig_missingness(products: pd.DataFrame, comments: pd.DataFrame) -> go.Figure:
    rows = []
    for name, df in [("products", products), ("comments", comments)]:
        for col in df.columns:
            rows.append({"table": name, "column": col,
                         "missing_%": round(100 * df[col].isna().mean(), 1)})
    m = pd.DataFrame(rows)
    m = m[m["missing_%"] > 0].sort_values("missing_%")
    return px.bar(m, x="missing_%", y="column", color="table", orientation="h",
                  template=_TEMPLATE, title="Missing values by column")


ALL_FIGURES = {
    "recommendation_balance": fig_recommendation_balance,
    "top_categories": fig_top_categories,
    "top_brands": fig_top_brands,
    "price_distribution": fig_price_distribution,
    "rating_distribution": fig_rating_distribution,
    "comments_per_product": fig_comments_per_product,
    "text_length": fig_text_length,
    "missingness": fig_missingness,
}


def run() -> dict:
    """Load the cleaned tables, build every figure, and write them to artifacts."""
    products = pd.read_parquet(config.PRODUCTS_CLEAN)
    comments = pd.read_parquet(config.COMMENTS_CLEAN)
    stats = summary_stats(products, comments)

    for name, fn in ALL_FIGURES.items():
        arg = (products,) if fn in (fig_top_categories, fig_top_brands,
                                    fig_price_distribution, fig_rating_distribution) else \
              (comments,) if fn in (fig_recommendation_balance, fig_comments_per_product,
                                    fig_text_length) else (products, comments)
        try:
            fig = fn(*arg)
            fig.write_html(config.FIGURES_DIR / f"{name}.html", include_plotlyjs="cdn")
        except Exception as e:
            log.warning("figure %s failed: %s", name, e)

    (config.FIGURES_DIR / "eda_summary.json").write_text(
        json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")
    log.info("EDA figures + summary written to %s", config.FIGURES_DIR)
    return stats

In [ ]:
summary_stats(products_df, comments_df)

In [ ]:
fig_recommendation_balance(comments_df).show()
fig_price_distribution(products_df).show()
fig_top_categories(products_df).show()

## 2 · Phase 2 — Grounded shopping assistant
### 2.A Shared LLM wrapper + API budget tracking

The following cells are shared infrastructure. The four **official Phase-2 capabilities (2.1–2.4)** are demonstrated together below after the assistant is built.


In [ ]:
import json
import logging
import time


log = logging.getLogger("digikala.llm")


class BudgetTracker:
    """Track hosted API attempts, successful calls, tokens and budget usage."""

    def __init__(self, cap_usd: float = config.BUDGET_USD, log_path=config.BUDGET_LOG):
        self.cap = cap_usd
        self.log_path = log_path
        self.attempted_calls = 0
        self.successful_calls = 0
        self.failed_calls = 0
        self.in_tokens = 0
        self.out_tokens = 0
        self.spent = 0.0
        self.estimated_list_cost = 0.0

    @property
    def calls(self):
        return self.successful_calls

    def can_spend(self) -> bool:
        return self.spent < self.cap

    def _write(self, row):
        try:
            with open(self.log_path, "a", encoding="utf-8") as fh:
                fh.write(json.dumps(row, ensure_ascii=False) + "\n")
        except OSError:
            pass

    def record_attempt(self, provider: str, model: str):
        self.attempted_calls += 1
        self._write({
            "t": time.time(),
            "event": "attempt",
            "provider": provider,
            "model": model,
        })

    def record_failure(self, provider: str, model: str, status=None, error=""):
        self.failed_calls += 1
        self._write({
            "t": time.time(),
            "event": "failure",
            "provider": provider,
            "model": model,
            "status": status,
            "error": str(error)[:300],
        })

    def record_success(
        self,
        in_tok: int,
        out_tok: int,
        *,
        provider: str,
        model: str,
        price_per_m=(0.0, 0.0),
        billed: bool = False,
    ):
        self.successful_calls += 1
        self.in_tokens += int(in_tok or 0)
        self.out_tokens += int(out_tok or 0)

        estimated = (
            (int(in_tok or 0) * float(price_per_m[0]))
            + (int(out_tok or 0) * float(price_per_m[1]))
        ) / 1e6

        actual = estimated if billed else 0.0
        self.estimated_list_cost += estimated
        self.spent += actual

        self._write({
            "t": time.time(),
            "event": "success",
            "provider": provider,
            "model": model,
            "in": int(in_tok or 0),
            "out": int(out_tok or 0),
            "estimated_list_cost": round(estimated, 6),
            "tracked_cost": round(actual, 6),
            "billed": bool(billed),
        })

        return actual

    def summary(self) -> dict:
        resolved = self.successful_calls + self.failed_calls
        return {
            "api_attempts": self.attempted_calls,
            "successful_calls": self.successful_calls,
            "failed_calls": self.failed_calls,
            "resolved_attempts": resolved,
            "unresolved_attempts": max(0, self.attempted_calls - resolved),
            "input_tokens": self.in_tokens,
            "output_tokens": self.out_tokens,
            "total_cost_usd": round(self.spent, 6),
            "estimated_list_cost_usd": round(self.estimated_list_cost, 6),
            "operational_safety_cap_usd": self.cap,
            "project_credit_limit_usd": float(
                getattr(config, "PROJECT_CREDIT_LIMIT_USD", self.cap)
            ),
            "remaining_safety_budget_usd": round(self.cap - self.spent, 6),
        }


class LLM:
    """Backend-agnostic chat model. Resolves its provider from mode + config."""

    def __init__(
        self,
        mode: str | None = None,
        budget: BudgetTracker | None = None,
        temperature: float = config.LLM_TEMPERATURE,
        max_tokens: int = config.LLM_MAX_NEW_TOKENS,
    ):
        self.mode = mode or config.RUN_MODE
        self.budget = budget or BudgetTracker()
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.last_cost_usd = 0.0
        self._cache: dict = {}
        self.cache_hits = 0
        self._hf = None
        self.provider = self._resolve_provider()
        self.hosted_disabled_reason = None
        self.preferred_network_path = None

    def _provider_with_key(self, provider_name: str) -> dict:
        import os

        p = dict(config.PROVIDERS[provider_name])
        p["provider_name"] = provider_name

        key_env = p.get("api_key_env")
        key_names = (
            (key_env,)
            if isinstance(key_env, str)
            else tuple(key_env or ())
        )

        p["api_key"] = ""
        p["api_key_source"] = None
        p["recognized_key_envs"] = list(key_names)

        for env_name in key_names:
            value = os.environ.get(env_name, "")

            if value:
                p["api_key"] = value
                p["api_key_source"] = env_name
                break

        return p

    def _resolve_provider(self) -> dict | None:
        if self.mode == "hosted_auto":
            order = list(config.HOSTED_PROVIDER_ORDER)
            hint = str(config.HOSTED_PROVIDER_HINT or "").strip().lower()

            if hint in config.PROVIDERS:
                order = [hint] + [name for name in order if name != hint]

            checked = []

            for provider_name in order:
                p = self._provider_with_key(provider_name)
                checked.extend(p.get("recognized_key_envs", []))

                if p.get("api_key"):
                    p["auto_selected"] = True
                    p["recognized_key_envs_all"] = list(dict.fromkeys(checked))
                    return p

            # Preserve a diagnostic object even when no key is found.
            return {
                "provider_name": None,
                "model": None,
                "base_url": None,
                "api_key": "",
                "api_key_source": None,
                "auto_selected": False,
                "recognized_key_envs_all": list(dict.fromkeys(checked)),
                "price_per_m": (0.0, 0.0),
                "price_note": None,
                "billed": False,
            }

        if self.mode == "project":
            p = self._provider_with_key(config.PROJECT_PROVIDER)
            p["paid"] = True
            return p

        if self.mode == "free":
            p = self._provider_with_key(config.FREE_PROVIDER)
            p["paid"] = False
            return p

        if self.mode == "paid":
            p = self._provider_with_key("paid")
            p["paid"] = True
            return p

        return None

    @property
    def backend(self) -> str:
        if self.mode == "local":
            return config.LOCAL_BACKEND
        if self.mode in ("hosted_auto", "project", "free", "paid") and self.provider:
            return f"{self.mode}:{self.provider['model']}"
        return self.mode

    def available(self) -> bool:
        if self.mode == "extractive":
            return False
        if self.mode in ("hosted_auto", "project", "free", "paid"):
            return bool(self.provider and self.provider.get("api_key"))
        return True

    def _apply_model_price(self, model: str):
        """Metis uses the fixed configured engineering token-rate estimate."""
        return

    def _diagnostic_chat_probe(
        self,
        *,
        model: str,
        network_path: str,
        trust_env: bool,
    ) -> dict:
        """Make one tiny real Chat Completions call and account for it."""
        import requests

        p = self.provider or {}
        provider_name = str(p.get("provider_name") or "")

        payload = {
            "model": model,
            "temperature": 0,
            "messages": [
                {
                    "role": "user",
                    "content": "Reply with exactly: OK",
                }
            ],
        }

        if provider_name == "metis":
            payload["max_tokens"] = 32
        else:
            payload["max_completion_tokens"] = 32


        session = requests.Session()
        session.trust_env = trust_env

        self.budget.record_attempt(provider_name, model)

        try:
            r = session.post(
                f"{p['base_url']}/chat/completions",
                timeout=(
                    config.API_CONNECT_TIMEOUT_S,
                    config.API_READ_TIMEOUT_S,
                ),
                headers={
                    "Authorization": f"Bearer {p['api_key']}",
                    "Content-Type": "application/json",
                    "Connection": "close",
                },
                json=payload,
            )
        except requests.RequestException as e:
            self.budget.record_failure(
                provider_name,
                model,
                status=None,
                error=f"diagnostic {network_path}: {str(e)[:260]}",
            )
            return {
                "status": None,
                "success": False,
                "model": model,
                "network_path": network_path,
                "error": str(e)[:300],
                "input_tokens": 0,
                "output_tokens": 0,
            }
        finally:
            try:
                session.close()
            except Exception:
                pass

        if not r.ok:
            self.budget.record_failure(
                provider_name,
                model,
                status=int(r.status_code),
                error=f"diagnostic {network_path}: {r.text[:260]}",
            )
            return {
                "status": int(r.status_code),
                "success": False,
                "model": model,
                "network_path": network_path,
                "error": r.text[:300],
                "input_tokens": 0,
                "output_tokens": 0,
            }

        data = r.json()
        usage = data.get("usage", {}) or {}
        in_tok = int(usage.get("prompt_tokens", 0) or 0)
        out_tok = int(usage.get("completion_tokens", 0) or 0)

        # If the probe switches model, update the configured price before
        # recording the successful request.
        self.provider["model"] = model
        self._apply_model_price(model)

        self.budget.record_success(
            in_tok,
            out_tok,
            provider=provider_name,
            model=model,
            price_per_m=self.provider.get(
                "price_per_m",
                config.PAID_PRICE_PER_M,
            ),
            billed=bool(
                self.provider.get(
                    "billed",
                    self.provider.get("paid", False),
                )
            ),
        )

        content = ""
        try:
            content = str(
                data["choices"][0]["message"].get("content", "") or ""
            ).strip()
        except Exception:
            pass

        return {
            "status": int(r.status_code),
            "success": True,
            "model": model,
            "network_path": network_path,
            "error": None,
            "input_tokens": in_tok,
            "output_tokens": out_tok,
            "content_preview": content[:80],
        }

    def diagnose(self) -> dict:
        """Check the hosted provider without exposing the API key.

        `/models` is useful but is not the final authority for this project:
        some organization/project permission setups may reject model listing
        while Chat Completions are still the operation we actually need.
        Therefore, if model listing does not establish readiness, a bounded
        tiny Chat Completions probe is attempted and fully counted in API
        requests/tokens/cost.
        """
        if self.mode not in ("hosted_auto", "project", "free", "paid"):
            return {
                "mode": self.mode,
                "hosted": False,
                "available": self.available(),
                "request_attempts": 0,
                "chat_probe_attempts": 0,
            }

        p = self.provider or {}
        result = {
            "mode": self.mode,
            "provider": p.get("provider_name"),
            "model": p.get("model"),
            "key_present": bool(p.get("api_key")),
            "key_source": p.get("api_key_source"),
            "recognized_key_envs": p.get(
                "recognized_key_envs_all",
                p.get("recognized_key_envs", []),
            ),
            "models_status": None,
            "model_listed": None,
            "model_auto_selected": False,
            "request_attempts": 0,
            "chat_probe_attempts": 0,
            "chat_probe_success": False,
            "chat_probe_status": None,
            "chat_probe_model": None,
            "chat_probe_results": [],
            "network_path": None,
            "network_attempts": [],
            "error": None,
        }

        if not p.get("api_key"):
            result["error"] = (
                "No recognized hosted API key is visible to this kernel."
            )
            return result

        import requests

        paths = [
            ("environment", True),
            ("direct_no_env_proxy", False),
        ]

        # ----- 1) Standard model-list diagnostic ----------------------
        environment_transport_failed = False

        for path_name, trust_env in paths:
            session = requests.Session()
            session.trust_env = trust_env
            result["request_attempts"] += 1

            try:
                r = session.get(
                    f"{p['base_url']}/models",
                    headers={
                        "Authorization": f"Bearer {p['api_key']}",
                        "Connection": "close",
                    },
                    timeout=(config.API_CONNECT_TIMEOUT_S, 30),
                )

                result["models_status"] = int(r.status_code)
                result["network_path"] = path_name
                result["network_attempts"].append({
                    "path": path_name,
                    "status": int(r.status_code),
                    "error": None,
                })

                if r.ok:
                    model_rows = [
                        x
                        for x in r.json().get("data", [])
                        if isinstance(x, dict) and x.get("id")
                    ]
                    ids = {str(x.get("id")) for x in model_rows}

                    selected_model = str(p.get("model") or "")
                    model_listed = selected_model in ids
                    auto_selected = False

                    if (
                        not model_listed
                        and self.mode in ("project", "hosted_auto")
                        and ids
                    ):
                        preferred = [
                            "gpt-4o-mini",
                            "gpt-4.1-mini",
                            "gpt-4.1-nano",
                            "gpt-5-mini",
                            "openai/gpt-oss-20b",
                            "openai/gpt-oss-120b",
                            "gpt-4o",
                        ]

                        for candidate in preferred:
                            if candidate in ids:
                                selected_model = candidate
                                model_listed = True
                                auto_selected = True
                                break

                        if not model_listed:
                            blocked = (
                                "embed", "whisper", "tts", "audio",
                                "image", "dall", "moderation",
                            )
                            text_candidates = [
                                mid
                                for mid in sorted(ids)
                                if not any(
                                    word in mid.lower()
                                    for word in blocked
                                )
                            ]

                            if text_candidates:
                                selected_model = text_candidates[0]
                                model_listed = True
                                auto_selected = True

                    if model_listed:
                        self.provider["model"] = selected_model
                        self._apply_model_price(selected_model)
                        p["model"] = selected_model
                        result["model"] = selected_model
                        result["model_listed"] = True
                        result["model_auto_selected"] = bool(auto_selected)
                        result["available_model_count"] = len(ids)
                        result["error"] = None
                        self.preferred_network_path = path_name
                        return result

                    # Model listing itself worked, but configured model is not
                    # available. Continue to the tiny chat probes below.
                    result["model"] = selected_model
                    result["model_listed"] = False
                    result["model_auto_selected"] = bool(auto_selected)
                    result["available_model_count"] = len(ids)
                    result["error"] = (
                        "Configured model not present in the model list; "
                        "trying a bounded Chat Completions probe."
                    )
                    self.preferred_network_path = path_name
                    break

                result["error"] = r.text[:300]

                if r.status_code == 401:
                    # Invalid authentication is authoritative.
                    return result

                if 400 <= r.status_code < 500 and r.status_code != 403:
                    return result

            except requests.RequestException as e:
                if path_name == "environment":
                    environment_transport_failed = True

                result["network_attempts"].append({
                    "path": path_name,
                    "status": None,
                    "error": str(e)[:300],
                })
                result["error"] = str(e)[:300]

            finally:
                session.close()

        # ----- 2) Tiny real chat probe --------------------------------
        # If the normal environment route timed out, start with direct on the
        # probe so we do not pay the same proxy timeout again.
        if (
            self.preferred_network_path == "direct_no_env_proxy"
            or environment_transport_failed
        ):
            probe_paths = [("direct_no_env_proxy", False)]
        elif self.preferred_network_path == "environment":
            probe_paths = [
                ("environment", True),
                ("direct_no_env_proxy", False),
            ]
        else:
            probe_paths = [
                ("environment", True),
                ("direct_no_env_proxy", False),
            ]

        provider_name = str(p.get("provider_name") or "")
        current_model = str(p.get("model") or "")

        probe_models = [current_model] if current_model else []

        # Keep the diagnostic bounded: at most two model IDs and at most two
        # network paths. Failed 403s consume no token budget.
        probe_models = probe_models[:2]

        for model in probe_models:
            for path_name, trust_env in probe_paths:
                result["chat_probe_attempts"] += 1

                probe = self._diagnostic_chat_probe(
                    model=model,
                    network_path=path_name,
                    trust_env=trust_env,
                )
                result["chat_probe_results"].append(probe)
                result["chat_probe_status"] = probe.get("status")
                result["chat_probe_model"] = model

                if probe.get("success"):
                    result["chat_probe_success"] = True
                    result["model"] = model
                    result["network_path"] = path_name
                    result["error"] = None
                    self.preferred_network_path = path_name
                    self.provider["model"] = model
                    self._apply_model_price(model)
                    return result

                # 401 is an authentication failure; changing model/path cannot
                # repair the key.
                if probe.get("status") == 401:
                    result["error"] = probe.get("error")
                    return result

                # If the environment path has a transport failure, immediately
                # try direct; if direct returns 403, try the second bounded model
                # in case the course/project key has model permissions.
                if (
                    path_name == "environment"
                    and probe.get("status") == 403
                ):
                    continue

            # Continue to the second bounded model after a direct 403.

        if result["chat_probe_results"]:
            result["error"] = (
                result["chat_probe_results"][-1].get("error")
                or result.get("error")
            )

        return result

    def generate(self, system: str, user: str) -> str | None:
        """Return the reply, or None to signal the extractive fallback."""
        self.last_cost_usd = 0.0

        if not self.available():
            return None

        if self.hosted_disabled_reason:
            return None

        key = (self.mode, system, user)

        if key in self._cache:
            self.cache_hits += 1
            return self._cache[key]

        try:
            if self.mode == "local" and config.LOCAL_BACKEND == "ollama":
                text = self._ollama(system, user)
            elif self.mode == "local":
                text = self._hf_generate(system, user)
            else:
                text = self._openai_compatible(system, user)
        except Exception as e:
            log.warning(
                "LLM error (%s), falling back to extractive: %s",
                self.backend,
                str(e)[:220],
            )
            return None

        self._cache[key] = text
        return text

    # -- local transformers --
    def _load_hf(self):
        if self._hf is None:
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer

            log.info("loading %s", config.HF_LLM_MODEL)
            tok = AutoTokenizer.from_pretrained(config.HF_LLM_MODEL)
            model = AutoModelForCausalLM.from_pretrained(
                config.HF_LLM_MODEL,
                dtype=torch.float16,
                device_map="cuda" if torch.cuda.is_available() else "cpu",
            )
            self._hf = (tok, model)

        return self._hf

    def _hf_generate(self, system: str, user: str) -> str:
        import torch

        tok, model = self._load_hf()
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]
        text = tok.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = tok(text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=self.max_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=tok.eos_token_id,
            )

        return tok.decode(
            out[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        ).strip()

    def _ollama(self, system: str, user: str) -> str:
        import requests

        r = requests.post(
            f"{config.OLLAMA_BASE_URL}/api/chat",
            timeout=300,
            json={
                "model": config.OLLAMA_MODEL,
                "stream": False,
                "options": {
                    "temperature": self.temperature,
                    "num_predict": self.max_tokens,
                },
                "messages": [
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
            },
        )
        r.raise_for_status()
        return r.json()["message"]["content"].strip()

    # -- hosted OpenAI-compatible --
    def _openai_compatible(self, system: str, user: str) -> str:
        import requests

        p = self.provider

        chargeable = bool(p.get("billed", p.get("paid", False)))

        if self.budget.attempted_calls >= int(config.MAX_HOSTED_CHAT_ATTEMPTS):
            raise RuntimeError(
                "hosted API attempt cap reached; refusing further requests "
                "to protect the non-rechargeable course credit"
            )

        if chargeable and not self.budget.can_spend():
            raise RuntimeError(
                "operational API safety cap reached; refusing chargeable API call"
            )

        if self.mode == "hosted_auto":
            provider_name = str(p.get("provider_name") or "")
        elif self.mode == "project":
            provider_name = config.PROJECT_PROVIDER
        elif self.mode == "free":
            provider_name = config.FREE_PROVIDER
        else:
            provider_name = "paid"

        payload = {
            "model": p["model"],
            "temperature": self.temperature,
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
        }

        if provider_name == "metis":
            payload["max_tokens"] = self.max_tokens
        else:
            payload["max_completion_tokens"] = self.max_tokens


        retryable_statuses = {429, 500, 502, 503, 504}
        max_rounds = max(1, int(config.API_MAX_ATTEMPTS))
        last_error = None

        # Reuse the path that passed preflight. On some Windows/VPN setups
        # the environment proxy times out while the direct no-env-proxy path
        # works. Avoid paying the broken proxy timeout before every chat call.
        for round_idx in range(1, max_rounds + 1):
            if self.preferred_network_path == "direct_no_env_proxy":
                network_paths = [("direct_no_env_proxy", False)]
            elif self.preferred_network_path == "environment":
                network_paths = [
                    ("environment", True),
                    ("direct_no_env_proxy", False),
                ]
            else:
                network_paths = [
                    ("environment", True),
                    ("direct_no_env_proxy", False),
                ]

            for path_name, trust_env in network_paths:
                if self.budget.attempted_calls >= int(config.MAX_HOSTED_CHAT_ATTEMPTS):
                    raise RuntimeError(
                        "hosted API attempt cap reached during retry loop"
                    )

                if chargeable and not self.budget.can_spend():
                    raise RuntimeError(
                        "operational API safety cap reached during retry loop"
                    )

                session = requests.Session()
                session.trust_env = trust_env
                self.budget.record_attempt(provider_name, p["model"])

                try:
                    r = session.post(
                        f"{p['base_url']}/chat/completions",
                        timeout=(
                            config.API_CONNECT_TIMEOUT_S,
                            config.API_READ_TIMEOUT_S,
                        ),
                        headers={
                            "Authorization": f"Bearer {p['api_key']}",
                            "Content-Type": "application/json",
                            "Connection": "close",
                        },
                        json=payload,
                    )
                except requests.RequestException as e:
                    self.budget.record_failure(
                        provider_name,
                        p["model"],
                        status=None,
                        error=f"{path_name}: {str(e)[:260]}",
                    )
                    last_error = RuntimeError(
                        f"network error via {path_name}: {str(e)[:260]}"
                    )
                    session.close()
                    # Try the direct path immediately after an environment
                    # transport failure.  If direct also fails, the round ends.
                    continue
                finally:
                    try:
                        session.close()
                    except Exception:
                        pass

                if not r.ok:
                    self.budget.record_failure(
                        provider_name,
                        p["model"],
                        status=int(r.status_code),
                        error=f"{path_name}: {r.text[:260]}",
                    )
                    last_error = RuntimeError(
                        f"HTTP {r.status_code} via {path_name}: {r.text[:260]}"
                    )

                    # Authentication/validation errors will not improve with a
                    # different proxy path.
                    if r.status_code in (400, 401, 404, 422):
                        raise last_error

                    # 403 can depend on the network/IP route, so allow the
                    # direct path to be attempted once.  If the direct path is
                    # also forbidden, stop calling the hosted provider for the
                    # rest of this notebook run.  Repeating a forbidden request
                    # only inflates latency/API-attempt counts without helping.
                    if r.status_code == 403 and path_name == "environment":
                        continue

                    if r.status_code == 403 and path_name == "direct_no_env_proxy":
                        self.hosted_disabled_reason = (
                            "HTTP 403 from the direct hosted-provider path; "
                            "remaining answers use grounded extractive fallback."
                        )
                        raise last_error

                    if r.status_code not in retryable_statuses:
                        raise last_error

                    # Retryable provider errors end this network-path loop and
                    # are retried in the next round.
                    break

                data = r.json()
                usage = data.get("usage", {})

                self.last_cost_usd = self.budget.record_success(
                    usage.get("prompt_tokens", 0),
                    usage.get("completion_tokens", 0),
                    provider=provider_name,
                    model=p["model"],
                    price_per_m=p.get("price_per_m", config.PAID_PRICE_PER_M),
                    billed=bool(p.get("billed", p.get("paid", False))),
                )

                content = data["choices"][0]["message"].get("content", "")
                return str(content or "").strip()

            if round_idx < max_rounds:
                time.sleep(float(config.API_RETRY_BASE_S) * round_idx)

        raise last_error or RuntimeError("hosted API call failed")



def judge_llm(budget: BudgetTracker | None = None) -> "LLM":
    mode = "extractive" if config.JUDGE_MODE == "none" else config.JUDGE_MODE
    return LLM(mode=mode, budget=budget)


### 2.B Shared retrieval — dense + BM25 with candidate-limited weighted RRF

Dense multilingual similarity and lexical BM25 are combined after structured filters. Only meaningful top candidates enter RRF; zero-score sparse documents are not given artificial ranks.


In [ ]:
import logging
import math
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd


log = logging.getLogger("digikala.retrieval")
_model = None
_model_load_source = None


# ---- embedding model (lazy, GPU with CPU fallback) ----------------------
def get_model():
    global _model, _model_load_source

    if _model is None:
        from sentence_transformers import SentenceTransformer

        device = config.EMBEDDING_DEVICE
        try:
            import torch
            if device == "cuda" and not torch.cuda.is_available():
                device = "cpu"
        except Exception:
            device = "cpu"

        log.info("loading %s on %s", config.EMBEDDING_MODEL, device)

        try:
            _model = SentenceTransformer(
                config.EMBEDDING_MODEL,
                device=device,
                local_files_only=True,
            )
            _model_load_source = "local_cache"
        except Exception:
            _model = SentenceTransformer(
                config.EMBEDDING_MODEL,
                device=device,
            )
            _model_load_source = "download_or_cache_fallback"

    return _model


def embed(texts, batch_size: int = 256) -> np.ndarray:
    texts = [t or "" for t in texts]
    if not texts:
        return np.zeros((0, 384), dtype="float32")
    return get_model().encode(texts, batch_size=batch_size, convert_to_numpy=True,
                              normalize_embeddings=True,
                              show_progress_bar=len(texts) > 5000).astype("float32")


# ---- BM25 (Okapi) — the sparse half of hybrid retrieval -----------------
class BM25Okapi:
    """Okapi BM25 backed by a scipy-sparse term-document matrix.

    A CountVectorizer builds the doc×term counts once (C-optimized), so this scales
    to ~1M products: scoring a query touches only the columns of its terms, and the
    whole index (matrix + idf + doc lengths + vocab) saves/loads from disk in
    seconds instead of rebuilding a Python inverted index every process start.
    """

    def __init__(self, counts, doc_len, idf, vocab, k1: float = 1.5, b: float = 0.75):
        from scipy.sparse import csc_matrix
        self.k1, self.b = k1, b
        self.counts = counts.tocsc() if not isinstance(counts, csc_matrix) else counts
        self.doc_len = doc_len.astype(np.float32)
        self.n = self.counts.shape[0]
        self.avgdl = float(self.doc_len.mean()) if self.n else 1.0
        self.idf = idf.astype(np.float32)
        self.vocab = vocab                            # term -> column index

    @classmethod
    def from_texts(cls, texts, k1: float = 1.5, b: float = 0.75):
        from sklearn.feature_extraction.text import CountVectorizer
        # corpus texts are already normalized (*_norm columns) -> fast regex tokenizer
        vec = CountVectorizer(tokenizer=pt.tokenize_norm, token_pattern=None,
                              lowercase=False, preprocessor=None)
        texts = list(texts)
        try:
            counts = vec.fit_transform(texts)             # docs x terms, CSR
            vocab = vec.vocabulary_
        except ValueError:
            # Can happen when every review in a very small slice is empty after
            # normalization. Keep retrieval alive and let dense search carry it.
            from scipy.sparse import csr_matrix
            counts = csr_matrix((len(texts), 0), dtype=np.float32)
            vocab = {}

        doc_len = np.asarray(counts.sum(axis=1)).ravel()
        if counts.shape[1] == 0:
            idf = np.zeros(0, dtype=np.float32)
        else:
            df = np.asarray((counts > 0).sum(axis=0)).ravel()
            n = counts.shape[0]
            idf = np.log(1 + (n - df + 0.5) / (df + 0.5))
        return cls(counts, doc_len, idf, vocab, k1, b)

    def get_scores(self, query) -> np.ndarray:
        scores = np.zeros(self.n, dtype=np.float32)
        avgdl = max(self.avgdl, 1e-6)
        denom_len = self.k1 * (1 - self.b + self.b * self.doc_len / avgdl)
        for term in set(pt.tokenize(query)):
            col = self.vocab.get(term)
            if col is None:
                continue
            c = self.counts.getcol(col)               # sparse column of term freqs
            rows = c.indices
            freq = c.data.astype(np.float32)
            scores[rows] += self.idf[col] * freq * (self.k1 + 1) / (freq + denom_len[rows])
        return scores

    def save(self, path):
        from scipy.sparse import save_npz
        path = Path(path)
        save_npz(path / "bm25_counts.npz", self.counts)
        np.save(path / "bm25_doc_len.npy", self.doc_len)
        np.save(path / "bm25_idf.npy", self.idf)
        import json
        (path / "bm25_vocab.json").write_text(json.dumps(self.vocab), encoding="utf-8")

    @classmethod
    def load(cls, path, k1: float = 1.5, b: float = 0.75):
        from scipy.sparse import load_npz
        import json
        path = Path(path)
        counts = load_npz(path / "bm25_counts.npz")
        doc_len = np.load(path / "bm25_doc_len.npy")
        idf = np.load(path / "bm25_idf.npy")
        vocab = json.loads((path / "bm25_vocab.json").read_text(encoding="utf-8"))
        return cls(counts, doc_len, idf, vocab, k1, b)


def rrf_fuse(rank_lists, k: int = 60, weights=None) -> dict:
    """Reciprocal-rank fusion over short candidate lists.

    Weights are optional and let sparse lexical evidence count slightly more for
    exact product attributes while keeping dense semantic retrieval in the mix.
    """
    if weights is None:
        weights = [1.0] * len(rank_lists)

    scores: dict = {}

    for ranks, weight in zip(rank_lists, weights):
        for rank, idx in enumerate(ranks):
            scores[idx] = scores.get(idx, 0.0) + float(weight) / (k + rank + 1)

    return scores


def _minmax(v):
    lo, hi = v.min(), v.max()
    return np.zeros_like(v) if hi - lo < 1e-9 else (v - lo) / (hi - lo)


def product_filter_mask(products: pd.DataFrame, filters: dict) -> np.ndarray:
    mask = np.ones(len(products), dtype=bool)
    f = filters or {}
    if f.get("category"):
        cat = str(f["category"])
        col_mask = np.zeros(len(products), dtype=bool)
        for col in ("category1_norm", "category2_norm", "sub_category_norm"):
            if col in products:
                col_mask |= products[col].fillna("").map(lambda v: cat in str(v)).to_numpy()
        mask &= col_mask
    if f.get("brand"):
        mask &= products["brand_norm"].fillna("").map(lambda v: str(f["brand"]) in str(v)).to_numpy()
    price = products["price_clean"].to_numpy(dtype=float)
    if f.get("price_min") is not None:
        mask &= np.where(np.isnan(price), False, price >= f["price_min"])
    if f.get("price_max") is not None:
        mask &= np.where(np.isnan(price), False, price <= f["price_max"])
    if f.get("exclude_fake", True) and "is_fake" in products:
        mask &= ~products["is_fake"].fillna(False).astype(bool).to_numpy()
    return mask


# ---- the retriever ------------------------------------------------------
class ProductIndex:
    """Full-catalogue product retrieval: dense vectors + BM25, fused with RRF."""

    def __init__(self, products: pd.DataFrame, vectors: np.ndarray, bm25: BM25Okapi, rrf_k: int = 60):
        self.products = products.reset_index(drop=True)
        # keep as-is (a memmap stays on disk) unless the dtype needs converting
        self.vectors = vectors if vectors.dtype == np.float32 else vectors.astype("float32")
        self.bm25 = bm25
        self.rrf_k = rrf_k

    @classmethod
    def build(cls, products: pd.DataFrame) -> "ProductIndex":
        texts = products["product_text_norm"].fillna("").tolist()
        log.info("embedding %d products", len(texts))
        vectors = embed(texts)
        bm25 = BM25Okapi.from_texts(texts)
        return cls(products, vectors, bm25)

    def save(self, path=config.PRODUCT_INDEX_DIR):
        path = Path(path)
        path.mkdir(parents=True, exist_ok=True)
        np.save(path / "vectors.npy", self.vectors)
        self.products.to_parquet(path / "products.parquet", index=False)
        self.bm25.save(path)                          # persist the sparse BM25 so load is fast
        log.info("saved product index (%d) to %s", len(self.products), path)

    @classmethod
    def load(cls, path=config.PRODUCT_INDEX_DIR) -> "ProductIndex":
        path = Path(path)
        products = pd.read_parquet(path / "products.parquet")
        vectors = np.load(path / "vectors.npy", mmap_mode="r")
        bm25 = BM25Okapi.load(path)
        return cls(products, vectors, bm25)

    def search(
        self,
        query: str,
        filters: dict | None = None,
        k: int = config.TOP_K,
    ) -> list[dict]:
        qv = embed([query])[0]
        dense = self.vectors @ qv
        sparse = self.bm25.get_scores(query)

        mask = product_filter_mask(self.products, filters)
        valid = np.flatnonzero(mask)

        if valid.size == 0:
            return []

        pool = min(
            int(valid.size),
            max(int(config.RRF_CANDIDATE_POOL), int(k) * 20),
        )

        d_order = valid[
            np.argsort(-dense[valid], kind="stable")[:pool]
        ]

        sparse_valid = valid[sparse[valid] > 0]

        if sparse_valid.size:
            s_order = sparse_valid[
                np.argsort(-sparse[sparse_valid], kind="stable")[:pool]
            ]
            fused = rrf_fuse(
                [d_order.tolist(), s_order.tolist()],
                k=self.rrf_k,
                weights=[
                    config.PRODUCT_RRF_DENSE_WEIGHT,
                    config.PRODUCT_RRF_SPARSE_WEIGHT,
                ],
            )
        else:
            fused = rrf_fuse(
                [d_order.tolist()],
                k=self.rrf_k,
            )

        top = sorted(
            fused.items(),
            key=lambda kv: kv[1],
            reverse=True,
        )[:k]

        out = []

        for idx, score in top:
            r = self.products.iloc[idx]
            out.append({
                "rank": len(out) + 1,
                "product_id": int(r["product_id"]),
                "title": r["title_fa"],
                "brand": r["brand_norm"],
                "price": float(r["price_clean"]) if pd.notna(r["price_clean"]) else None,
                "rate": float(r["product_rate_clean"]) if pd.notna(r["product_rate_clean"]) else None,
                "rate_count": int(r["rate_count"]) if pd.notna(r["rate_count"]) else 0,
                "comment_count": int(r.get("comment_count", 0)),
                "category": r.get("category1_norm", ""),
                "score": round(float(score), 5),
            })

        return out



class ReviewRetriever:
    """Per-product hybrid review search with query-aware reranking."""

    _NEGATIVE_CUES = (
        "مشکل", "ایراد", "عیب", "بد", "ضعف", "منفی", "ناراضی",
        "خراب", "معیوب", "نمی ارزد", "نمی‌ارزد", "عدم توصیه",
        "نقاط ضعف", "معایب",
    )
    _POSITIVE_CUES = (
        "خوب", "مثبت", "مزیت", "مزایا", "رضایت", "راضی",
        "ارزش خرید", "پیشنهاد", "نقاط قوت",
    )

    def __init__(
        self,
        comments_by_product: dict,
        rrf_k: int = 60,
    ):
        self.by_product = comments_by_product
        self.rrf_k = rrf_k

    @staticmethod
    def _rating_norm(rate: np.ndarray) -> np.ndarray:
        rate = np.asarray(rate, dtype=float)

        if rate.size == 0:
            return rate

        finite = np.isfinite(rate)

        if not finite.any():
            return np.full(len(rate), 0.5, dtype=float)

        vals = rate.copy()
        fill = float(np.nanmedian(vals[finite]))
        vals[~finite] = fill

        lo = float(np.min(vals))
        hi = float(np.max(vals))

        if hi - lo < 1e-9:
            return np.full(len(vals), 0.5, dtype=float)

        return (vals - lo) / (hi - lo)

    def _polarity(self, query: str) -> str:
        q = pt.normalize(query)

        if any(x in q for x in self._NEGATIVE_CUES):
            return "negative"

        if any(x in q for x in self._POSITIVE_CUES):
            return "positive"

        return "neutral"

    def retrieve(
        self,
        query: str,
        product_id: int,
        k: int = config.TOP_K,
        rerank: bool = True,
    ) -> list[dict]:
        rev = self.by_product.get(int(product_id))

        if rev is None or rev.empty:
            return []

        rev = rev.reset_index(drop=True)
        texts = rev["comment_text_norm"].fillna("").tolist()

        qv = embed([query])[0]
        dense = embed(texts) @ qv
        sparse = BM25Okapi.from_texts(texts).get_scores(query)

        pool = min(
            len(rev),
            max(int(config.REVIEW_CANDIDATE_POOL), int(k) * 10),
        )

        order_d = np.argsort(-dense, kind="stable")[:pool].tolist()

        positive_sparse = np.flatnonzero(sparse > 0)

        if positive_sparse.size:
            order_s = positive_sparse[
                np.argsort(-sparse[positive_sparse], kind="stable")[:pool]
            ].tolist()
            fused = rrf_fuse(
                [order_d, order_s],
                k=self.rrf_k,
                weights=[1.0, 1.05],
            )
        else:
            fused = rrf_fuse([order_d], k=self.rrf_k)

        if rerank and fused:
            idxs = list(fused)
            base = np.array([fused[i] for i in idxs], dtype=float)
            base = _minmax(base)

            likes = rev["likes"].fillna(0).to_numpy(dtype=float)
            buyer = rev["is_buyer"].fillna(False).astype(float).to_numpy()
            lk = _minmax(np.log1p(likes[idxs]))

            rate_all = pd.to_numeric(
                rev["rate_clean"],
                errors="coerce",
            ).to_numpy(dtype=float)
            rate_norm = self._rating_norm(rate_all)

            status = (
                rev["recommendation_status"]
                .fillna("")
                .astype(str)
                .to_numpy()
            )

            polarity = self._polarity(query)
            intent_score = np.full(len(idxs), 0.5, dtype=float)

            has_disadv = (
                rev["disadvantages_norm"].fillna("").astype(str).str.len().gt(0).to_numpy()
                if "disadvantages_norm" in rev
                else np.zeros(len(rev), dtype=bool)
            )
            has_adv = (
                rev["advantages_norm"].fillna("").astype(str).str.len().gt(0).to_numpy()
                if "advantages_norm" in rev
                else np.zeros(len(rev), dtype=bool)
            )

            if polarity == "negative":
                for j, idx in enumerate(idxs):
                    status_score = 1.0 if status[idx] == "not_recommended" else (
                        0.5 if status[idx] == "no_idea" else 0.0
                    )
                    intent_score[j] = (
                        0.55 * status_score
                        + 0.30 * (1.0 - rate_norm[idx])
                        + 0.15 * float(has_disadv[idx])
                    )

            elif polarity == "positive":
                for j, idx in enumerate(idxs):
                    status_score = 1.0 if status[idx] == "recommended" else (
                        0.5 if status[idx] == "no_idea" else 0.0
                    )
                    intent_score[j] = (
                        0.60 * status_score
                        + 0.25 * rate_norm[idx]
                        + 0.15 * float(has_adv[idx])
                    )

            final_scores = {}

            for j, idx in enumerate(idxs):
                if polarity == "neutral":
                    score = (
                        0.84 * base[j]
                        + 0.09 * lk[j]
                        + 0.07 * buyer[idx]
                    )
                elif polarity == "negative":
                    iw = float(config.REVIEW_NEGATIVE_INTENT_WEIGHT)
                    score = (
                        (0.93 - iw) * base[j]
                        + 0.07 * lk[j]
                        + 0.03 * buyer[idx]
                        + iw * intent_score[j]
                    )
                else:
                    iw = float(config.REVIEW_POSITIVE_INTENT_WEIGHT)
                    score = (
                        (0.93 - iw) * base[j]
                        + 0.07 * lk[j]
                        + 0.03 * buyer[idx]
                        + iw * intent_score[j]
                    )

                final_scores[idx] = float(score)

            fused = final_scores

        top = sorted(
            fused.items(),
            key=lambda kv: kv[1],
            reverse=True,
        )[:k]

        out = []

        for idx, score in top:
            r = rev.iloc[idx]
            out.append({
                "rank": len(out) + 1,
                "comment_id": int(r["comment_id"]),
                "product_id": int(r["product_id"]),
                "text": r["comment_text_norm"],
                "rate": float(r["rate_clean"]) if pd.notna(r["rate_clean"]) else None,
                "recommendation_status": str(r["recommendation_status"]),
                "likes": int(r["likes"]) if pd.notna(r["likes"]) else 0,
                "is_buyer": bool(r["is_buyer"]) if pd.notna(r["is_buyer"]) else False,
                "has_advantage": bool(str(r.get("advantages_norm", "") or "").strip()),
                "has_disadvantage": bool(str(r.get("disadvantages_norm", "") or "").strip()),
                "score": round(float(score), 5),
            })

        return out


# ---- build helpers ------------------------------------------------------
def _prepare_products(products: pd.DataFrame, comments: pd.DataFrame) -> pd.DataFrame:
    products = products.copy()
    counts = comments.groupby("product_id").size()
    products["comment_count"] = products["product_id"].map(counts).fillna(0).astype(int)
    return products


def build_product_index(sample_comments: int | None = None) -> ProductIndex:
    """Build + persist the product index from the cleaned tables."""
    products = pd.read_parquet(config.PRODUCTS_CLEAN)
    comments = pd.read_parquet(config.COMMENTS_CLEAN, columns=["product_id"])
    products = _prepare_products(products, comments)
    idx = ProductIndex.build(products)
    idx.save()
    return idx


def load_comments_by_product(only_with_text: bool = True) -> dict:
    """Group cleaned comments by product_id for on-demand review retrieval."""
    comments = pd.read_parquet(config.COMMENTS_CLEAN)
    if only_with_text and "has_text" in comments:
        comments = comments[comments["has_text"].astype(bool)]
    return {int(pid): g.reset_index(drop=True)
            for pid, g in comments.groupby("product_id", sort=False) if pd.notna(pid)}

retrieval = types.SimpleNamespace(get_model=get_model, embed=embed, BM25Okapi=BM25Okapi,
    rrf_fuse=rrf_fuse, product_filter_mask=product_filter_mask, ProductIndex=ProductIndex,
    ReviewRetriever=ReviewRetriever, load_comments_by_product=load_comments_by_product)

### 2.C Shared intent router + entity resolution

In [ ]:
import re
from dataclasses import dataclass, field

import pandas as pd


_GENERIC_VALUES = {"متفرقه", "نامشخص", "سایر"}

_CMP = (
    "مقایسه", "تفاوت", "فرق", "کدوم بهتر", "کدام بهتر",
    "بهتره", "بهتر است", "vs", "در برابر"
)

_QA = (
    "آیا", "چطور", "چگونه", "چرا", "چقدر",
    "مشکل", "مشکلات", "ایراد", "ایرادها", "ایرادات",
    "سایز", "اندازه", "جنس", "کیفیت", "مناسب",
    "خوب", "بد", "رضایت", "راضی", "ارزش خرید",
    "باتری", "دوام", "نظر"
)

_DISCOVERY = (
    "پیشنهاد", "پیشنهاد بده", "پیشنهاد کن",
    "معرفی", "معرفی کن", "گزینه", "گزینه‌ها", "گزینه های", "گزینه‌های",
    "برای خرید", "خرید", "بخرم", "اقتصادی", "ارزان", "ارزانتر",
    "ارزان‌تر", "بهترین"
)

_MNG = (
    "شکایت", "شکایت‌ها", "شکایات", "تحلیل", "عملکرد",
    "مشکلات پرتکرار", "پرتکرارترین", "پرفروش",
    "کمترین", "بیشترین", "محبوب", "نارضایتی",
    "نرخ توصیه", "الگوی رضایت", "بازار", "مدیر", "مدیریتی",
    "برندها"
)


def _has_cue(text, cues) -> bool:
    """Match routing cues without accidental substring matches."""
    t = pt.normalize(text)
    tokens = set(pt.tokenize(t))

    for cue in cues:
        cue_n = pt.normalize(cue)
        cue_tokens = pt.tokenize(cue_n)

        if not cue_tokens:
            continue

        if len(cue_tokens) == 1:
            if cue_tokens[0] in tokens:
                return True
        elif cue_n in t:
            return True

    return False


@dataclass
class Catalog:
    products: pd.DataFrame
    comments_by_product: dict
    category_values: dict = field(default_factory=dict)
    brand_values: set = field(default_factory=set)
    _by_id: pd.DataFrame = None                       # products indexed by product_id
    reviewed_title_tokens: dict = field(default_factory=dict)  # pid -> token set (reviewed only)

    @classmethod
    def build(cls, products: pd.DataFrame, comments_by_product: dict) -> "Catalog":
        products = products.reset_index(drop=True)
        by_id = products.dropna(subset=["product_id"]).copy()
        by_id["product_id"] = by_id["product_id"].astype(int)
        by_id = by_id.set_index("product_id", drop=False)
        cats = {c: {v for v in products[c].dropna().unique() if str(v).strip() and str(v) not in _GENERIC_VALUES}
                for c in ("category1_norm", "category2_norm", "sub_category_norm") if c in products}
        brands = {v for v in products["brand_norm"].dropna().unique() if str(v).strip() and str(v) not in _GENERIC_VALUES}
        # only reviewed products are candidates for fuzzy name->id resolution (QA needs reviews)
        reviewed = {}
        for pid in comments_by_product:
            if pid in by_id.index:
                reviewed[int(pid)] = set(pt.tokenize_norm(by_id.at[int(pid), "title_norm"]))
        return cls(products, comments_by_product, cats, brands, by_id, reviewed)

    @property
    def product_lookup(self):                          # kept for compatibility with callers
        return self._by_id

    def product(self, pid):
        pid = int(pid)
        if self._by_id is not None and pid in self._by_id.index:
            return self._by_id.loc[pid].to_dict()
        return None


def match_known_value(query, values, min_ratio: float = 0.6):
    t = pt.normalize(query)
    t_tokens = set(pt.tokenize(t))
    best, best_score = None, 0.0
    for v in values:
        vn = pt.normalize(v).strip()
        if not vn or len(vn) < 3 or vn in _GENERIC_VALUES:
            continue
        v_tokens = set(pt.tokenize(vn))
        if not v_tokens:
            continue
        if vn in t:
            score = len(vn) + 100.0
        else:
            ratio = len(t_tokens & v_tokens) / max(1, len(v_tokens))
            if ratio < min_ratio:
                continue
            score = ratio * len(v_tokens)
        if score > best_score:
            best, best_score = vn, score
    return best


def match_known_brand(query, values):
    """Use exact phrase boundaries for brands to avoid accidental preference-word matches."""
    t = pt.normalize(query)
    best = None

    for v in values:
        vn = pt.normalize(v).strip()

        if not vn or len(vn) < 2 or vn in _GENERIC_VALUES:
            continue

        if re.search(rf"(?<!\w){re.escape(vn)}(?!\w)", t):
            if best is None or len(vn) > len(best):
                best = vn

    return best


def extract_product_ids(catalog: Catalog, query) -> list[int]:
    t = pt.normalize(query)
    found: list[int] = []
    for m in re.finditer(r"\b(\d{6,9})\b", t):
        pid = int(m.group(1))
        if pid in catalog._by_id.index and pid not in found:
            found.append(pid)
    return found


def resolve_product_id(catalog: Catalog, text):
    """Fuzzy map a description to a product id, searching only reviewed products
    (a Q&A only makes sense for a product that actually has reviews)."""
    t = pt.normalize(text)
    m = re.search(r"\b(\d{6,9})\b", t)
    if m and int(m.group(1)) in catalog._by_id.index:
        return int(m.group(1))
    q_tokens = set(pt.tokenize(t))
    if not q_tokens:
        return None
    best_pid, best = None, 0.0
    for pid, title_tokens in catalog.reviewed_title_tokens.items():
        if not title_tokens:
            continue
        score = len(q_tokens & title_tokens) / max(1, len(title_tokens))
        if score > best:
            best_pid, best = pid, score
    return best_pid if best >= 0.5 else None


def resolve_scope(catalog: Catalog, text) -> dict:
    best, best_score = None, 0.0
    for kind, values in catalog.category_values.items():
        v = match_known_value(text, values)
        if v and len(v) > best_score:
            best, best_score = (kind, v), len(v)
    v = match_known_brand(text, catalog.brand_values)
    if v and len(v) > best_score:
        best, best_score = ("brand_norm", v), len(v)
    return {"kind": best[0], "value": best[1]} if best else {}


def extract_filters(catalog: Catalog, query) -> dict:
    filters: dict = {}
    best_cat, best_len = None, 0
    for values in catalog.category_values.values():
        v = match_known_value(query, values)
        if v and len(v) > best_len:
            best_cat, best_len = v, len(v)
    if best_cat:
        filters["category"] = best_cat
    b = match_known_brand(query, catalog.brand_values)
    if b:
        filters["brand"] = b
    filters.update(pt.extract_price_constraint(query, config.TOMAN_TO_RIAL))
    return filters


@dataclass
class Route:
    intent: str
    product_id: int | None = None
    product_ids: list = field(default_factory=list)
    scope: dict = field(default_factory=dict)
    filters: dict = field(default_factory=dict)
    needs_clarification: bool = False


class IntentRouter:
    def __init__(self, catalog: Catalog):
        self.c = catalog

    def route(self, query) -> Route:
        t = pt.normalize(query)
        ids = extract_product_ids(self.c, query)
        scope = resolve_scope(self.c, query)

        has_cmp_cue = _has_cue(t, _CMP)
        has_qa_cue = _has_cue(t, _QA)
        has_discovery_cue = _has_cue(t, _DISCOVERY)
        has_managerial_cue = _has_cue(t, _MNG)

        # Explicit comparison first.
        if len(ids) >= 2:
            return Route("comparison", product_ids=ids)

        if has_cmp_cue:
            return Route(
                "comparison",
                product_ids=ids,
                needs_clarification=len(ids) < 2,
            )

        # One explicit product ID is product-specific.
        if len(ids) == 1:
            return Route("product_qa", product_id=ids[0])

        # Aggregate/category analytics require a strong managerial cue.
        if scope and has_managerial_cue:
            return Route("managerial", scope=scope)

        # Strong shopping/discovery wording wins before fuzzy title resolution.
        if has_discovery_cue:
            return Route(
                "discovery",
                filters=extract_filters(self.c, query),
            )

        # Fuzzy product resolution is only used for genuine QA-like wording.
        if has_qa_cue:
            pid = resolve_product_id(self.c, query)

            if pid is not None:
                return Route("product_qa", product_id=pid)

        return Route(
            "discovery",
            filters=extract_filters(self.c, query),
        )


### 2.D Shared grounded prompts + evidence formatters

In [ ]:
SYSTEM_CORE = (
    "تو یک دستیار خرید دیجی‌کالا هستی که فقط بر پایهٔ داده‌های واقعی پاسخ می‌دهی.\n"
    "۱) فقط از «مدارک» استفاده کن؛ دانش قبلی یا عددسازی ممنوع.\n"
    "۲) ادعاهای مربوط به یک محصول یا بازبینی را با [محصول شناسه] یا [بازبینی شناسه] ارجاع بده؛ آمار تجمیعی فقط از مدارک داده‌شده استفاده کند.\n"
    "۳) اگر مدارک کافی نیست، صریحاً بنویس «اطلاعات کافی موجود نیست».\n"
    "۴) پاسخ فارسی، کوتاه و ساختاریافته باشد.\n"
    "۵) عددها را عیناً از مدارک کپی کن.\n"
)
DISCOVERY_SYSTEM = SYSTEM_CORE + "\nوظیفه: پیشنهاد و رتبه‌بندی محصول. حداکثر {k} پیشنهاد، هرکدام با [محصول ...] و دلیل کوتاه."
QA_SYSTEM = SYSTEM_CORE + "\nوظیفه: پاسخ به پرسش دربارهٔ یک محصول، فقط از بازبینی‌ها. هر ادعا با [بازبینی ...]. اگر پرسش دربارهٔ ایراد/ضعف است فقط نکتهٔ منفیِ صریح موجود در متن بازبینی را گزارش کن و جملهٔ مثبت را به ضعف تبدیل نکن. اگر شواهد منفی کافی نیست، همان را صریح بگو. اگر پرسش دربارهٔ مزیت/رضایت است، فقط شواهد مثبت روشن را گزارش کن. از عبارت‌های کلی مثل «همهٔ بازبینی‌ها»، «تمام نظرها» یا «هیچ بازبینی» استفاده نکن؛ فقط دربارهٔ شواهد انتخاب‌شده و آمار قطعی داده‌شده حرف بزن. حداکثر {max_lines} خط."
COMPARISON_SYSTEM = SYSTEM_CORE + "\nوظیفه: مقایسه را در سه بخش «واقعیت‌های مستقیم محصول»، «شواهد بازبینی‌ها» و «جمع‌بندی/استنباط» بنویس؛ نبودِ داده را صریح بگو."
MANAGERIAL_SYSTEM = SYSTEM_CORE + "\nوظیفه: تحلیل مدیریتی (شکایت‌ها، محصولات با نرخ توصیهٔ پایین، رضایت برندها). حداکثر {max_paragraphs} پاراگراف."

# LLM-as-judge prompts (Phase 4)
JUDGE_FAITH_SYS = ("تو یک ارزیاب هستی. «پایبندی به منبع» را بسنج: آیا هر ادعای پاسخ در منابع "
                   "ارجاع‌شده پشتیبانی می‌شود؟ فقط یک عدد ۰ تا ۵ بنویس.")
JUDGE_REL_SYS = ("تو یک ارزیاب هستی. «مفید و مرتبط بودن پاسخ» را بسنج: آیا مستقیم و مرتبط به "
                 "سؤال جواب داده؟ فقط یک عدد ۰ تا ۵ بنویس.")
JUDGE_USER = "سؤال: {query}\n\nپاسخ سیستم:\n{answer}\n\nمنابع:\n{sources}\n\nنمره (۰ تا ۵):"


def evidence_products(rows) -> str:
    out = []
    for r in rows:
        reason = r.get("preference_reason", "")
        extra = f" | دلیل رتبه‌بندی: {reason}" if reason else ""
        out.append(
            f"[محصول {r['product_id']}] {r['title']} | برند: {r['brand'] or 'نامشخص'} | "
            f"قیمت: {format_toman(r['price'])} تومان | امتیاز: "
            f"{r['rate'] if r['rate'] is not None else 'نامشخص'} | "
            f"تعداد نظر: {r['comment_count']}{extra}"
        )
    return "\n".join(out)


def evidence_reviews(rows) -> str:
    m = {"recommended": "توصیه", "not_recommended": "عدم توصیه", "no_idea": "نظری ندارد"}
    out = []
    for r in rows:
        out.append(f"[بازبینی {r['comment_id']}] (محصول {r['product_id']}) "
                   f"امتیاز {r['rate'] if r['rate'] is not None else '?'} | "
                   f"{m.get(r['recommendation_status'], 'نامشخص')} | پسند {r['likes']}\nمتن: {r['text']}")
    return "\n".join(out)

prompts = types.SimpleNamespace(SYSTEM_CORE=SYSTEM_CORE, DISCOVERY_SYSTEM=DISCOVERY_SYSTEM,
    QA_SYSTEM=QA_SYSTEM, COMPARISON_SYSTEM=COMPARISON_SYSTEM, MANAGERIAL_SYSTEM=MANAGERIAL_SYSTEM,
    JUDGE_FAITH_SYS=JUDGE_FAITH_SYS, JUDGE_REL_SYS=JUDGE_REL_SYS, JUDGE_USER=JUDGE_USER,
    evidence_products=evidence_products, evidence_reviews=evidence_reviews)

### 2.E Assistant implementation — route → retrieve → answer → verify citations

In [ ]:
import re
import time
from dataclasses import dataclass, field

import numpy as np
import pandas as pd


# ---- managerial aggregates ---------------------------------------------
_COMPLAINT_TERMS = ["خراب", "عیب", "ایراد", "مشکل", "بوی", "شکست", "شکسته", "پاره",
                    "افتضاح", "پشیمان", "بی کیفیت", "تقلبی", "فیک", "معیوب", "جنس بد",
                    "کیفیت بد", "حساسیت", "جوش", "ترک", "خش", "لک", "چروک", "بو میده"]
_NORM_TERMS = sorted({pt.normalize(t) for t in _COMPLAINT_TERMS if pt.normalize(t)}, key=len, reverse=True)


def _negative_review_mask(rev: pd.DataFrame) -> pd.Series:
    """Scale-aware negative-review mask used in review analytics."""
    status = rev["recommendation_status"].fillna("").astype(str)
    rate = pd.to_numeric(rev["rate_clean"], errors="coerce")

    finite = rate.dropna()
    if finite.empty:
        low_rate = pd.Series(False, index=rev.index)
    else:
        # Digikala exports may use either a 0..5-like or 0..100-like rate scale.
        threshold = 2.0 if float(finite.max()) <= 5.0 else 40.0
        low_rate = rate <= threshold

    return (status == "not_recommended") | low_rate.fillna(False)


def review_stats(catalog: Catalog, product_id, light: bool = False) -> dict:
    """Aggregate one product's reviews. light=True skips the top advantages/
    disadvantages groupby — used by managerial, which ranks products but doesn't
    need per-product pros/cons (much faster over hundreds of products)."""
    rev = catalog.comments_by_product.get(int(product_id))
    if rev is None or rev.empty:
        return {
            "product_id": int(product_id), "n_reviews": 0,
            "n_labeled_recommendation": 0, "rec_rate": None,
            "not_rec_rate": None, "no_idea_rate": None, "avg_rate": None,
            "complaint_count": 0, "top_advantages": [], "top_disadvantages": [],
        }
    n = len(rev)
    status = rev["recommendation_status"].fillna("").astype(str)
    valid_status = status.isin(config.RECOMMENDATION_CLASSES)
    n_labeled = int(valid_status.sum())
    rec = int((status == "recommended").sum())
    not_rec = int((status == "not_recommended").sum())
    no_idea = int((status == "no_idea").sum())
    avg = float(rev["rate_clean"].dropna().mean()) if rev["rate_clean"].notna().any() else None

    def _top(col, k=3):
        if col not in rev:
            return []

        col_s = rev[col].fillna("").astype(str)
        valid = col_s[col_s.str.len() > 0]

        if valid.empty:
            return []

        ranked = (
            valid.to_frame()
            .assign(_likes=rev.loc[valid.index, "likes"].fillna(0))
            .groupby(col, sort=False)["_likes"]
            .sum()
            .sort_values(ascending=False)
        )

        rows = []

        for text in ranked.head(k).index:
            candidates = rev.loc[valid[valid == text].index].copy()
            candidates = candidates.sort_values("likes", ascending=False)
            r = candidates.iloc[0]

            rows.append({
                "text": str(text),
                "comment_id": int(r["comment_id"]),
                "product_id": int(r["product_id"]),
                "rate": float(r["rate_clean"]) if pd.notna(r["rate_clean"]) else None,
                "recommendation_status": str(r["recommendation_status"]),
                "likes": int(r["likes"]) if pd.notna(r["likes"]) else 0,
                "is_buyer": bool(r["is_buyer"]) if pd.notna(r["is_buyer"]) else False,
            })

        return rows

    neg = _negative_review_mask(rev)
    out = {
        "product_id": int(product_id),
        "n_reviews": n,
        "n_labeled_recommendation": n_labeled,
        "rec_rate": round(rec / n_labeled, 3) if n_labeled else None,
        "not_rec_rate": round(not_rec / n_labeled, 3) if n_labeled else None,
        "no_idea_rate": round(no_idea / n_labeled, 3) if n_labeled else None,
        "avg_rate": round(avg, 1) if avg is not None else None,
        "complaint_count": int(neg.sum()),
        "top_advantages": [],
        "top_disadvantages": [],
    }
    if not light:
        out["top_advantages"] = _top("advantages_norm")
        out["top_disadvantages"] = _top("disadvantages_norm")
    return out


def _top_complaint_terms(catalog: Catalog, pids, k: int = 8):
    per = {t: re.compile(r"(?:^|\s)" + re.escape(t) + r"(?:$|\s)") for t in _NORM_TERMS}
    counts: dict = {}
    for pid in pids:
        rev = catalog.comments_by_product.get(int(pid))
        if rev is None:
            continue
        max_rate = pd.to_numeric(rev["rate_clean"], errors="coerce").max()
        low_threshold = 2.0 if pd.notna(max_rate) and float(max_rate) <= 5.0 else 40.0

        for r in rev.itertuples():
            dis = pt.normalize(getattr(r, "disadvantages_norm", "") or "")
            is_neg = str(r.recommendation_status) == "not_recommended" or (
                pd.notna(r.rate_clean) and float(r.rate_clean) <= low_threshold)
            if not is_neg and not dis.strip():
                continue
            text = " " + dis + (" " + pt.normalize(getattr(r, "body_norm", "") or "") if is_neg else "") + " "
            for term, rx in per.items():
                m = len(rx.findall(text))
                if m:
                    counts[term] = counts.get(term, 0) + m
    top = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)[:k]
    return [{"term": t, "count": c} for t, c in top]


def managerial_summary(catalog: Catalog, scope, min_comments: int = 5) -> dict:
    kind = scope.get("kind")
    value = pt.normalize(scope.get("value", ""))
    prods = catalog.products

    if not (kind and kind in prods):
        return {"scope": scope, "n_products": 0}

    scoped = prods[
        prods[kind].fillna("").str.strip() == value
    ]

    if scoped.empty:
        return {"scope": scope, "n_products": 0}

    reviewed = set(catalog.comments_by_product)
    pids = [
        int(x)
        for x in scoped["product_id"].dropna().tolist()
        if int(x) in reviewed
    ]

    if not pids:
        return {"scope": scope, "n_products": 0}

    rows = []

    for pid in pids:
        s = review_stats(catalog, pid, light=True)
        row = catalog.product(pid) or {}
        s.update({
            "title": row.get("title_fa", ""),
            "price": row.get("price_clean"),
            "brand": row.get("brand_norm", ""),
            "rate": row.get("product_rate_clean"),
        })
        rows.append(s)

    df = pd.DataFrame(rows)

    total_reviews = int(df["n_reviews"].sum())
    total_complaints = int(df["complaint_count"].sum())

    total_labeled_recommendations = int(df["n_labeled_recommendation"].sum())

    if total_labeled_recommendations:
        weighted_rec_rate = float(
            (df["rec_rate"].fillna(0) * df["n_labeled_recommendation"]).sum()
            / total_labeled_recommendations
        )
    else:
        weighted_rec_rate = None

    product_mean_rec_rate = (
        float(df["rec_rate"].mean())
        if df["rec_rate"].notna().any()
        else None
    )

    brand_sat = []

    if kind in ("category1_norm", "category2_norm", "sub_category_norm"):
        for brand, g in df.groupby("brand", dropna=True):
            brand = str(brand).strip()

            if not brand or brand in _GENERIC_VALUES:
                continue

            n_reviews = int(g["n_reviews"].sum())
            n_labeled = int(g["n_labeled_recommendation"].sum())

            if len(g) < 2 or n_reviews < 5 or n_labeled < 5:
                continue

            rec_rate = float(
                (g["rec_rate"].fillna(0) * g["n_labeled_recommendation"]).sum()
                / n_labeled
            )

            valid_rate = g["avg_rate"].notna()
            if valid_rate.any():
                rate_weights = g.loc[valid_rate, "n_reviews"].clip(lower=1)
                avg_rate = float(
                    np.average(
                        g.loc[valid_rate, "avg_rate"],
                        weights=rate_weights,
                    )
                )
            else:
                avg_rate = None

            brand_sat.append({
                "brand": brand,
                "n_products": int(len(g)),
                "n_reviews": n_reviews,
                "n_labeled_recommendation": n_labeled,
                "review_weighted_rec_rate": round(rec_rate, 3),
                "review_weighted_avg_rate": round(avg_rate, 2) if avg_rate is not None else None,
            })

        brand_sat = sorted(
            brand_sat,
            key=lambda x: (
                x["n_reviews"],
                x["review_weighted_rec_rate"],
            ),
            reverse=True,
        )[:8]

    threshold = int(min_comments)
    low = pd.DataFrame()

    for candidate_threshold in [threshold, 3, 2]:
        if candidate_threshold > threshold:
            continue

        candidate = df[
            (df["n_reviews"] >= candidate_threshold)
            & df["rec_rate"].notna()
        ].sort_values(
            ["rec_rate", "n_reviews"],
            ascending=[True, False],
        )

        if len(candidate):
            low = candidate.head(8)
            threshold = candidate_threshold
            break

    low_list = [
        {
            "product_id": int(r.product_id),
            "title": r.title,
            "rec_rate": r.rec_rate,
            "n_reviews": int(r.n_reviews),
            "rate": r.rate,
            "price": r.price,
        }
        for r in low.itertuples(index=False)
    ]

    return {
        "scope": scope,
        "product_ids": pids,
        "n_products": int(len(df)),
        "n_reviews": total_reviews,
        "n_labeled_recommendations": total_labeled_recommendations,
        "total_complaints": total_complaints,
        "complaints_per_100_reviews": (
            round(100 * total_complaints / total_reviews, 1)
            if total_reviews
            else None
        ),
        "avg_rate": (
            round(float(df["avg_rate"].mean()), 1)
            if df["avg_rate"].notna().any()
            else None
        ),
        "avg_rec_rate_product_mean": (
            round(product_mean_rec_rate, 3)
            if product_mean_rec_rate is not None
            else None
        ),
        "review_weighted_rec_rate": (
            round(weighted_rec_rate, 3)
            if weighted_rec_rate is not None
            else None
        ),
        "low_recommendation_min_reviews": int(threshold) if len(low) else None,
        "brand_satisfaction": brand_sat,
        "low_recommendation_products": low_list,
        "top_complaint_terms": _top_complaint_terms(catalog, pids),
    }



# ---- comparison evidence quality guards --------------------------------
_PROCON_EMPTY_PHRASES = {
    "",
    "ندارد",
    "نداره",
    "ندارم",
    "نداشت",
    "نداشتم",
    "هیچ",
    "هیچی",
    "موردی ندارد",
    "مورد خاصی ندارد",
    "نکته منفی ندارد",
    "نقطه ضعف ندارد",
    "عیبی ندارد",
    "ایرادی ندارد",
    "مشکلی ندارد",
    "ندارد ندارد",
    "ندارد، ندارد",
}

_POSITIVE_EVIDENCE_HINTS = (
    "خوب", "عالی", "راضی", "رضایت", "مناسب", "جذاب",
    "ارزش خرید", "پیشنهاد", "با کیفیت", "باکیفیت", "سرگرم",
    "خوشش اومد", "خوشش آمد", "کیفیتش مناسبه",
)

# Strong negative wording that is meaningful on its own.
# Generic negation words such as "ندارد" / "نیست" are intentionally excluded
# because raw disadvantages sometimes contain only "ندارد" and get appended
# to an otherwise positive review.
_STRONG_NEGATIVE_EVIDENCE_HINTS = (
    "بد", "ضعیف", "مشکل", "ایراد", "عیب", "خراب", "معیوب",
    "ناراضی", "نامناسب", "تقلبی", "فیک", "شکسته", "شکست",
    "پاره", "افتضاح", "بی کیفیت", "بی‌کیفیت", "کیفیت پایین",
    "تیز", "خیلی کوچیک", "خیلی کوچک", "کوچیکه", "کوچکه",
    "زمان بر", "زمان‌بر", "سخت", "شل", "بو میده", "بو می‌دهد",
    "نمی ارزد", "نمی‌ارزد", "ارزش نداره", "ارزش ندارد",
)


def _procon_flags(row: dict) -> dict:
    text = pt.normalize(row.get("text", "") or "")
    compact = re.sub(r"[\s،,؛;.!؟?]+", " ", text).strip()

    placeholder = (
        compact in _PROCON_EMPTY_PHRASES
        or compact.replace(" ", "") in {
            "ندارد",
            "ندارم",
            "نداشت",
            "نداشتم",
            "نداردندارد",
            "هیچ",
            "هیچی",
        }
    )

    has_pos = any(h in compact for h in _POSITIVE_EVIDENCE_HINTS)
    has_strong_neg = any(
        h in compact
        for h in _STRONG_NEGATIVE_EVIDENCE_HINTS
    )

    rate = row.get("rate")
    status = str(row.get("recommendation_status", ""))

    metadata_positive = (
        status == "recommended"
        and rate is not None
        and pd.notna(rate)
        and float(rate) >= 4.0
    )

    metadata_negative = (
        status == "not_recommended"
        or (
            rate is not None
            and pd.notna(rate)
            and float(rate) <= 2.5
        )
    )

    return {
        "text": compact,
        "placeholder": bool(placeholder),
        "has_positive_language": bool(has_pos),
        "has_strong_negative_language": bool(has_strong_neg),
        "metadata_positive": bool(metadata_positive),
        "metadata_negative": bool(metadata_negative),
        "evidence_type": str(row.get("evidence_type", "")),
    }


def _accept_positive_evidence(row: dict) -> bool:
    f = _procon_flags(row)

    if not f["text"] or f["placeholder"]:
        return False

    # Do not label an explicitly negative sentence as a strength.
    if (
        f["has_strong_negative_language"]
        and not f["has_positive_language"]
    ):
        return False

    if f["evidence_type"] == "advantage":
        return bool(
            f["has_positive_language"]
            or f["metadata_positive"]
            or not f["has_strong_negative_language"]
        )

    return bool(
        f["has_positive_language"]
        or (
            f["metadata_positive"]
            and not f["has_strong_negative_language"]
        )
    )


def _accept_negative_evidence(row: dict) -> bool:
    f = _procon_flags(row)

    if not f["text"] or f["placeholder"]:
        return False

    # A visibly positive review must not appear under "weaknesses" just
    # because its raw metadata contains "disadvantages=ندارد" or an
    # inconsistent recommendation label.
    if (
        f["has_positive_language"]
        and not f["has_strong_negative_language"]
    ):
        return False

    # For retrieved full reviews, require actual negative wording or a
    # consistent negative metadata signal with no positive wording.
    if f["evidence_type"] == "retrieved_review":
        return bool(
            f["has_strong_negative_language"]
            or (
                f["metadata_negative"]
                and not f["has_positive_language"]
            )
        )

    # A raw disadvantages field is accepted when it is non-empty/non-placeholder
    # and not visibly positive. Strong negative wording is preferred, while a
    # concise disadvantage such as "قیمت" or "وزن" is still allowed.
    if f["evidence_type"] == "disadvantage":
        return bool(
            not f["has_positive_language"]
            and not f["placeholder"]
        )

    return bool(
        f["has_strong_negative_language"]
        or (
            f["metadata_negative"]
            and not f["has_positive_language"]
        )
    )


# ---- citation verification + answer container ---------------------------
_CITE_P = re.compile(r"\[محصول\s*(\d+)\]")
_CITE_R = re.compile(r"\[بازبینی\s*(\d+)\]")
_MISSING = ("اطلاعات کافی موجود نیست", "موجود نیست")


def verify_citations(text, allowed_products, allowed_reviews) -> str:
    text = _CITE_P.sub(lambda m: m.group(0) if int(m.group(1)) in allowed_products else "", text)
    text = _CITE_R.sub(lambda m: m.group(0) if int(m.group(1)) in allowed_reviews else "", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


@dataclass
class Answer:
    intent: str
    query: str
    text: str
    citations: list = field(default_factory=list)
    review_citations: list = field(default_factory=list)
    sources: list = field(default_factory=list)
    missing_info: bool = False
    tier: str = "extractive"
    latency_s: float = 0.0
    cost_usd: float = 0.0
    needs_clarification: bool = False


class ShoppingAssistant:
    def __init__(self, catalog: Catalog, product_index, review_retriever, llm=None, final_k: int = 8):
        self.c = catalog
        self.pidx = product_index
        self.rrev = review_retriever
        self.llm = llm or LLM(mode="extractive")
        self.router = IntentRouter(catalog)
        self.final_k = final_k
        self.generated_rejections = 0

    def answer(self, query) -> Answer:
        # Per-answer cost must start at zero. Without this reset, a deterministic
        # answer that does not call the LLM can inherit the previous hosted
        # request's cost in Phase-4 per-query accounting.
        self.llm.last_cost_usd = 0.0

        t0 = time.time()
        route = self.router.route(query)
        if route.needs_clarification:
            a = Answer(route.intent, query, "برای مقایسه، دو محصول را با شناسه مشخص کنید.",
                       needs_clarification=True)
        elif route.intent == "discovery":
            a = self._discover(query, route.filters)
        elif route.intent == "product_qa":
            a = self._qa(query, route.product_id)
        elif route.intent == "comparison":
            a = self._compare(query, route.product_ids)
        else:
            a = self._managerial(query, route.scope)
        a.latency_s = round(time.time() - t0, 3)
        a.cost_usd = getattr(self.llm, "last_cost_usd", 0.0)
        return a

    def _gen(self, system, user, extractive, allowed_p, allowed_r):
        g = self.llm.generate(system, user)

        if not g or not g.strip():
            return extractive, "extractive"

        clean = verify_citations(g, allowed_p, allowed_r)

        # A citation-only response is not a useful generated answer.
        without_cites = re.sub(
            r"\[(?:محصول|بازبینی)\s*\d+\]",
            " ",
            clean,
        )
        meaningful_tokens = [
            t for t in pt.tokenize(without_cites)
            if len(t) > 1
        ]

        if len(meaningful_tokens) < 8:
            return extractive, "extractive"

        if (allowed_p or allowed_r) and not re.search(
            r"\[(?:محصول|بازبینی)\s*\d+\]",
            clean,
        ):
            self.generated_rejections += 1
            return extractive, "extractive"

        # A completion that ends on a connector/punctuation fragment is usually
        # token-truncated.  Prefer the complete deterministic answer instead of
        # exposing a visibly cut-off LLM response.
        tail = pt.normalize(clean).rstrip()
        last_token = pt.tokenize(tail)[-1] if pt.tokenize(tail) else ""

        if (
            (tail and tail[-1] in "،؛,:-")
            or last_token in {"و", "یا", "که", "اما", "با", "از", "به", "در", "برای"}
        ):
            self.generated_rejections += 1
            return extractive, "extractive"

        return clean, "llm"

    def _discover(self, query, filters):
        hits = self.pidx.search(
            query,
            filters,
            k=max(40, self.final_k * 5),
        )

        if not hits:
            return Answer(
                "discovery",
                query,
                "هیچ محصولی با این فیلترها یافت نشد (اطلاعات کافی موجود نیست).",
                missing_info=True,
            )

        qn = pt.normalize(query)
        cheap_pref = any(
            w in qn
            for w in ("اقتصادی", "ارزان", "ارزون", "قیمت مناسب", "مقرون به صرفه", "مقرون‌به‌صرفه")
        )
        satisfaction_pref = any(
            w in qn
            for w in (
                "رضایت", "راضی", "توصیه کاربران", "پیشنهاد کاربران",
                "نظر کاربران خوب", "خریداران راضی",
            )
        )
        quality_pref = any(
            w in qn
            for w in ("باکیفیت", "کیفیت خوب", "بهترین", "خوب")
        ) or satisfaction_pref

        # The retrieval stage finds relevant candidates.  When the user asks
        # about affordability, quality, or user satisfaction, reranking uses
        # transparent product facts plus recommendation evidence from sampled
        # reviews.  This makes "رضایت کاربران" an actual data signal rather
        # than merely a text keyword.
        if cheap_pref or quality_pref:
            retrieval = _minmax(np.array([h["score"] for h in hits], dtype=float))

            prices = np.array([
                float(h["price"]) if h["price"] is not None and h["price"] > 0 else np.nan
                for h in hits
            ])
            price_score = np.full(len(hits), 0.5, dtype=float)
            good_price = np.isfinite(prices)

            if good_price.sum() >= 2:
                logged = np.log1p(prices[good_price])
                price_score[good_price] = 1.0 - _minmax(logged)

            rates = np.array([
                float(h["rate"]) if h["rate"] is not None else np.nan
                for h in hits
            ])
            quality_score = np.full(len(hits), 0.5, dtype=float)
            good_rate = np.isfinite(rates)

            if good_rate.sum() >= 2:
                quality_score[good_rate] = _minmax(rates[good_rate])

            counts = np.log1p(
                np.array([h.get("rate_count", 0) for h in hits], dtype=float)
            )
            confidence = _minmax(counts)

            review_rec_rates = []
            review_support = []

            for h in hits:
                stats = review_stats(self.c, int(h["product_id"]))
                rec = stats.get("rec_rate")
                n_lab = int(stats.get("n_labeled_recommendation", 0) or 0)
                review_rec_rates.append(np.nan if rec is None else float(rec))
                review_support.append(n_lab)
                h["review_rec_rate"] = rec
                h["review_labeled_count"] = n_lab

            review_rec_rates = np.array(review_rec_rates, dtype=float)
            satisfaction_score = np.full(len(hits), 0.5, dtype=float)
            good_rec = np.isfinite(review_rec_rates)

            if good_rec.sum() >= 2:
                satisfaction_score[good_rec] = _minmax(review_rec_rates[good_rec])

            review_confidence = _minmax(
                np.log1p(np.array(review_support, dtype=float))
            )

            # Quality/satisfaction claims are less trustworthy when based on a
            # single vote/review.  Keep relevance dominant, but penalize very
            # low-support candidates whenever enough better-supported options
            # exist in the same retrieved pool.
            product_support = np.array([
                max(
                    int(h.get("rate_count", 0) or 0),
                    int(h.get("review_labeled_count", 0) or 0),
                )
                for h in hits
            ], dtype=int)
            reliable_support = product_support >= 3

            if satisfaction_pref:
                # Retrieval stays dominant, but real review recommendation
                # evidence becomes a visible ranking signal.
                final_score = (
                    0.60 * retrieval
                    + 0.18 * satisfaction_score
                    + 0.07 * review_confidence
                )

                if cheap_pref:
                    final_score += 0.08 * price_score

                if quality_pref:
                    final_score += 0.05 * quality_score

                final_score += 0.02 * confidence

            else:
                final_score = 0.72 * retrieval

                if cheap_pref and quality_pref:
                    final_score += (
                        0.13 * price_score
                        + 0.10 * quality_score
                        + 0.05 * confidence
                    )
                elif cheap_pref:
                    final_score += 0.23 * price_score + 0.05 * confidence
                else:
                    final_score += 0.23 * quality_score + 0.05 * confidence

            if quality_pref and reliable_support.sum() >= self.final_k:
                final_score = final_score - np.where(
                    reliable_support,
                    0.0,
                    0.12,
                )

            order = np.argsort(-final_score, kind="stable")
            hits = [hits[int(i)] for i in order]

            for rank, h in enumerate(hits, 1):
                h["rank"] = rank
                reasons = []

                if cheap_pref and h["price"] is not None:
                    reasons.append("قیمت مناسب‌تر در میان نامزدهای بازیابی‌شده")

                if quality_pref and h["rate"] is not None:
                    reasons.append(
                        f"امتیاز {h['rate']} با {h.get('rate_count', 0)} رأی"
                    )

                if satisfaction_pref:
                    rec = h.get("review_rec_rate")
                    n_lab = int(h.get("review_labeled_count", 0) or 0)

                    if rec is not None and n_lab:
                        reasons.append(
                            f"{rec:.0%} توصیه در {n_lab} بازبینی دارای وضعیت معتبر"
                        )
                    else:
                        reasons.append(
                            "شواهد وضعیت پیشنهاد کاربران در نمونه کافی نیست"
                        )

                support = max(
                    int(h.get("rate_count", 0) or 0),
                    int(h.get("review_labeled_count", 0) or 0),
                )
                h["evidence_support"] = support

                if quality_pref and support < 3:
                    reasons.append("پشتیبانی داده محدود است")

                h["preference_reason"] = "؛ ".join(reasons)

        ev = hits[:self.final_k]
        allowed = {h["product_id"] for h in ev}

        lines = []

        for h in ev:
            reason = h.get("preference_reason", "")
            reason_text = f"؛ دلیل: {reason}" if reason else ""
            lines.append(
                f"{h['rank']}. [محصول {h['product_id']}] {h['title']} — "
                f"امتیاز {h['rate'] if h['rate'] is not None else 'نامشخص'}، "
                f"قیمت {pt.format_toman(h['price'])} تومان، "
                f"برند {h['brand'] or 'نامشخص'}{reason_text}"
            )

        extractive = "پیشنهادهای برتر بر اساس درخواست شما:\n" + "\n".join(lines)

        text, tier = self._gen(
            prompts.DISCOVERY_SYSTEM.format(k=len(ev)),
            f"درخواست: {query}\nفیلترها: {filters}\n\nمدارک:\n"
            f"{prompts.evidence_products(ev)}\n\nپاسخ:",
            extractive,
            allowed,
            set(),
        )

        return Answer(
            "discovery",
            query,
            text,
            citations=list(allowed),
            sources=ev,
            tier=tier,
            missing_info=any(p in text for p in _MISSING),
        )

    def _qa(self, query, pid):
        if pid is None:
            return Answer(
                "product_qa",
                query,
                "محصول شناسایی نشد؛ شناسه یا نام کامل را ذکر کنید.",
                missing_info=True,
            )

        prod = self.c.product(pid) or {}
        title = prod.get("title_fa", "")
        candidates = self.rrev.retrieve(
            query,
            pid,
            k=max(20, self.final_k * 3),
        )

        polarity = self.rrev._polarity(query)

        def evidence_flags(row):
            txt = pt.normalize(row.get("text", ""))
            has_neg_text = any(cue in txt for cue in self.rrev._NEGATIVE_CUES)
            has_pos_text = any(cue in txt for cue in self.rrev._POSITIVE_CUES)

            if row.get("has_disadvantage", False):
                has_neg_text = True

            if row.get("has_advantage", False):
                has_pos_text = True

            return has_pos_text, has_neg_text

        if polarity == "negative":
            focused = []

            for h in candidates:
                _, has_neg = evidence_flags(h)
                rate = h.get("rate")
                status = h.get("recommendation_status")

                metadata_negative = (
                    status == "not_recommended"
                    and rate is not None
                    and float(rate) <= 3.0
                )

                if has_neg or metadata_negative:
                    focused.append(h)

            hits = focused[:self.final_k]

        elif polarity == "positive":
            focused = []

            for h in candidates:
                has_pos, has_neg = evidence_flags(h)
                rate = h.get("rate")
                status = h.get("recommendation_status")

                metadata_positive = (
                    status == "recommended"
                    and rate is not None
                    and float(rate) >= 4.0
                )

                # A clearly negative review is not reused as positive evidence
                # just because its recommendation label is positive.
                if (has_pos or metadata_positive) and not has_neg:
                    focused.append(h)

            hits = focused[:self.final_k]

        else:
            hits = candidates[:self.final_k]

        if not hits:
            # Do not force opposite-polarity evidence into an answer.
            if candidates and polarity in ("negative", "positive"):
                direction = "منفی" if polarity == "negative" else "مثبت"
                return Answer(
                    "product_qa",
                    query,
                    f"برای [محصول {pid}] بازبینی مرتبط وجود دارد، اما شواهد {direction} "
                    f"کافی برای پاسخ مطمئن پیدا نشد (اطلاعات کافی موجود نیست).",
                    citations=[pid],
                    sources=candidates[:self.final_k],
                    missing_info=True,
                )

            return Answer(
                "product_qa",
                query,
                f"برای [محصول {pid}] بازبینی‌ای نیست (اطلاعات کافی موجود نیست).",
                citations=[pid],
                missing_info=True,
            )

        allowed_r = {h["comment_id"] for h in hits}
        facts = review_stats(self.c, pid)

        if facts["n_reviews"]:
            n_lab = facts.get("n_labeled_recommendation", 0)
            if n_lab:
                head = (
                    f"دربارهٔ [محصول {pid}] ({title}): {facts['n_reviews']} بازبینی در نمونه داریم؛ "
                    f"از {n_lab} بازبینی دارای وضعیت پیشنهاد، {facts['rec_rate']:.0%} توصیه، "
                    f"{facts['not_rec_rate']:.0%} عدم توصیه و {facts['no_idea_rate']:.0%} بدون نظر قطعی بوده‌اند. "
                    f"[محصول {pid}]"
                )
            else:
                head = (
                    f"دربارهٔ [محصول {pid}] ({title}): {facts['n_reviews']} بازبینی در نمونه داریم، "
                    f"اما وضعیت پیشنهاد معتبر کافی ثبت نشده است. [محصول {pid}]"
                )
        else:
            head = f"[محصول {pid}] ({title}):"

        if polarity == "negative":
            section = "ایرادها، مشکلات و نکات منفی مرتبط که در بازبینی‌های بازیابی‌شده دیده می‌شود:"
        elif polarity == "positive":
            section = "نقاط قوت و تجربه‌های مثبت مرتبط در بازبینی‌های بازیابی‌شده:"
        else:
            section = "بازبینی‌های مرتبط با پرسش:"

        evidence_lines = [
            f"- «{h['text']}» [بازبینی {h['comment_id']}]"
            for h in hits
        ]

        extractive = head + "\n" + section + "\n" + "\n".join(evidence_lines)

        selected_rates = [
            float(h["rate"])
            for h in hits
            if h.get("rate") is not None and pd.notna(h.get("rate"))
        ]

        if selected_rates:
            selected_stats = (
                f"تعداد شواهد انتخاب‌شده: {len(hits)}؛ "
                f"کمترین امتیاز همین شواهد: {min(selected_rates):g}؛ "
                f"بیشترین امتیاز همین شواهد: {max(selected_rates):g}."
            )
        else:
            selected_stats = (
                f"تعداد شواهد انتخاب‌شده: {len(hits)}؛ "
                "امتیاز عددی معتبر برای این شواهد کافی نیست."
            )

        text, tier = self._gen(
            prompts.QA_SYSTEM.format(max_lines=8),
            f"محصول: [محصول {pid}] {title}\n"
            f"پرسش: {query}\n"
            f"آمار قطعی شواهد انتخاب‌شده: {selected_stats}\n\n"
            f"مدارک:\n{prompts.evidence_reviews(hits)}\n\nپاسخ:",
            extractive,
            {pid},
            allowed_r,
        )

        # Universal claims were a real failure in an earlier run: the model
        # stated that all selected reviews had rating >= 3 while one cited
        # review had rating 0.  The deterministic fallback is safer than
        # keeping a fluent but contradictory aggregate claim.
        if tier == "llm" and re.search(
            r"(?:همه|تمام)\s*(?:ی|ٔ)?\s*(?:بازبینی|بازبینی‌ها|نظر|نظرها)|"
            r"هیچ\s*(?:بازبینی|نظر)",
            pt.normalize(text),
        ):
            self.generated_rejections += 1
            text, tier = extractive, "extractive"

        return Answer(
            "product_qa",
            query,
            text,
            citations=[pid],
            review_citations=list(allowed_r),
            sources=hits,
            tier=tier,
            missing_info=any(p in text for p in _MISSING),
        )

    def _compare(self, query, pids):
        facts = []
        positive_by_product = {}
        negative_by_product = {}
        all_review_rows = []

        def _dedupe_rows(rows, k=3):
            out = []
            seen = set()

            for row in rows:
                cid = int(row["comment_id"])

                if cid in seen:
                    continue

                seen.add(cid)
                out.append(row)

                if len(out) >= k:
                    break

            return out

        def _review_polarity(row):
            text_n = pt.normalize(row.get("text", ""))
            rate = row.get("rate")
            status = row.get("recommendation_status")

            has_neg = any(
                cue in text_n
                for cue in self.rrev._NEGATIVE_CUES
            )
            has_pos = any(
                cue in text_n
                for cue in self.rrev._POSITIVE_CUES
            )

            if row.get("has_disadvantage", False):
                has_neg = True

            if row.get("has_advantage", False):
                has_pos = True

            if status == "not_recommended":
                has_neg = True

            if (
                rate is not None
                and pd.notna(rate)
                and float(rate) <= 2.5
            ):
                has_neg = True

            if (
                status == "recommended"
                and rate is not None
                and pd.notna(rate)
                and float(rate) >= 4.0
            ):
                has_pos = True

            if has_neg:
                return "negative"

            if has_pos:
                return "positive"

            return "neutral"

        for pid in pids:
            p = self.c.product(pid) or {}
            stats = review_stats(self.c, pid)

            facts.append({
                "product_id": int(pid),
                "title": p.get("title_fa", "نامشخص"),
                "price": p.get("price_clean"),
                "rate": p.get("product_rate_clean"),
                "brand": p.get("brand_norm", ""),
                "stats": stats,
            })

            positives = [
                dict(r, evidence_type="advantage")
                for r in stats.get("top_advantages", [])
            ]
            negatives = [
                dict(r, evidence_type="disadvantage")
                for r in stats.get("top_disadvantages", [])
            ]

            # Aspect-aware retrieval supplements explicit advantage/disadvantage
            # fields when those fields are sparse.
            aspect_hits = self.rrev.retrieve(
                query,
                pid,
                k=max(6, min(10, self.final_k + 2)),
            )

            for row in aspect_hits:
                pol = _review_polarity(row)

                if pol == "negative":
                    negatives.append(dict(row, evidence_type="retrieved_review"))
                elif pol == "positive":
                    positives.append(dict(row, evidence_type="retrieved_review"))

            positives = [
                row
                for row in positives
                if _accept_positive_evidence(row)
            ]
            negatives = [
                row
                for row in negatives
                if _accept_negative_evidence(row)
            ]

            positives = _dedupe_rows(positives, k=3)
            negatives = _dedupe_rows(negatives, k=3)

            positive_by_product[int(pid)] = positives
            negative_by_product[int(pid)] = negatives
            all_review_rows.extend(positives)
            all_review_rows.extend(negatives)

        # Global de-duplication is only for the citation whitelist/source payload.
        review_rows = []
        seen_reviews = set()

        for row in all_review_rows:
            cid = int(row["comment_id"])

            if cid in seen_reviews:
                continue

            seen_reviews.add(cid)
            review_rows.append(row)

        allowed_p = {
            int(f["product_id"])
            for f in facts
        }
        allowed_r = {
            int(r["comment_id"])
            for r in review_rows
        }

        fact_lines = ["۱) واقعیت‌های مستقیم محصول"]

        for f in facts:
            rec = f["stats"].get("rec_rate")
            complaints = int(f["stats"].get("complaint_count", 0) or 0)

            fact_lines.append(
                f"- [محصول {f['product_id']}] {f['title']} | "
                f"برند: {f['brand'] or 'نامشخص'} | "
                f"قیمت: {pt.format_toman(f['price'])} تومان | "
                f"امتیاز محصول: {f['rate'] if f['rate'] is not None else 'نامشخص'} | "
                f"تعداد بازبینی: {f['stats']['n_reviews']} | "
                f"وضعیت پیشنهاد معتبر: {f['stats'].get('n_labeled_recommendation', 0)} | "
                f"نرخ توصیه: {f'{rec:.0%}' if rec is not None else 'نامشخص'} | "
                f"بازبینی‌های دارای نشانهٔ شکایت: {complaints}"
            )

        evidence_lines = ["۲) شواهد بازبینی‌های کاربران"]

        for f in facts:
            pid = int(f["product_id"])
            evidence_lines.append(
                f"- [محصول {pid}] نقاط قوت / تجربه‌های مثبت:"
            )

            positives = positive_by_product.get(pid, [])

            if positives:
                for row in positives:
                    evidence_lines.append(
                        f"  - «{row['text']}» [بازبینی {row['comment_id']}]"
                    )
            else:
                evidence_lines.append(
                    "  - شواهد مثبت کافی در نمونه پیدا نشد."
                )

            evidence_lines.append(
                f"- [محصول {pid}] نقاط ضعف / تجربه‌های منفی:"
            )

            negatives = negative_by_product.get(pid, [])

            if negatives:
                for row in negatives:
                    evidence_lines.append(
                        f"  - «{row['text']}» [بازبینی {row['comment_id']}]"
                    )
            else:
                evidence_lines.append(
                    "  - نقطهٔ ضعف قابل اتکای کافی در نمونه پیدا نشد."
                )

        inference_lines = ["۳) جمع‌بندی / استنباط از داده‌های بالا"]

        rec_candidates = [
            f for f in facts
            if f["stats"].get("rec_rate") is not None
        ]
        priced = [
            f for f in facts
            if f["price"] is not None
            and pd.notna(f["price"])
            and float(f["price"]) > 0
        ]

        if rec_candidates:
            best_value = max(
                f["stats"]["rec_rate"]
                for f in rec_candidates
            )
            best_rec = [
                f for f in rec_candidates
                if abs(
                    f["stats"]["rec_rate"] - best_value
                ) < 1e-12
            ]

            if len(best_rec) == 1:
                winner = best_rec[0]
                inference_lines.append(
                    f"- اگر رضایت کاربران اولویت اصلی باشد، "
                    f"[محصول {winner['product_id']}] در دادهٔ فعلی "
                    f"نرخ توصیهٔ بالاتری دارد "
                    f"({winner['stats']['rec_rate']:.0%})."
                )
            else:
                tied = " و ".join(
                    f"[محصول {f['product_id']}]"
                    for f in best_rec
                )
                inference_lines.append(
                    f"- از نظر نرخ توصیه، {tied} در دادهٔ فعلی برابرند "
                    f"({best_value:.0%})؛ شواهد قوت/ضعف و قیمت برای تصمیم "
                    f"نهایی مهم می‌شوند."
                )

        if priced:
            cheapest = min(
                priced,
                key=lambda x: float(x["price"]),
            )
            inference_lines.append(
                f"- اگر قیمت اولویت اصلی باشد، "
                f"[محصول {cheapest['product_id']}] با قیمت "
                f"{pt.format_toman(cheapest['price'])} تومان ارزان‌تر است."
            )

        complaint_candidates = [
            f for f in facts
            if int(f["stats"].get("n_reviews", 0) or 0) > 0
        ]

        if len(complaint_candidates) >= 2:
            complaint_rates = []

            for f in complaint_candidates:
                n_reviews = int(f["stats"].get("n_reviews", 0) or 0)
                n_complaints = int(
                    f["stats"].get("complaint_count", 0) or 0
                )
                complaint_rates.append(
                    (n_complaints / max(1, n_reviews), f)
                )

            complaint_rates.sort(
                key=lambda x: x[0]
            )
            low_rate, low_item = complaint_rates[0]
            high_rate, _ = complaint_rates[-1]

            if high_rate - low_rate >= 0.10:
                inference_lines.append(
                    f"- در نمونهٔ فعلی، [محصول {low_item['product_id']}] "
                    f"سهم کمتری از بازبینی‌های دارای نشانهٔ شکایت دارد "
                    f"({low_rate:.0%})."
                )

        if len(inference_lines) == 1:
            inference_lines.append(
                "- برای نتیجه‌گیری قطعی اطلاعات کافی موجود نیست."
            )

        extractive = "\n".join(
            fact_lines
            + [""]
            + evidence_lines
            + [""]
            + inference_lines
        )

        # Comparison stays deterministic: every factual statement and
        # recommendation is computed from the selected facts/reviews.
        text, tier = extractive, "extractive"

        return Answer(
            "comparison",
            query,
            text,
            citations=list(allowed_p),
            review_citations=list(allowed_r),
            sources={
                "facts": facts,
                "positive_reviews": positive_by_product,
                "negative_reviews": negative_by_product,
            },
            tier=tier,
            missing_info=any(
                p in text
                for p in _MISSING
            ),
        )

    def _managerial(self, query, scope):
        summary = managerial_summary(self.c, scope)

        if not summary.get("n_products"):
            return Answer(
                "managerial",
                query,
                f"برای دامنهٔ {scope} داده‌ای نیست (اطلاعات کافی موجود نیست).",
                missing_info=True,
            )

        terms = " ".join(
            t["term"]
            for t in summary.get("top_complaint_terms", [])[:3]
        )

        # Start from low-recommendation products because they are the most
        # plausible source of complaint evidence, then widen to the category.
        low_pids = [
            int(p["product_id"])
            for p in summary.get("low_recommendation_products", [])
        ]
        pids = low_pids + [
            int(pid)
            for pid in summary.get("product_ids", [])
            if int(pid) not in set(low_pids)
        ]

        complaint_reviews = []

        if terms and pids:
            for pid in pids[:20]:
                complaint_reviews += self.rrev.retrieve(
                    terms,
                    pid,
                    k=2,
                )

        def _is_negative_complaint(row):
            text_n = pt.normalize(row.get("text", ""))
            direct_negative = any(
                cue in text_n
                for cue in self.rrev._NEGATIVE_CUES
            )

            rate = row.get("rate")
            low_rate = (
                rate is not None
                and pd.notna(rate)
                and float(rate) <= 2.5
            )

            return bool(
                direct_negative
                or row.get("has_disadvantage", False)
                or row.get("recommendation_status") == "not_recommended"
                or low_rate
            )

        # Deduplicate and retain only genuinely negative evidence.  Earlier
        # runs accidentally displayed positive sentences under "complaints".
        tmp = []
        seen = set()

        for r in complaint_reviews:
            cid = int(r["comment_id"])

            if cid in seen or not _is_negative_complaint(r):
                continue

            seen.add(cid)
            tmp.append(r)

        complaint_reviews = tmp[:8]

        allowed_p = {
            int(p["product_id"])
            for p in summary["low_recommendation_products"]
        }
        allowed_r = {
            int(r["comment_id"])
            for r in complaint_reviews
        }

        weighted_rec = summary.get("review_weighted_rec_rate")
        threshold = summary.get("low_recommendation_min_reviews")

        ev = [
            f"دامنه: {scope.get('value')}",
            f"تعداد محصول دارای بازبینی: {summary['n_products']} | "
            f"تعداد بازبینی: {summary['n_reviews']} | "
            f"وضعیت پیشنهاد معتبر: {summary.get('n_labeled_recommendations', 0)} | "
            f"میانگین امتیاز بازبینی: {summary['avg_rate']} | "
            f"نرخ توصیهٔ وزن‌دار در بین وضعیت‌های معتبر: "
            f"{f'{weighted_rec:.1%}' if weighted_rec is not None else 'نامشخص'}",
            "شکایت‌های پرتکرار: "
            + (
                ", ".join(
                    f"{t['term']}({t['count']})"
                    for t in summary["top_complaint_terms"]
                )
                or "نامشخص"
            ),
        ]

        brands = summary.get("brand_satisfaction") or []

        if brands:
            ev.append("الگوی رضایت برندها:")

            for b in brands[:5]:
                ev.append(
                    f"- {b['brand']} | {b['n_products']} محصول | "
                    f"{b['n_reviews']} بازبینی | "
                    f"نرخ توصیهٔ وزن‌دار {b['review_weighted_rec_rate']:.1%}"
                )

        if threshold is not None:
            ev.append(
                f"محصولات با نرخ توصیهٔ پایین "
                f"(حداقل {threshold} بازبینی در نمونه):"
            )
        else:
            ev.append(
                "محصولات با نرخ توصیهٔ پایین: دادهٔ کافی برای آستانهٔ حداقل دو بازبینی موجود نیست."
            )

        ev += [
            f"- [محصول {p['product_id']}] {p['title']} | "
            f"نرخ توصیه {p['rec_rate']:.1%} | {p['n_reviews']} بازبینی"
            for p in summary["low_recommendation_products"]
        ]

        evidence_text = "\n".join(ev)

        if complaint_reviews:
            extractive = (
                evidence_text
                + "\n\nنمونهٔ شکایت‌های قابل ردیابی:\n"
                + "\n".join(
                    f"- «{r['text']}» [بازبینی {r['comment_id']}]"
                    for r in complaint_reviews
                )
            )
        else:
            extractive = evidence_text

        # Managerial analytics are rendered deterministically from measured
        # aggregates and cited complaint reviews. This prevents unsupported
        # generalizations about price/quality while reducing API cost/latency.
        text, tier = extractive, "extractive"

        return Answer(
            "managerial",
            query,
            text,
            citations=list(allowed_p),
            review_citations=list(allowed_r),
            sources=[summary],
            tier=tier,
            missing_info=any(p in text for p in _MISSING),
        )


# ---- non-LLM lexical baseline (evaluation control) ----------------------
class LexicalBaseline:
    """Control: token-overlap product retrieval + arithmetic, no embeddings, no LLM."""

    def __init__(self, catalog: Catalog):
        self.c = catalog

    def discover(self, query, k: int = 5):
        f = extract_filters(self.c, query)
        mask = product_filter_mask(self.c.products, f)
        sub = self.c.products[mask]
        q = set(pt.tokenize(query))
        sub = sub.assign(_ov=sub["product_text_norm"].map(lambda t: len(q & set(pt.tokenize_norm(t)))))
        sub = sub.sort_values(["_ov", "product_rate_clean", "rate_count"], ascending=False).head(k)
        return sub[["product_id", "title_fa", "price_clean", "product_rate_clean"]]


def build_assistant(llm=None, final_k: int = 8) -> ShoppingAssistant:
    """Build from cached artifacts when available, otherwise from notebook data."""
    index_files = [
        config.PRODUCT_INDEX_DIR / "vectors.npy",
        config.PRODUCT_INDEX_DIR / "products.parquet",
        config.PRODUCT_INDEX_DIR / "bm25_counts.npz",
    ]

    if all(p.exists() for p in index_files) and config.COMMENTS_CLEAN.exists():
        idx = retrieval.ProductIndex.load()
        by_product = retrieval.load_comments_by_product()
    else:
        if "products_df" not in globals() or "comments_df" not in globals():
            raise RuntimeError("Run Phase 1 and Phase 2 index-building cells first.")

        prepared = _prepare_products(products_df, comments_df)
        idx = ProductIndex.build(prepared)
        idx.save()

        by_product = {
            int(pid): g.reset_index(drop=True)
            for pid, g in comments_df[comments_df["has_text"]].groupby("product_id")
            if pd.notna(pid)
        }

    cat = Catalog.build(idx.products, by_product)
    rrev = retrieval.ReviewRetriever(by_product)

    return ShoppingAssistant(cat, idx, rrev, llm=llm, final_k=final_k)


### Official Phase-2 capabilities and demos

- **2.1 Product discovery:** natural Persian need → price/brand/category filters + hybrid ranking.
- **2.2 Review-based Q&A:** retrieve only real reviews of the target product and return review IDs as evidence.
- **2.3 Product comparison:** direct product facts, review evidence, and final inference are explicitly separated.
- **2.4 Managerial analytics:** category/brand-level complaint patterns, low-recommendation products, and brand satisfaction.

The next cell builds the shared assistant. The following demo cell runs one example of each capability.


In [ ]:
products_df = _prepare_products(products_df, comments_df)
pidx = ProductIndex.build(products_df)
pidx.save()

by_product = {
    int(pid): g.reset_index(drop=True)
    for pid, g in comments_df[comments_df["has_text"]].groupby("product_id")
    if pd.notna(pid)
}

catalog = Catalog.build(pidx.products, by_product)
api_budget = BudgetTracker()
assistant = ShoppingAssistant(
    catalog,
    pidx,
    ReviewRetriever(by_product),
    llm=LLM(mode=RUN_MODE, budget=api_budget),
)

print("assistant ready | products", len(catalog.products), "| reviewed", len(by_product))
print("cached index:", config.PRODUCT_INDEX_DIR)

api_preflight_requests = 0

if RUN_MODE in ("hosted_auto", "project", "free", "paid"):
    main_llm_provider_snapshot = {
        key: value
        for key, value in (assistant.llm.provider or {}).items()
        if key != "api_key"
    }
    main_llm_diagnostic = assistant.llm.diagnose()
    api_preflight_requests = int(main_llm_diagnostic.get("request_attempts", 0))

    # diagnose() may auto-select a model; refresh the safe snapshot afterwards.
    main_llm_provider_snapshot = {
        key: value
        for key, value in (assistant.llm.provider or {}).items()
        if key != "api_key"
    }
    print("main LLM diagnostic:")
    print(json.dumps(main_llm_diagnostic, ensure_ascii=False, indent=2))

    provider_ready = bool(
        main_llm_diagnostic.get("key_present")
        and (
            (
                main_llm_diagnostic.get("models_status") == 200
                and main_llm_diagnostic.get("model_listed") is True
            )
            or main_llm_diagnostic.get("chat_probe_success") is True
        )
    )

    if not provider_ready:
        print("Official Metis API preflight failed; Phase 2 will use grounded extractive fallback. Final submission checks will stay false until the course Metis API succeeds.")
        assistant.llm = LLM(mode="extractive", budget=api_budget)


In [ ]:
def show(ans):
    print(
        f"intent={ans.intent} tier={ans.tier} latency={ans.latency_s}s "
        f"cites={ans.citations[:3]} rev={ans.review_citations[:3]}"
    )
    # Demo answers are intentionally printed in full.  Truncating at 900
    # characters made the comparison and managerial examples look incomplete
    # even when the underlying Answer object was correct.
    print(ans.text, "\n")


def choose_demo_context(catalog):
    p = catalog.products.copy()

    if "is_fake" in p:
        p = p[~p["is_fake"].fillna(False).astype(bool)].copy()

    p["comment_count"] = pd.to_numeric(
        p["comment_count"], errors="coerce"
    ).fillna(0).astype(int)

    fa = p[
        p["category1_norm"]
        .fillna("")
        .astype(str)
        .str.contains(r"[آ-ی]", regex=True)
    ].copy()

    category_stats = (
        fa.groupby("category1_norm")
        .agg(
            n_products=("product_id", "nunique"),
            n_reviews=("comment_count", "sum"),
        )
        .reset_index()
    )

    strong_categories = category_stats[
        (category_stats["n_products"] >= 8)
        & (category_stats["n_reviews"] >= 80)
    ].sort_values(
        ["n_reviews", "n_products"],
        ascending=False,
    )

    cat_order = strong_categories["category1_norm"].tolist()

    if not cat_order:
        cat_order = (
            category_stats
            .sort_values(["n_reviews", "n_products"], ascending=False)
            ["category1_norm"]
            .tolist()
        )

    fallback_context = None

    for cat_name in cat_order:
        g = fa[fa["category1_norm"] == cat_name].copy()
        pair_pool = g[g["comment_count"] >= 3].copy()

        if len(pair_pool) < 2:
            continue

        pair = (
            pair_pool
            .sort_values(
                ["comment_count", "rate_count", "product_rate_clean"],
                ascending=False,
            )
            ["product_id"]
            .head(2)
            .astype(int)
            .tolist()
        )

        if len(pair) < 2:
            continue

        category_products = int(g["product_id"].nunique())
        category_reviews = int(g["comment_count"].sum())

        # Keep a same-category fallback even if this category cannot support a
        # meaningful price-constrained demo.
        if fallback_context is None:
            no_budget_q = f"یک {cat_name} اقتصادی و باکیفیت پیشنهاد بده"
            no_budget_filters = extract_filters(catalog, no_budget_q)
            no_budget_matches = int(
                product_filter_mask(catalog.products, no_budget_filters).sum()
            )
            fallback_context = (
                pair,
                str(cat_name),
                None,
                no_budget_matches,
                category_products,
                category_reviews,
            )

        priced = g[
            pd.to_numeric(g["price_clean"], errors="coerce").gt(0)
        ].copy()

        if len(priced) < 5:
            continue

        price_toman = (
            pd.to_numeric(priced["price_clean"], errors="coerce")
            .dropna()
            / config.TOMAN_TO_RIAL
        )

        # A mid/high quantile gives a realistic budget while leaving multiple
        # candidates below it.
        q = float(price_toman.quantile(0.70))
        step = 100_000
        candidate_budget = max(
            step,
            int(math.ceil(q / step) * step),
        )

        discovery_q = (
            f"یک {cat_name} اقتصادی و باکیفیت "
            f"زیر {candidate_budget:,} تومان پیشنهاد بده"
        )
        filters = extract_filters(catalog, discovery_q)
        n_matches = int(
            product_filter_mask(catalog.products, filters).sum()
        )

        # Prefer the first high-support category whose *actual parsed query*
        # produces multiple matches.  This demonstrates type + price + quality
        # in the official Discovery demo instead of only unit-testing price.
        if n_matches >= 2:
            return (
                pair,
                str(cat_name),
                int(candidate_budget),
                n_matches,
                category_products,
                category_reviews,
            )

    if fallback_context is not None:
        return fallback_context

    # Last resort: choose two reviewed products, keeping the notebook runnable.
    top = (
        fa.sort_values(
            ["comment_count", "rate_count"],
            ascending=False,
        )
        .head(2)
    )

    pair = top["product_id"].astype(int).tolist()
    cat_name = str(top.iloc[0]["category1_norm"]) if len(top) else "نامشخص"
    query = f"یک {cat_name} اقتصادی و باکیفیت پیشنهاد بده"
    filters = extract_filters(catalog, query)
    n_matches = int(product_filter_mask(catalog.products, filters).sum())

    return (
        pair,
        cat_name,
        None,
        n_matches,
        int((fa["category1_norm"] == cat_name).sum()),
        int(fa.loc[fa["category1_norm"] == cat_name, "comment_count"].sum()),
    )

def build_demo_queries(catalog):
    (
        pair,
        cat,
        budget_toman,
        n_matches,
        category_products,
        category_reviews,
    ) = choose_demo_context(catalog)

    if budget_toman is None:
        discovery = f"یک {cat} اقتصادی و باکیفیت پیشنهاد بده"
    else:
        discovery = (
            f"یک {cat} اقتصادی و باکیفیت زیر {budget_toman:,} تومان پیشنهاد بده"
        )

    return [
        discovery,
        f"در دستهٔ {cat} چند گزینه با رضایت کاربران معرفی کن",
        f"آیا محصول {pair[0]} کیفیت خوبی دارد و کاربران راضی بودند؟",
        f"مشکلات و ایرادهای محصول {pair[1]} چیست؟",
        f"محصول {pair[0]} و محصول {pair[1]} را از نظر کیفیت و رضایت کاربران مقایسه کن",
        f"پرتکرارترین شکایت‌ها و نقاط ضعف در دستهٔ {cat} چیست؟",
    ], {
        "product_ids": pair,
        "category": cat,
        "budget_toman": budget_toman,
        "discovery_filter_matches": n_matches,
        "category_products": category_products,
        "category_reviews": category_reviews,
    }


demo_queries, demo_context = build_demo_queries(catalog)

print("demo context:", demo_context)

_expected_demo_intents = [
    "discovery",
    "discovery",
    "product_qa",
    "product_qa",
    "comparison",
    "managerial",
]

router_diagnostic = []

for _query, _expected in zip(demo_queries, _expected_demo_intents):
    _actual = assistant.router.route(_query).intent

    router_diagnostic.append({
        "query": _query,
        "expected": _expected,
        "actual": _actual,
        "passed": _actual == _expected,
    })

print("router diagnostic:")
print(pd.DataFrame(router_diagnostic).to_string(index=False))

_router_failures = [
    row for row in router_diagnostic
    if not row["passed"]
]

if _router_failures:
    raise RuntimeError(
        "Router regression detected: "
        + json.dumps(_router_failures, ensure_ascii=False)
    )

print("router checks: OK")

# Four visible Phase-2 capability demos.
demo_answers = {}

for idx in (0, 2, 4, 5):
    demo_answers[idx] = assistant.answer(demo_queries[idx])
    show(demo_answers[idx])

comparison_evidence_diagnostic = {
    "checked": False,
    "invalid_positive": [],
    "invalid_negative": [],
    "passed": True,
}

_demo_compare = demo_answers.get(4)

if _demo_compare is not None and isinstance(_demo_compare.sources, dict):
    comparison_evidence_diagnostic["checked"] = True

    for _pid, _rows in _demo_compare.sources.get("positive_reviews", {}).items():
        for _row in _rows:
            if not _accept_positive_evidence(_row):
                comparison_evidence_diagnostic["invalid_positive"].append({
                    "product_id": int(_pid),
                    "comment_id": int(_row["comment_id"]),
                    "text": str(_row.get("text", "")),
                })

    for _pid, _rows in _demo_compare.sources.get("negative_reviews", {}).items():
        for _row in _rows:
            if not _accept_negative_evidence(_row):
                comparison_evidence_diagnostic["invalid_negative"].append({
                    "product_id": int(_pid),
                    "comment_id": int(_row["comment_id"]),
                    "text": str(_row.get("text", "")),
                })

    comparison_evidence_diagnostic["passed"] = not (
        comparison_evidence_diagnostic["invalid_positive"]
        or comparison_evidence_diagnostic["invalid_negative"]
    )

print("comparison evidence diagnostic:")
print(json.dumps(comparison_evidence_diagnostic, ensure_ascii=False, indent=2))

if not comparison_evidence_diagnostic["passed"]:
    raise RuntimeError(
        "Comparison evidence polarity regression detected: "
        + json.dumps(comparison_evidence_diagnostic, ensure_ascii=False)
    )


## 3 · Phase 3 — Recommendation-status prediction (Macro-F1)

The official model uses **review text only**. Metadata such as star rating, likes, or buyer status is deliberately excluded from the main classifier because it can act as a strong proxy for `recommendation_status`.

Splits are also **grouped by `product_id`**, so a product never appears in both training and held-out test data. This is a stricter leakage control than a plain random row split.


In [ ]:
import json
import logging

import joblib
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from scipy.sparse import csr_matrix, hstack


log = logging.getLogger("digikala.phase3")

RANDOM_STATE = config.RANDOM_SEED
FA_TOKEN_PATTERN = r"[؀-ۿ0-9A-Za-z]+"
MAX_PER_CLASS = 30_000

# These fields are deliberately excluded from the primary classifier.
# They are used only in a diagnostic ablation to quantify how much
# target-adjacent/post-hoc metadata can inflate apparent performance.
LEAKAGE_ABLATION_FEATURES = ["rate_clean", "likes", "is_buyer"]

PERSIAN_STOPWORDS = [
    "و", "در", "به", "از", "که", "این", "با", "را", "برای", "رو", "هم", "یک", "ها",
    "است", "نیز", "شد", "شود", "می", "خواهد", "بر", "آن", "تا", "کرد", "دارد", "بود",
    "اما", "اگر", "هر", "همه", "خیلی", "بیشتر", "کمتر", "مثل", "مانند", "حتی",
]


def _load() -> pd.DataFrame:
    df = comments_df.copy()

    mask = (
        df["recommendation_valid"].fillna(False)
        & df["has_text"].fillna(False)
        & df["product_id"].notna()
    )

    df = df.loc[mask].copy()

    # Identical text must not leak across splits.
    df = df.drop_duplicates(subset="comment_text_norm", keep="first")

    return df.reset_index(drop=True)


def _stratified_cap(df: pd.DataFrame, col: str, cap: int) -> pd.DataFrame:
    parts = [
        g.sample(n=min(len(g), cap), random_state=RANDOM_STATE)
        for _, g in df.groupby(col, sort=False)
    ]

    return (
        pd.concat(parts)
        .sample(frac=1, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )


def _vectorizer(max_features: int = 50_000) -> TfidfVectorizer:
    return TfidfVectorizer(
        token_pattern=FA_TOKEN_PATTERN,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        max_features=max_features,
        stop_words=PERSIAN_STOPWORDS,
    )


def _pipeline(clf) -> Pipeline:
    return Pipeline([
        ("tfidf", _vectorizer()),
        ("clf", clf),
    ])


def _macro_f1(y_true, y_pred, n_classes: int) -> float:
    return f1_score(
        y_true,
        y_pred,
        labels=list(range(n_classes)),
        average="macro",
        zero_division=0,
    )


def _group_train_val_test(sample: pd.DataFrame):
    """About 52.5/17.5/30 split, with disjoint product groups."""
    split_test = GroupShuffleSplit(
        n_splits=1,
        test_size=0.30,
        random_state=RANDOM_STATE,
    )

    train_val_idx, test_idx = next(
        split_test.split(sample, groups=sample["product_id"])
    )

    train_val = sample.iloc[train_val_idx].reset_index(drop=True)
    test = sample.iloc[test_idx].reset_index(drop=True)

    split_val = GroupShuffleSplit(
        n_splits=1,
        test_size=0.25,
        random_state=RANDOM_STATE + 1,
    )

    train_idx, val_idx = next(
        split_val.split(train_val, groups=train_val["product_id"])
    )

    train = train_val.iloc[train_idx].reset_index(drop=True)
    val = train_val.iloc[val_idx].reset_index(drop=True)

    assert not (set(train["product_id"]) & set(val["product_id"]))
    assert not (set(train["product_id"]) & set(test["product_id"]))
    assert not (set(val["product_id"]) & set(test["product_id"]))

    return train, val, test



def _prepare_leakage_numeric(
    frame: pd.DataFrame,
    *,
    medians: pd.Series | None = None,
):
    out = pd.DataFrame(index=frame.index)

    out["rate_clean"] = pd.to_numeric(
        frame["rate_clean"],
        errors="coerce",
    )

    out["likes"] = np.log1p(
        pd.to_numeric(
            frame["likes"],
            errors="coerce",
        ).clip(lower=0)
    )

    out["is_buyer"] = (
        frame["is_buyer"]
        .fillna(False)
        .astype(bool)
        .astype(float)
    )

    if medians is None:
        medians = out.median(numeric_only=True).fillna(0.0)

    out = out.fillna(medians).fillna(0.0)

    return out.astype(float), medians


def _phase3_leakage_ablation(
    *,
    sample: pd.DataFrame,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    final_text_pipeline: Pipeline,
    text_only_macro_f1: float,
    n_classes: int,
) -> dict:
    required = set(LEAKAGE_ABLATION_FEATURES)
    missing = sorted(required - set(sample.columns))

    if missing:
        return {
            "available": False,
            "missing_features": missing,
            "primary_model_unchanged": True,
            "note": "Ablation skipped because required metadata columns are missing.",
        }

    train_final = pd.concat(
        [train_df, val_df],
        ignore_index=True,
    )

    vectorizer = final_text_pipeline.named_steps["tfidf"]

    x_train_text = vectorizer.transform(
        train_final["comment_text_norm"]
    )
    x_test_text = vectorizer.transform(
        test_df["comment_text_norm"]
    )

    num_train_df, medians = _prepare_leakage_numeric(train_final)
    num_test_df, _ = _prepare_leakage_numeric(
        test_df,
        medians=medians,
    )

    scaler = StandardScaler()
    x_train_num = scaler.fit_transform(num_train_df)
    x_test_num = scaler.transform(num_test_df)

    x_train = hstack(
        [x_train_text, csr_matrix(x_train_num)],
        format="csr",
    )
    x_test = hstack(
        [x_test_text, csr_matrix(x_test_num)],
        format="csr",
    )

    diagnostic_clf = LogisticRegression(
        max_iter=5000,
        tol=1e-3,
        class_weight="balanced",
        C=1.0,
        solver="saga",
        random_state=RANDOM_STATE,
    )

    diagnostic_clf.fit(
        x_train,
        train_final["target_encoded"],
    )

    diagnostic_pred = diagnostic_clf.predict(x_test)

    diagnostic_f1 = round(
        _macro_f1(
            test_df["target_encoded"],
            diagnostic_pred,
            n_classes,
        ),
        4,
    )

    # Quantify the residual product leakage of a naive random row split.
    random_train, random_test = train_test_split(
        sample,
        test_size=0.30,
        stratify=sample["target_encoded"],
        random_state=RANDOM_STATE,
    )

    random_train_products = set(
        random_train["product_id"].astype(int)
    )
    random_test_products = set(
        random_test["product_id"].astype(int)
    )

    overlap = random_train_products & random_test_products

    random_overlap_pct = (
        100.0
        * len(overlap)
        / max(1, len(random_test_products))
    )

    lift = round(
        float(diagnostic_f1) - float(text_only_macro_f1),
        4,
    )

    return {
        "available": True,
        "primary_model_unchanged": True,
        "primary_split": "product_grouped",
        "features_primary_model": ["comment_text_norm"],
        "features_ablation_only": LEAKAGE_ABLATION_FEATURES,
        "text_only_grouped_test_macro_f1": round(
            float(text_only_macro_f1),
            4,
        ),
        "text_plus_target_adjacent_metadata_macro_f1": diagnostic_f1,
        "target_adjacent_metadata_lift": lift,
        "naive_random_split_product_overlap_pct": round(
            float(random_overlap_pct),
            2,
        ),
        "interpretation": (
            "rate_clean is strongly target-adjacent and likes/is_buyer are "
            "post-hoc/contextual metadata. Any lift is treated as evidence "
            "of leakage/inflation risk; the submitted classifier stays text-only."
        ),
    }


def train_and_save() -> dict:
    """Text-only 3-class classifier with product-grouped validation/test splits."""
    df = _load()
    log.info("usable text rows: %d", len(df))

    le = LabelEncoder()
    df["target_encoded"] = le.fit_transform(df["recommendation_status"])

    sample = _stratified_cap(df, "target_encoded", MAX_PER_CLASS)
    train_df, val_df, test_df = _group_train_val_test(sample)

    X_train = train_df["comment_text_norm"]
    y_train = train_df["target_encoded"]
    X_val = val_df["comment_text_norm"]
    y_val = val_df["target_encoded"]
    X_test = test_df["comment_text_norm"]
    y_test = test_df["target_encoded"]

    n_classes = len(le.classes_)

    baselines = {}

    majority = _pipeline(DummyClassifier(strategy="most_frequent"))
    majority.fit(X_train, y_train)
    baselines["majority"] = round(
        _macro_f1(y_val, majority.predict(X_val), n_classes),
        4,
    )

    logreg = _pipeline(
        LogisticRegression(
            max_iter=5000,
            tol=1e-3,
            class_weight="balanced",
            C=1.0,
            solver="saga",
            random_state=RANDOM_STATE,
        )
    )
    logreg.fit(X_train, y_train)
    baselines["tfidf_logreg_val"] = round(
        _macro_f1(y_val, logreg.predict(X_val), n_classes),
        4,
    )

    # Configuration is fixed before touching the held-out product groups.
    train_final = pd.concat([train_df, val_df], ignore_index=True)

    final = _pipeline(
        LogisticRegression(
            max_iter=5000,
            tol=1e-3,
            class_weight="balanced",
            C=1.0,
            solver="saga",
            random_state=RANDOM_STATE,
        )
    )

    final.fit(
        train_final["comment_text_norm"],
        train_final["target_encoded"],
    )

    y_pred = final.predict(X_test)
    test_macro_f1 = round(_macro_f1(y_test, y_pred, n_classes), 4)

    leakage_ablation = _phase3_leakage_ablation(
        sample=sample,
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        final_text_pipeline=final,
        text_only_macro_f1=test_macro_f1,
        n_classes=n_classes,
    )

    y_true_label = le.inverse_transform(np.asarray(y_test, dtype=int))
    y_pred_label = le.inverse_transform(np.asarray(y_pred, dtype=int))

    labels = list(le.classes_)

    report = classification_report(
        y_true_label,
        y_pred_label,
        labels=labels,
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(
        y_true_label,
        y_pred_label,
        labels=labels,
    ).tolist()

    err = pd.DataFrame({
        "comment_text": X_test.reset_index(drop=True),
        "true": y_true_label,
        "pred": y_pred_label,
    })

    err = err[err["true"] != err["pred"]].copy()

    error_pairs = (
        err.groupby(["true", "pred"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .to_dict("records")
    )

    failure_examples = (
        err.sample(n=min(12, len(err)), random_state=RANDOM_STATE)
        .assign(comment_text=lambda x: x["comment_text"].str.slice(0, 350))
        .to_dict("records")
        if len(err)
        else []
    )

    model_path = config.MODELS_DIR / "recommendation_model.pkl"

    joblib.dump({
        "pipeline": final,
        "label_encoder": le,
        "labels": labels,
        "input": "comment_text_norm_only",
    }, model_path)

    metrics = {
        "n_rows": int(len(df)),
        "n_sampled": int(len(sample)),
        "split": "product_grouped_train_val_test",
        "text_only": True,
        "n_train": int(len(train_df)),
        "n_val": int(len(val_df)),
        "n_test": int(len(test_df)),
        "n_train_products": int(train_df["product_id"].nunique()),
        "n_val_products": int(val_df["product_id"].nunique()),
        "n_test_products": int(test_df["product_id"].nunique()),
        "baselines_val_macro_f1": baselines,
        "test_macro_f1": test_macro_f1,
        "leakage_ablation": leakage_ablation,
        "labels": labels,
        "confusion_matrix": cm,
        "classification_report": report,
        "error_pairs": error_pairs,
        "failure_examples": failure_examples,
        "model_path": str(model_path),
    }

    (config.METRICS_DIR / "phase3_metrics.json").write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    if len(err):
        err.to_csv(
            config.METRICS_DIR / "phase3_misclassifications.csv",
            index=False,
            encoding="utf-8-sig",
        )

    log.info("test Macro-F1 %.4f | saved %s", test_macro_f1, model_path.name)

    return metrics


def fig_confusion(metrics: dict):
    import plotly.express as px

    cm = np.array(metrics["confusion_matrix"])
    labels = metrics["labels"]
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)

    return px.imshow(
        cm_norm,
        x=labels,
        y=labels,
        text_auto=".2f",
        color_continuous_scale="Blues",
        labels={
            "x": "predicted",
            "y": "true",
            "color": "row-normalized",
        },
        title=f"Confusion matrix (grouped test Macro-F1 = {metrics['test_macro_f1']})",
    )


def load_model():
    return joblib.load(config.MODELS_DIR / "recommendation_model.pkl")


def predict(texts) -> list[str]:
    """Predict from raw review text only."""
    bundle = load_model()

    if isinstance(texts, str):
        texts = [texts]

    clean_texts = [pt.normalize(t) for t in texts]
    preds = bundle["pipeline"].predict(clean_texts)

    return [
        bundle["label_encoder"].inverse_transform([int(p)])[0]
        for p in preds
    ]


In [ ]:
p3_metrics = train_and_save()

print("validation baselines:", p3_metrics["baselines_val_macro_f1"])
print("GROUPED TEST Macro-F1:", p3_metrics["test_macro_f1"])
print("\nPhase-3 leakage / target-adjacent metadata ablation:")
print(
    json.dumps(
        p3_metrics.get("leakage_ablation", {}),
        ensure_ascii=False,
        indent=2,
    )
)
print(
    "products train/val/test:",
    p3_metrics["n_train_products"],
    p3_metrics["n_val_products"],
    p3_metrics["n_test_products"],
)

fig_confusion(p3_metrics).show()


per_class_report = (
    pd.DataFrame(p3_metrics["classification_report"]).T
    .loc[p3_metrics["labels"], ["precision", "recall", "f1-score", "support"]]
    .round(3)
)
print("\nPer-class grouped-test report:")
display(per_class_report)


In [ ]:
predict([
    "کیفیتش عالی بود و کاملا راضی بودم، پیشنهاد می‌کنم",
    "خیلی بد بود و اصلا ارزش خرید نداشت",
    "معمولی بود، نه خیلی خوب نه خیلی بد",
])


## 4 · Phase 4 — Evaluation

The evaluation is deliberately separated into measurable pieces:

- **Retrieval:** a reproducible **Persian need-style pseudo-gold benchmark**; lexical baseline vs hybrid RRF. It is programmatically derived from catalogue metadata and is **not presented as human-labeled**.
- **Grounding / response quality:** a response set from categories different from the visible demo is evaluated with citation validity and transparent deterministic relevance/task-completion/grounding proxies; optional LLM-as-Judge if configured.
- **Prediction:** held-out product-grouped Macro-F1 from Phase 3.
- **Latency & API accounting:** measured per query, with hosted API attempts, failures, successful calls, token counts, tracked cost, and list-price estimate when available.
- **Failure analysis:** actual retrieval losses, weakly grounded answers, missing-information cases, and classifier confusion pairs.

The deterministic quality scores are reported as **proxies**, not as replacements for human evaluation.


The response-quality set is intentionally **separate from the visible Phase-2 demo** and spans multiple high-support categories. It is still programmatically selected rather than human-labeled, and this limitation is reported explicitly.


In [ ]:
import json
import logging
import re
import time

import numpy as np
import pandas as pd


log = logging.getLogger("digikala.phase4")

_CITE_ANY = re.compile(r"\[(?:محصول|بازبینی)\s*(\d+)\]")
_CITE_P_EVAL = re.compile(r"\[محصول\s*(\d+)\]")
_CITE_R_EVAL = re.compile(r"\[بازبینی\s*(\d+)\]")


# ---- ranking metrics ---------------------------------------------------
def _dcg(rels):
    return sum((2 ** r - 1) / np.log2(i + 2) for i, r in enumerate(rels))


def ranking_metrics(retrieved_ids, relevant_ids, k=None) -> dict:
    k = k or len(retrieved_ids)
    retrieved = list(retrieved_ids)[:k]
    relevant_ids = set(int(x) for x in relevant_ids)

    hits = [1 if int(i) in relevant_ids else 0 for i in retrieved]

    recall = sum(hits) / max(1, len(relevant_ids))
    mrr = next(
        (1.0 / rank for rank, i in enumerate(retrieved, 1) if int(i) in relevant_ids),
        0.0,
    )

    ideal = [1] * min(len(relevant_ids), k)
    ndcg = _dcg(hits) / max(_dcg(ideal), 1e-9)

    return {
        "recall": float(recall),
        "mrr": float(mrr),
        "ndcg": float(ndcg),
    }


_CATEGORY_FA = {
    "beauty": "محصول آرایشی و بهداشتی",
    "personal care": "محصول مراقبت شخصی",
    "book": "کتاب یا کالای فرهنگی",
    "stationary": "لوازم تحریر",
    "home": "کالای خانه",
    "kitchen": "کالای آشپزخانه",
    "mobile": "کالای مرتبط با موبایل",
    "computer": "کالای کامپیوتری",
    "electronic": "کالای الکترونیکی",
    "fashion": "پوشاک یا اکسسوری",
    "sport": "کالای ورزشی",
    "food": "محصول خوراکی",
}


def _fa_category_label(category: str) -> str:
    cat = pt.normalize(category)
    low = cat.lower()

    for key, label in _CATEGORY_FA.items():
        if key in low:
            return label

    # Keep native Persian categories as-is; do not put raw English taxonomy
    # strings into a benchmark that is described as Persian.
    if re.search(r"[آ-ی]", cat):
        return cat

    return "کالا"


def build_natural_retrieval_cases(assistant, n_queries: int = 20):
    """Build a diverse Persian pseudo-gold retrieval benchmark.

    Gold items are still programmatically derived from catalogue metadata, so
    this is explicitly *not* human annotation.  To avoid the old benchmark
    being dominated by one high-volume category, at most four cases are taken
    from the same Persian category label.
    """
    p = assistant.c.products.copy()

    required = [
        "product_id",
        "title_norm",
        "brand_norm",
        "category1_norm",
        "category2_norm",
        "sub_category_norm",
    ]

    for col in required:
        if col not in p:
            p[col] = ""

    p = p[
        p["product_id"].notna()
        & p["brand_norm"].notna()
        & (p["brand_norm"].astype(str).str.strip() != "")
        & (~p["brand_norm"].isin(_GENERIC_VALUES))
    ].copy()

    if "comment_count" in p:
        p = p[p["comment_count"] > 0]

    common = {
        "مدل", "سری", "اصل", "اورجینال", "جدید", "محصول", "عدد",
        "بسته", "مناسب", "کیفیت", "خوب", "رنگ", "طرح",
    }

    candidates = []
    seen_queries = set()
    seen_targets = set()

    # Start with the user-facing Persian category, then progressively use more
    # specific taxonomy levels.  Final selection is category-stratified.
    specs = [
        ("brand_norm", "category1_norm"),
        ("brand_norm", "category2_norm"),
        ("brand_norm", "sub_category_norm"),
    ]

    max_per_category = max(
        2,
        min(4, int(np.ceil(n_queries / 5))),
    )

    category_counts = Counter()

    for brand_col, cat_col in specs:
        tmp = p[
            p[cat_col].notna()
            & (p[cat_col].astype(str).str.strip() != "")
            & (~p[cat_col].isin(_GENERIC_VALUES))
        ].copy()

        grouped = []

        for (brand, category), g in tmp.groupby(
            [brand_col, cat_col],
            sort=False,
        ):
            if len(g) < 2 or len(g) > 60:
                continue

            strength = int(
                g.get(
                    "comment_count",
                    pd.Series(0, index=g.index),
                ).sum()
            )
            grouped.append(
                (strength, str(brand), str(category), g.copy())
            )

        grouped.sort(
            key=lambda x: x[0],
            reverse=True,
        )

        for _, brand, category, g in grouped:
            category_fa = _fa_category_label(category)

            if category_counts[category_fa] >= max_per_category:
                continue

            g = g.sort_values(
                ["comment_count", "rate_count"],
                ascending=False,
            )

            token_df = Counter()

            for title in g["title_norm"].fillna(""):
                token_df.update(
                    set(pt.tokenize_norm(title))
                )

            forbidden = (
                set(pt.tokenize_norm(brand))
                | set(pt.tokenize_norm(category))
                | common
            )

            for target in g.head(min(6, len(g))).itertuples():
                target_id = int(target.product_id)

                if target_id in seen_targets:
                    continue

                title_tokens = [
                    t
                    for t in pt.tokenize_norm(
                        str(target.title_norm)
                    )
                    if len(t) >= 2
                    and not t.isdigit()
                    and t not in forbidden
                    and re.search(r"[آ-ی]", t)
                ]

                if not title_tokens:
                    continue

                title_tokens = sorted(
                    dict.fromkeys(title_tokens),
                    key=lambda t: (
                        token_df.get(t, 999999),
                        -len(t),
                    ),
                )

                desc = " ".join(
                    title_tokens[:2]
                )

                q = (
                    f"یک {category_fa} از برند {brand} می‌خواهم؛ "
                    f"ترجیحاً مدلی که مشخصهٔ «{desc}» را داشته باشد."
                )

                if q in seen_queries:
                    continue

                seen_queries.add(q)
                seen_targets.add(target_id)
                category_counts[category_fa] += 1

                candidates.append({
                    "query": q,
                    "relevant_ids": [target_id],
                    "brand": brand,
                    "category": category,
                    "category_fa": category_fa,
                    "target_title": str(target.title_norm),
                    "gold_source": (
                        "programmatic_pseudo_gold_"
                        "brand_category_partial_title"
                    ),
                })

                break

            if len(candidates) >= n_queries:
                break

        if len(candidates) >= n_queries:
            break

    return candidates[:n_queries]


def evaluate_retrieval_benchmark(assistant, n_queries: int = 20, k: int = 10):
    cases = build_natural_retrieval_cases(assistant, n_queries=n_queries)

    if not cases:
        raise RuntimeError("Could not build natural retrieval benchmark from the current sample.")

    baseline = LexicalBaseline(assistant.c)
    rows = []

    for case in cases:
        q = case["query"]
        relevant = set(case["relevant_ids"])
        filters = extract_filters(assistant.c, q)

        t0 = time.time()
        hybrid_hits = assistant.pidx.search(q, filters=filters, k=k)
        hybrid_latency = time.time() - t0
        hybrid_ids = [int(x["product_id"]) for x in hybrid_hits]
        hm = ranking_metrics(hybrid_ids, relevant, k=k)

        t0 = time.time()
        base_hits = baseline.discover(q, k=k)
        base_latency = time.time() - t0
        base_ids = [int(x) for x in base_hits["product_id"].tolist()]
        bm = ranking_metrics(base_ids, relevant, k=k)

        rows.append({
            "query": q,
            "n_relevant": len(relevant),
            "gold_source": case["gold_source"],
            "hybrid_recall@k": round(hm["recall"], 4),
            "hybrid_mrr": round(hm["mrr"], 4),
            "hybrid_ndcg@k": round(hm["ndcg"], 4),
            "hybrid_latency_s": round(hybrid_latency, 4),
            "baseline_recall@k": round(bm["recall"], 4),
            "baseline_mrr": round(bm["mrr"], 4),
            "baseline_ndcg@k": round(bm["ndcg"], 4),
            "baseline_latency_s": round(base_latency, 4),
            "hybrid_top_ids": hybrid_ids,
            "baseline_top_ids": base_ids,
            "relevant_ids": sorted(relevant),
        })

    details = pd.DataFrame(rows)

    hybrid_quality = float(details["hybrid_ndcg@k"].mean())
    baseline_quality = float(details["baseline_ndcg@k"].mean())
    hybrid_latency = float(details["hybrid_latency_s"].mean())
    baseline_latency = float(details["baseline_latency_s"].mean())

    if hybrid_quality > baseline_quality + 1e-9:
        quality_verdict = "hybrid_better"
    elif abs(hybrid_quality - baseline_quality) <= 1e-9:
        quality_verdict = "tie"
    else:
        quality_verdict = "lexical_baseline_better"

    latency_speedup = (
        baseline_latency / hybrid_latency
        if hybrid_latency > 0
        else None
    )

    if quality_verdict == "hybrid_better":
        recommended_quality_retriever = "hybrid"
    elif quality_verdict == "lexical_baseline_better":
        recommended_quality_retriever = "lexical_baseline"
    else:
        recommended_quality_retriever = "tie"

    summary = {
        "n_queries": int(len(details)),
        "k": int(k),
        "benchmark_type": "reproducible_programmatic_pseudo_gold_not_human_labeled",
        "quality_verdict_by_ndcg": quality_verdict,
        "recommended_retriever_for_quality": recommended_quality_retriever,
        "hybrid_quality_improvement_claim_supported": (
            quality_verdict == "hybrid_better"
        ),
        "hybrid_latency_speedup_vs_baseline": (
            round(float(latency_speedup), 2)
            if latency_speedup is not None
            else None
        ),
        "hybrid": {
            "recall@k": round(float(details["hybrid_recall@k"].mean()), 4),
            "mrr": round(float(details["hybrid_mrr"].mean()), 4),
            "ndcg@k": round(float(details["hybrid_ndcg@k"].mean()), 4),
            "mean_latency_s": round(float(details["hybrid_latency_s"].mean()), 4),
        },
        "lexical_baseline": {
            "recall@k": round(float(details["baseline_recall@k"].mean()), 4),
            "mrr": round(float(details["baseline_mrr"].mean()), 4),
            "ndcg@k": round(float(details["baseline_ndcg@k"].mean()), 4),
            "mean_latency_s": round(float(details["baseline_latency_s"].mean()), 4),
        },
        "hybrid_minus_baseline": {
            "recall@k": round(float(
                details["hybrid_recall@k"].mean()
                - details["baseline_recall@k"].mean()
            ), 4),
            "mrr": round(float(
                details["hybrid_mrr"].mean()
                - details["baseline_mrr"].mean()
            ), 4),
            "ndcg@k": round(float(
                details["hybrid_ndcg@k"].mean()
                - details["baseline_ndcg@k"].mean()
            ), 4),
        },
    }

    return cases, details, summary


def evaluate_retrieval_component_ablation(
    assistant,
    cases,
    k: int = 10,
):
    """Dense-only vs BM25-only vs Hybrid on the same natural pseudo-gold cases."""
    idx = assistant.pidx
    rows = []

    for case in cases:
        query = case["query"]
        relevant = set(int(x) for x in case["relevant_ids"])
        filters = extract_filters(assistant.c, query)

        qv = embed([query])[0]
        dense_scores = idx.vectors @ qv
        sparse_scores = idx.bm25.get_scores(query)

        mask = product_filter_mask(idx.products, filters)
        valid = np.flatnonzero(mask)

        if valid.size == 0:
            dense_ids = []
            bm25_ids = []
            hybrid_ids = []
        else:
            dense_order = valid[
                np.argsort(
                    -dense_scores[valid],
                    kind="stable",
                )[:k]
            ]
            dense_ids = (
                idx.products.iloc[dense_order]["product_id"]
                .astype(int)
                .tolist()
            )

            sparse_valid = valid[
                sparse_scores[valid] > 0
            ]

            if sparse_valid.size:
                bm25_order = sparse_valid[
                    np.argsort(
                        -sparse_scores[sparse_valid],
                        kind="stable",
                    )[:k]
                ]
                bm25_ids = (
                    idx.products.iloc[bm25_order]["product_id"]
                    .astype(int)
                    .tolist()
                )
            else:
                bm25_ids = []

            pool = min(
                int(valid.size),
                max(
                    int(config.RRF_CANDIDATE_POOL),
                    int(k) * 20,
                ),
            )

            d_pool = valid[
                np.argsort(
                    -dense_scores[valid],
                    kind="stable",
                )[:pool]
            ]

            sparse_pool_valid = valid[
                sparse_scores[valid] > 0
            ]

            if sparse_pool_valid.size:
                s_pool = sparse_pool_valid[
                    np.argsort(
                        -sparse_scores[sparse_pool_valid],
                        kind="stable",
                    )[:pool]
                ]

                fused = rrf_fuse(
                    [d_pool.tolist(), s_pool.tolist()],
                    k=idx.rrf_k,
                    weights=[
                        config.PRODUCT_RRF_DENSE_WEIGHT,
                        config.PRODUCT_RRF_SPARSE_WEIGHT,
                    ],
                )
            else:
                fused = rrf_fuse(
                    [d_pool.tolist()],
                    k=idx.rrf_k,
                )

            hybrid_order = [
                item_idx
                for item_idx, _ in sorted(
                    fused.items(),
                    key=lambda kv: kv[1],
                    reverse=True,
                )[:k]
            ]

            hybrid_ids = (
                idx.products.iloc[hybrid_order]["product_id"]
                .astype(int)
                .tolist()
            )

        dm = ranking_metrics(dense_ids, relevant, k=k)
        bm = ranking_metrics(bm25_ids, relevant, k=k)
        hm = ranking_metrics(hybrid_ids, relevant, k=k)

        rows.append({
            "query": query,
            "category": case.get("category_fa", case.get("category")),
            "relevant_ids": sorted(relevant),
            "dense_recall@k": round(dm["recall"], 4),
            "dense_mrr": round(dm["mrr"], 4),
            "dense_ndcg@k": round(dm["ndcg"], 4),
            "bm25_recall@k": round(bm["recall"], 4),
            "bm25_mrr": round(bm["mrr"], 4),
            "bm25_ndcg@k": round(bm["ndcg"], 4),
            "hybrid_recall@k": round(hm["recall"], 4),
            "hybrid_mrr": round(hm["mrr"], 4),
            "hybrid_ndcg@k": round(hm["ndcg"], 4),
        })

    details = pd.DataFrame(rows)

    methods = {}

    for method in ("dense", "bm25", "hybrid"):
        methods[method] = {
            "recall@k": round(
                float(details[f"{method}_recall@k"].mean()),
                4,
            ),
            "mrr": round(
                float(details[f"{method}_mrr"].mean()),
                4,
            ),
            "ndcg@k": round(
                float(details[f"{method}_ndcg@k"].mean()),
                4,
            ),
        }

    best_single = max(
        ("dense", "bm25"),
        key=lambda name: methods[name]["ndcg@k"],
    )

    lift = round(
        methods["hybrid"]["ndcg@k"]
        - methods[best_single]["ndcg@k"],
        4,
    )

    summary = {
        "available": True,
        "n_queries": int(len(details)),
        "k": int(k),
        "benchmark_relation": (
            "same_natural_pseudo_gold_cases_as_primary_retrieval_evaluation"
        ),
        "by_method": methods,
        "best_single_method_by_ndcg": best_single,
        "hybrid_minus_best_single_ndcg": lift,
        "hybrid_component_quality_improvement_supported": bool(
            lift > 1e-9
        ),
        "interpretation": (
            "This is a component ablation. It does not override the primary "
            "lexical-vs-Hybrid benchmark verdict, and no Hybrid quality gain "
            "is claimed unless measured nDCG exceeds the best single component."
        ),
    }

    return details, summary



# ---- response quality / grounding -------------------------------------
def citation_coverage(text: str) -> float:
    """Line-level citation coverage for substantive answer claims."""
    claim_lines = []

    for raw in str(text).splitlines():
        line = raw.strip()

        if not line:
            continue

        # Headings are structure, not factual claims.
        if line.endswith(":") and len(pt.tokenize_norm(line)) <= 8:
            continue

        if re.match(r"^\d+\)\s*", line):
            continue

        if set(line) <= {"-", "|", " "}:
            continue

        if len(pt.tokenize_norm(line)) < 3:
            continue

        claim_lines.append(line)

    if not claim_lines:
        return 0.0

    return float(
        sum(1 for line in claim_lines if _CITE_ANY.search(line))
        / len(claim_lines)
    )


def citation_validity(ans) -> float:
    cited_p = {int(x) for x in _CITE_P_EVAL.findall(ans.text)}
    cited_r = {int(x) for x in _CITE_R_EVAL.findall(ans.text)}

    allowed_p = {int(x) for x in ans.citations}
    allowed_r = {int(x) for x in ans.review_citations}

    total = len(cited_p) + len(cited_r)

    if total == 0:
        return 1.0 if ans.missing_info else 0.0

    valid = len(cited_p & allowed_p) + len(cited_r & allowed_r)

    return float(valid / total)


def _content_tokens(text: str) -> set[str]:
    stop = {
        "یک", "و", "در", "از", "به", "را", "برای", "با", "می", "است",
        "این", "آن", "چه", "آیا", "کدام", "محصول", "کالا", "میخواهم", "می‌خواهم",
    }

    return {
        t for t in pt.tokenize(text)
        if len(t) > 1 and t not in stop
    }


def _sources_text(ans) -> str:
    if ans.intent == "product_qa":
        return prompts.evidence_reviews(ans.sources)

    if ans.intent == "discovery":
        return prompts.evidence_products(ans.sources)

    if ans.intent == "comparison":
        src = ans.sources if isinstance(ans.sources, dict) else {}
        return json.dumps(src, ensure_ascii=False, default=str)

    if ans.intent == "managerial":
        return json.dumps(ans.sources, ensure_ascii=False, default=str)

    return ""


def task_completion_proxy(query: str, ans) -> float:
    """Intent-specific, deterministic task-completion score in [0, 5].

    This complements lexical overlap.  It checks whether the response actually
    contains the structures/evidence required by each Phase-2 capability and is
    explicitly reported as a proxy, not as human judgment.
    """
    text = str(ans.text)
    n_p = len(set(_CITE_P_EVAL.findall(text)))
    n_r = len(set(_CITE_R_EVAL.findall(text)))

    if ans.missing_info:
        return 3.0 if ("اطلاعات کافی" in text or "یافت نشد" in text) else 1.0

    if ans.intent == "discovery":
        score = 0.0
        score += 2.0 if n_p >= 3 else (1.0 if n_p >= 1 else 0.0)
        score += 1.0 if ("قیمت" in text) else 0.0
        score += 1.0 if ("امتیاز" in text or "توصیه" in text or "رضایت" in text) else 0.0
        score += 1.0 if ("پیشنهاد" in text or "برتر" in text or "گزینه" in text) else 0.0
        return min(5.0, score)

    if ans.intent == "product_qa":
        score = 0.0
        score += 1.0 if n_p >= 1 else 0.0
        score += 2.0 if n_r >= 2 else (1.0 if n_r >= 1 else 0.0)
        score += 1.0 if ("بازبینی" in text or "نظر" in text) else 0.0
        qn = pt.normalize(query)
        if any(x in qn for x in ("مشکل", "ایراد", "ضعف", "بد")):
            score += 1.0 if any(x in text for x in ("ایراد", "مشکل", "منفی", "ضعف")) else 0.0
        else:
            score += 1.0 if any(x in text for x in ("مثبت", "قوت", "توصیه", "راضی", "خوب")) else 0.0
        return min(5.0, score)

    if ans.intent == "comparison":
        score = 0.0
        score += 1.0 if n_p >= 2 else 0.0
        score += 1.0 if n_r >= 2 else 0.0
        score += 0.75 if "واقعیت‌های مستقیم" in text else 0.0
        score += 0.75 if "نقاط قوت" in text else 0.0
        score += 0.75 if "نقاط ضعف" in text else 0.0
        score += 0.75 if ("جمع‌بندی" in text or "استنباط" in text) else 0.0
        return min(5.0, score)

    if ans.intent == "managerial":
        score = 0.0
        score += 1.0 if ("شکایت" in text or "نارضایتی" in text) else 0.0
        score += 1.0 if ("برند" in text) else 0.0
        score += 1.0 if ("نرخ توصیه" in text) else 0.0
        score += 1.0 if ("تعداد بازبینی" in text or "تعداد محصول" in text) else 0.0
        score += 1.0 if (n_p >= 1 or n_r >= 1) else 0.0
        return min(5.0, score)

    return 0.0


def deterministic_quality_proxy(query: str, ans) -> dict:
    """Transparent, deterministic proxy scores in [0, 5].

    They are useful when no judge model is configured, but are not presented as
    equivalent to human evaluation.
    """
    q = _content_tokens(query)
    evidence = _content_tokens(_sources_text(ans))
    answer = _content_tokens(ans.text)

    if q:
        relevance_overlap = len(q & (answer | evidence)) / len(q)
    else:
        relevance_overlap = 0.0

    relevance = min(5.0, 5.0 * relevance_overlap)

    validity = citation_validity(ans)
    coverage = citation_coverage(ans.text)

    if ans.missing_info:
        grounding = 5.0 if validity == 1.0 else 2.5
    else:
        # Valid citations matter more than sentence-level citation density.
        grounding = 5.0 * (0.75 * validity + 0.25 * min(1.0, coverage / 0.5))

    return {
        "proxy_relevance_0_5": round(float(relevance), 3),
        "task_completion_proxy_0_5": round(float(task_completion_proxy(query, ans)), 3),
        "proxy_grounding_0_5": round(float(grounding), 3),
        "citation_validity": round(float(validity), 3),
        "citation_coverage": round(float(coverage), 3),
    }


def _judge_score(judge, system, user) -> int | None:
    text = judge.generate(system, user)

    if not text:
        return None

    digit_map = str.maketrans(
        "۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩",
        "01234567890123456789",
    )
    normalized = str(text).translate(digit_map)

    m = re.search(
        r"(?<!\d)([0-5])(?:\.0+)?(?!\d)",
        normalized,
    )

    return int(m.group(1)) if m else None


def evaluate_generative(assistant, queries, judge=None):
    rows = []

    for q in queries:
        t0 = time.time()
        a = assistant.answer(q)
        wall = time.time() - t0

        proxy = deterministic_quality_proxy(q, a)

        judge_rel = None
        judge_faith = None

        if judge is not None and judge.available():
            judge_rel = _judge_score(
                judge,
                prompts.JUDGE_REL_SYS,
                prompts.JUDGE_USER.format(query=q, answer=a.text, sources=""),
            )

            judge_faith = _judge_score(
                judge,
                prompts.JUDGE_FAITH_SYS,
                prompts.JUDGE_USER.format(
                    query=q,
                    answer=a.text,
                    sources=_sources_text(a),
                ),
            )

        rows.append({
            "query": q,
            "answer_text": a.text,
            "evidence_text": _sources_text(a),
            "intent": a.intent,
            "tier": a.tier,
            "latency_s": round(wall, 3),
            "cost_usd": round(a.cost_usd, 6),
            "missing_info": bool(a.missing_info),
            "n_citations": len(a.citations) + len(a.review_citations),
            **proxy,
            "judge_relevance_0_5": judge_rel,
            "judge_faithfulness_0_5": judge_faith,
        })

    per_query = pd.DataFrame(rows)

    by_intent = (
        per_query.groupby("intent")
        .agg(
            n=("query", "count"),
            mean_latency_s=("latency_s", "mean"),
            mean_proxy_relevance=("proxy_relevance_0_5", "mean"),
            mean_task_completion=("task_completion_proxy_0_5", "mean"),
            mean_proxy_grounding=("proxy_grounding_0_5", "mean"),
            mean_citation_validity=("citation_validity", "mean"),
            mean_citation_coverage=("citation_coverage", "mean"),
            missing_info_rate=("missing_info", "mean"),
        )
        .round(3)
        .reset_index()
    )

    return per_query, by_intent


def build_response_eval_queries(assistant, n_contexts: int = 2):
    """Build a small response-quality set separate from the visible demos.

    These cases are programmatically selected from different high-support
    categories in the current sample.  They are not human-labeled, but they
    avoid evaluating only the exact examples that were used in the demo cell.
    """
    p = assistant.c.products.copy()

    if "comment_count" not in p:
        p["comment_count"] = 0

    p["comment_count"] = pd.to_numeric(
        p["comment_count"],
        errors="coerce",
    ).fillna(0).astype(int)

    p = p[
        p["category1_norm"]
        .fillna("")
        .astype(str)
        .str.contains(r"[آ-ی]", regex=True)
    ].copy()

    demo_category = pt.normalize(
        globals()
        .get("demo_context", {})
        .get("category", "")
    )

    stats = (
        p.groupby("category1_norm")
        .agg(
            n_products=("product_id", "nunique"),
            n_reviews=("comment_count", "sum"),
        )
        .reset_index()
    )

    stats = stats[
        (stats["n_products"] >= 8)
        & (stats["n_reviews"] >= 80)
    ].sort_values(
        ["n_reviews", "n_products"],
        ascending=False,
    )

    queries = []
    expected = []
    contexts = []

    for row in stats.itertuples(index=False):
        cat = pt.normalize(row.category1_norm)

        if not cat or cat == demo_category:
            continue

        g = p[
            p["category1_norm"] == row.category1_norm
        ].copy()

        pair = (
            g[g["comment_count"] >= 3]
            .sort_values(
                ["comment_count", "rate_count", "product_rate_clean"],
                ascending=False,
            )
            ["product_id"]
            .head(2)
            .astype(int)
            .tolist()
        )

        if len(pair) < 2:
            continue

        q_block = [
            f"در دستهٔ {cat} چند محصول با رضایت خوب کاربران معرفی کن",
            f"آیا محصول {pair[0]} از نظر کیفیت و تجربهٔ کاربران ارزش خرید دارد؟",
            f"مهم‌ترین ایرادها و نقاط ضعف محصول {pair[1]} از نظر کاربران چیست؟",
            (
                f"محصول {pair[0]} و محصول {pair[1]} را از نظر قیمت، "
                f"رضایت کاربران و نقاط قوت و ضعف مقایسه کن"
            ),
            (
                f"در دستهٔ {cat} شکایت‌های پرتکرار چیست و کدام محصولات "
                f"با نظر کافی نرخ توصیهٔ پایین‌تری دارند؟"
            ),
        ]

        exp_block = [
            "discovery",
            "product_qa",
            "product_qa",
            "comparison",
            "managerial",
        ]

        queries.extend(q_block)
        expected.extend(exp_block)
        contexts.append({
            "category": cat,
            "product_ids": pair,
            "n_products": int(row.n_products),
            "n_reviews": int(row.n_reviews),
        })

        if len(contexts) >= n_contexts:
            break

    if not contexts:
        # Fallback is explicit and recorded; normally the 100k sample has many
        # qualifying categories.
        queries, demo_meta = build_demo_queries(assistant.c)
        expected = [
            "discovery",
            "discovery",
            "product_qa",
            "product_qa",
            "comparison",
            "managerial",
        ]
        contexts = [{
            "fallback_to_demo": True,
            **demo_meta,
        }]

    routing = []

    for q, exp in zip(queries, expected):
        actual = assistant.router.route(q).intent
        routing.append({
            "query": q,
            "expected": exp,
            "actual": actual,
            "passed": actual == exp,
        })

    failures = [
        row
        for row in routing
        if not row["passed"]
    ]

    if failures:
        raise RuntimeError(
            "Response-evaluation routing regression: "
            + json.dumps(
                failures,
                ensure_ascii=False,
            )
        )

    meta = {
        "source": (
            "held_out_programmatic_multi_category_not_human_labeled"
            if not contexts[0].get("fallback_to_demo")
            else "demo_fallback"
        ),
        "n_queries": int(len(queries)),
        "n_contexts": int(len(contexts)),
        "contexts": contexts,
        "routing_checks_passed": True,
        "routing": routing,
        "distinct_from_demo_category": bool(
            all(
                pt.normalize(c.get("category", "")) != demo_category
                for c in contexts
                if c.get("category")
            )
        ),
    }

    return queries, meta


# ---- failure analysis --------------------------------------------------
def build_failure_analysis(retrieval_details, per_query, p3_metrics=None, api_summary=None) -> pd.DataFrame:
    rows = []

    for _, r in retrieval_details.iterrows():
        if r["hybrid_mrr"] == 0:
            rows.append({
                "component": "retrieval",
                "case": r["query"],
                "reason": "hybrid returned no relevant product in top-k",
                "severity": "high",
                "improvement_or_next_step": (
                    "Inspect brand/category filters and candidate coverage; keep the lexical control "
                    "and do not claim universal hybrid superiority."
                ),
            })
        elif r["hybrid_mrr"] < r["baseline_mrr"]:
            rows.append({
                "component": "retrieval",
                "case": r["query"],
                "reason": "hybrid ranked the first relevant item below the lexical baseline",
                "severity": "medium",
                "improvement_or_next_step": (
                    "Weighted candidate-limited RRF was introduced after the earlier weak run; "
                    "remaining losses are kept as explicit failure cases."
                ),
            })

    for _, r in per_query.iterrows():
        if bool(r["missing_info"]):
            rows.append({
                "component": "generation",
                "case": r["query"],
                "reason": "assistant reported insufficient information",
                "severity": "medium",
                "improvement_or_next_step": (
                    "Retrieve a wider evidence pool; for positive/negative QA do not substitute "
                    "opposite-polarity reviews when direct evidence is absent."
                ),
            })

        if r["citation_validity"] < 1.0:
            rows.append({
                "component": "grounding",
                "case": r["query"],
                "reason": "one or more citations were not in the retrieved evidence set",
                "severity": "high",
                "improvement_or_next_step": (
                    "Keep citation whitelisting and fall back to the deterministic grounded answer "
                    "when generated citations are unusable."
                ),
            })

        if r["proxy_grounding_0_5"] < 3.5:
            rows.append({
                "component": "grounding",
                "case": r["query"],
                "reason": "deterministic grounding proxy below 3.5/5",
                "severity": "medium",
                "improvement_or_next_step": (
                    "Tighten evidence selection and require claim-level product/review citations."
                ),
            })

        if r["task_completion_proxy_0_5"] < 3.0:
            rows.append({
                "component": "response_quality",
                "case": r["query"],
                "reason": "intent-specific deterministic task-completion proxy below 3/5",
                "severity": "medium",
                "improvement_or_next_step": (
                    "Inspect whether the answer contains the required facts/evidence/sections for "
                    "its routed Phase-2 capability. Lexical overlap is retained only as a diagnostic."
                ),
            })

    if api_summary:
        failed_api = int(api_summary.get("failed_calls", 0))
        unresolved_api = int(api_summary.get("unresolved_attempts", 0))

        if failed_api:
            rows.append({
                "component": "hosted_api",
                "case": f"{failed_api} failed HTTP attempt(s)",
                "reason": "hosted LLM transport/provider attempts failed before fallback or retry",
                "severity": "medium",
                "improvement_or_next_step": (
                    "The client retries transient SSL/429/5xx failures and records every HTTP attempt; "
                    "the grounded extractive fallback keeps the system runnable."
                ),
            })

        if unresolved_api:
            rows.append({
                "component": "hosted_api",
                "case": f"{unresolved_api} unresolved HTTP attempt(s)",
                "reason": "API accounting invariant was not satisfied",
                "severity": "high",
                "improvement_or_next_step": "Inspect the hosted-client accounting before submission.",
            })

    if p3_metrics:
        for item in p3_metrics.get("error_pairs", [])[:5]:
            rows.append({
                "component": "prediction",
                "case": f"{item['true']} → {item['pred']}",
                "reason": f"{item['count']} held-out test reviews confused in this direction",
                "severity": "diagnostic",
                "improvement_or_next_step": (
                    "Inspect ambiguous examples; a stronger character/word feature mix or a compact "
                    "Persian transformer is a reasonable next experiment, but the current result is kept unchanged."
                ),
            })

    return pd.DataFrame(
        rows,
        columns=[
            "component", "case", "reason", "severity",
            "improvement_or_next_step",
        ],
    )


def run(assistant_obj=None, n_retrieval: int = 20) -> dict:
    """Run the complete evaluation and write reproducible artifacts."""
    assistant_eval = assistant_obj or build_assistant()

    judge = None

    if hasattr(assistant_eval, "llm") and hasattr(assistant_eval.llm, "budget"):
        budget = assistant_eval.llm.budget
    else:
        budget = BudgetTracker()

    if config.JUDGE_MODE != "none":
        judge = judge_llm(budget=budget)

    retrieval_cases, retrieval_details, retrieval_summary = evaluate_retrieval_benchmark(
        assistant_eval,
        n_queries=n_retrieval,
        k=10,
    )

    retrieval_ablation_details, retrieval_ablation_summary = (
        evaluate_retrieval_component_ablation(
            assistant_eval,
            retrieval_cases,
            k=10,
        )
    )

    queries, response_eval_meta = build_response_eval_queries(
        assistant_eval,
        n_contexts=2,
    )
    per_query, by_intent = evaluate_generative(
        assistant_eval,
        queries,
        judge=judge,
    )

    by_tier = (
        per_query.groupby("tier")
        .agg(
            n=("query", "count"),
            mean_latency_s=("latency_s", "mean"),
            mean_task_completion=("task_completion_proxy_0_5", "mean"),
            mean_proxy_grounding=("proxy_grounding_0_5", "mean"),
            mean_citation_validity=("citation_validity", "mean"),
        )
        .round(3)
        .reset_index()
    )

    p3 = {}

    p3_file = config.METRICS_DIR / "phase3_metrics.json"

    if p3_file.exists():
        p3 = json.loads(p3_file.read_text(encoding="utf-8"))

    failures = build_failure_analysis(
        retrieval_details,
        per_query,
        p3_metrics=p3,
        api_summary=budget.summary(),
    )

    # Preflight failures happen before chat attempts, so they are not part of
    # BudgetTracker.  Keep them in failure analysis explicitly.
    preflight = globals().get("main_llm_diagnostic")

    if isinstance(preflight, dict) and preflight.get("error"):
        api_row = pd.DataFrame([{
            "component": "hosted_api_preflight",
            "case": str(preflight.get("provider", "hosted provider")),
            "reason": str(preflight.get("error", ""))[:500],
            "severity": "medium",
            "improvement_or_next_step": (
                "The client tries the normal environment path and a direct no-env-proxy path; "
                "the grounded extractive fallback keeps all required project functions runnable."
            ),
        }])
        failures = pd.concat([failures, api_row], ignore_index=True)

    metrics = {
        "run_mode": config.RUN_MODE,
        "judge_mode": config.JUDGE_MODE,
        "retrieval_quality": retrieval_summary,
        "retrieval_component_ablation": retrieval_ablation_summary,
        "generation": {
            "n_queries": int(len(per_query)),
            "evaluation_set": response_eval_meta,
            "tier_counts": {str(k): int(v) for k, v in per_query["tier"].value_counts().to_dict().items()},
            "mean_latency_s": round(float(per_query["latency_s"].mean()), 3),
            # Cost attributable only to the held-out response-evaluation
            # queries in this Phase-4 slice. The authoritative full-run API
            # total is reported separately by api_accounting_final.json.
            "evaluation_generation_cost_usd": round(
                float(per_query["cost_usd"].sum()),
                6,
            ),
            "total_cost_usd": round(
                float(per_query["cost_usd"].sum()),
                6,
            ),
            "mean_proxy_relevance_0_5": round(float(per_query["proxy_relevance_0_5"].mean()), 3),
            "mean_task_completion_proxy_0_5": round(float(per_query["task_completion_proxy_0_5"].mean()), 3),
            "mean_proxy_grounding_0_5": round(float(per_query["proxy_grounding_0_5"].mean()), 3),
            "mean_citation_validity": round(float(per_query["citation_validity"].mean()), 3),
            "mean_citation_coverage": round(float(per_query["citation_coverage"].mean()), 3),
            "mean_judge_relevance_0_5": (
                None
                if per_query["judge_relevance_0_5"].dropna().empty
                else round(float(per_query["judge_relevance_0_5"].dropna().mean()), 3)
            ),
            "mean_judge_faithfulness_0_5": (
                None
                if per_query["judge_faithfulness_0_5"].dropna().empty
                else round(float(per_query["judge_faithfulness_0_5"].dropna().mean()), 3)
            ),
            "by_intent": by_intent.to_dict("records"),
            "by_tier": by_tier.to_dict("records"),
        },
        "prediction_macro_f1_grouped_test": p3.get("test_macro_f1"),
        "prediction_text_only": p3.get("text_only"),
        "failure_count": int(len(failures)),
        "cost": budget.summary(),
    }

    (config.METRICS_DIR / "phase4_metrics.json").write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    retrieval_details.to_csv(
        config.METRICS_DIR / "phase4_retrieval_benchmark.csv",
        index=False,
        encoding="utf-8-sig",
    )

    retrieval_ablation_details.to_csv(
        config.METRICS_DIR / "phase4_retrieval_component_ablation.csv",
        index=False,
        encoding="utf-8-sig",
    )

    per_query.to_csv(
        config.METRICS_DIR / "phase4_per_query.csv",
        index=False,
        encoding="utf-8-sig",
    )

    by_intent.to_csv(
        config.METRICS_DIR / "phase4_by_intent.csv",
        index=False,
        encoding="utf-8-sig",
    )

    by_tier.to_csv(
        config.METRICS_DIR / "phase4_by_tier.csv",
        index=False,
        encoding="utf-8-sig",
    )

    failures.to_csv(
        config.METRICS_DIR / "phase4_failures.csv",
        index=False,
        encoding="utf-8-sig",
    )

    return {
        "metrics": metrics,
        "retrieval_details": retrieval_details,
        "retrieval_ablation_details": retrieval_ablation_details,
        "per_query": per_query,
        "by_intent": by_intent,
        "by_tier": by_tier,
        "failures": failures,
        "response_eval_meta": response_eval_meta,
    }


In [ ]:
phase4 = run(assistant_obj=assistant, n_retrieval=20)

print("Retrieval benchmark:")
print(json.dumps(phase4["metrics"]["retrieval_quality"], ensure_ascii=False, indent=2))

print("\nRetrieval component ablation (same natural pseudo-gold cases):")
print(
    json.dumps(
        phase4["metrics"].get("retrieval_component_ablation", {}),
        ensure_ascii=False,
        indent=2,
    )
)

display(
    phase4["retrieval_ablation_details"][
        [
            "query",
            "dense_ndcg@k",
            "bm25_ndcg@k",
            "hybrid_ndcg@k",
        ]
    ]
)

print("\nHeld-out response / grounding evaluation summary:")
print(json.dumps(phase4["metrics"]["generation"], ensure_ascii=False, indent=2))

print("\nResponse-evaluation set:")
print(json.dumps(
    phase4.get("response_eval_meta", {}),
    ensure_ascii=False,
    indent=2,
))

display(
    phase4["retrieval_details"][
        [
            "query",
            "n_relevant",
            "hybrid_recall@k",
            "hybrid_mrr",
            "hybrid_ndcg@k",
            "baseline_recall@k",
            "baseline_mrr",
            "baseline_ndcg@k",
        ]
    ]
)

display(phase4["per_query"])
display(phase4["by_intent"])
display(phase4["by_tier"])

if RUN_MODE in ("hosted_auto", "project", "free", "paid"):
    print(
        "\nGeneration policy: discovery/product_qa use the hosted LLM when "
        "available with grounded fallback; comparison/managerial use "
        "deterministic grounded rendering."
    )
else:
    print(
        "\nGeneration policy: this submission run is fully deterministic "
        "grounded/extractive; no hosted API is required."
    )

print(
    "\nResponse-quality interpretation: this evaluation set is separate from the visible demos; lexical relevance is only a diagnostic; "
    "the intent-specific task-completion proxy checks whether each answer contains "
    "the structures/evidence required by its Phase-2 capability."
)

print("\nFailure analysis:")
if len(phase4["failures"]):
    display(phase4["failures"])
else:
    print("No failure was triggered by the current thresholds.")


retrieval_summary = phase4["metrics"]["retrieval_quality"]
verdict = retrieval_summary.get("quality_verdict_by_ndcg")
speedup = retrieval_summary.get("hybrid_latency_speedup_vs_baseline")

print("\nRetrieval interpretation:")
if verdict == "hybrid_better":
    print("- Hybrid has higher held-out pseudo-gold nDCG than the lexical baseline.")
elif verdict == "tie":
    print("- Hybrid and lexical baseline are tied on pseudo-gold nDCG.")
else:
    print("- Lexical baseline is slightly stronger on this pseudo-gold benchmark; no unsupported hybrid-quality claim is made.")

if speedup is not None:
    print(f"- Hybrid mean latency speedup vs lexical baseline: {speedup}x")


In [ ]:
print("Lexical baseline example:")
display(
    LexicalBaseline(catalog).discover(
        "یک کالای ارزان و باکیفیت",
        k=5,
    )
)

if p3_metrics.get("failure_examples"):
    print("\nPhase 3 misclassification examples:")
    display(pd.DataFrame(p3_metrics["failure_examples"]).head(8))


In [ ]:
# Hosted API status and accounting after the Phase-2 demos and Phase-4 evaluation.

if "main_llm_diagnostic" in globals():
    print("Hosted LLM diagnostic used by the main assistant:")
    print(json.dumps(main_llm_diagnostic, ensure_ascii=False, indent=2))
else:
    print("No hosted-provider diagnostic was needed for the selected RUN_MODE.")

# The main assistant already exercises the hosted model when RUN_MODE is a hosted mode.
# A separate smoke call is normally unnecessary, but the flag is kept for debugging.
if RUN_FREE_LLM_SMOKE_TEST:
    free_llm = LLM(
        mode="free",
        budget=api_budget,
        max_tokens=512,
    )

    diagnostic = free_llm.diagnose()
    print("\nOptional extra smoke diagnostic:")
    print(json.dumps(diagnostic, ensure_ascii=False, indent=2))

    provider_ready = (
        diagnostic.get("key_present")
        and diagnostic.get("models_status") == 200
        and diagnostic.get("model_listed") is True
    )

    if provider_ready:
        llm_assistant = ShoppingAssistant(
            catalog,
            pidx,
            ReviewRetriever(by_product),
            llm=free_llm,
        )

        smoke_pid = int(
            catalog.products
            .sort_values("comment_count", ascending=False)
            .iloc[0]["product_id"]
        )
        q = f"مشکلات و نقاط ضعف محصول {smoke_pid} چیست؟"
        ans = llm_assistant.answer(q)

        print(
            "\nOptional smoke answer | backend:",
            free_llm.backend,
            "| answer tier:",
            ans.tier,
        )
        print(ans.text[:1200])

final_api_summary = api_budget.summary()

_active_provider = getattr(getattr(assistant, "llm", None), "provider", None) or {}
_provider_snapshot = globals().get("main_llm_provider_snapshot", {}) or {}

_project_provider_name = (
    _active_provider.get("provider_name")
    or _provider_snapshot.get("provider_name")
)
_project_model = (
    _active_provider.get("model")
    or _provider_snapshot.get("model")
)
_project_base_url = (
    _active_provider.get("base_url")
    or _provider_snapshot.get("base_url")
)
_project_key_source = (
    _active_provider.get("api_key_source")
    or _provider_snapshot.get("api_key_source")
)
_project_key_present = bool(
    _active_provider.get("api_key")
    or _project_key_source
)

project_api_report = {
    "run_mode": RUN_MODE,
    "provider": _project_provider_name,
    "base_url": _project_base_url,
    "model": _project_model,
    "key_present": _project_key_present,
    "key_source": _project_key_source,
    "preflight_requests": int(globals().get("api_preflight_requests", 0)),
    "preflight": globals().get("main_llm_diagnostic"),
    "chat_probe_success": bool(
        globals().get("main_llm_diagnostic", {}).get(
            "chat_probe_success",
            False,
        )
    ),
    "chat_probe_model": globals().get(
        "main_llm_diagnostic",
        {},
    ).get("chat_probe_model"),
    "chat_probe_attempts": int(
        globals().get("main_llm_diagnostic", {}).get(
            "chat_probe_attempts",
            0,
        )
    ),
    "chat": final_api_summary,
    "price_basis": (
        _active_provider.get("price_note")
        or _provider_snapshot.get("price_note")
    ),
    "project_credit_limit_usd": float(config.PROJECT_CREDIT_LIMIT_USD),
    "operational_safety_stop_usd": float(config.API_SAFETY_STOP_USD),
    "max_hosted_chat_attempts": int(config.MAX_HOSTED_CHAT_ATTEMPTS),
    "total_http_requests": (
        int(globals().get("api_preflight_requests", 0))
        + int(final_api_summary.get("api_attempts", 0))
    ),
    "chat_requests": int(final_api_summary.get("api_attempts", 0)),
    "successful_chat_requests": int(final_api_summary.get("successful_calls", 0)),
    "input_tokens": int(final_api_summary.get("input_tokens", 0)),
    "output_tokens": int(final_api_summary.get("output_tokens", 0)),
    "tracked_cost_usd": float(final_api_summary.get("total_cost_usd", 0.0)),
    "estimated_list_cost_usd": float(
        final_api_summary.get("estimated_list_cost_usd", 0.0)
    ),
    "within_5_usd_budget": (
        float(final_api_summary.get("estimated_list_cost_usd", 0.0))
        <= float(config.PROJECT_CREDIT_LIMIT_USD)
    ),
    "within_operational_safety_stop": (
        float(final_api_summary.get("estimated_list_cost_usd", 0.0))
        <= float(config.API_SAFETY_STOP_USD)
    ),
    "cost_reporting_note": (
        "Token counts come from the API response. Cost is the configured "
        "token-rate estimate unless the provider/dashboard supplies a separate invoice."
    ),
}

_preflight_ok = bool(
    isinstance(project_api_report.get("preflight"), dict)
    and (
        (
            project_api_report["preflight"].get("models_status") == 200
            and project_api_report["preflight"].get("model_listed") is True
        )
        or project_api_report["preflight"].get("chat_probe_success") is True
    )
)

project_api_report["api_evidence_passed"] = bool(
    RUN_MODE in ("hosted_auto", "project", "free", "paid")
    and project_api_report["key_present"]
    and _preflight_ok
    and int(final_api_summary.get("successful_calls", 0)) >= 1
    and int(final_api_summary.get("input_tokens", 0)) > 0
    and int(final_api_summary.get("output_tokens", 0)) > 0
    and int(final_api_summary.get("unresolved_attempts", 0)) == 0
)

print("\nProject API report:")
print(json.dumps(project_api_report, ensure_ascii=False, indent=2))

print("\nAPI accounting after the main run:")
print(json.dumps(final_api_summary, ensure_ascii=False, indent=2))
print(
    "preflight requests:",
    int(globals().get("api_preflight_requests", 0)),
    "| chat HTTP attempts:",
    final_api_summary["api_attempts"],
    "| successful:",
    final_api_summary["successful_calls"],
    "| failed:",
    final_api_summary["failed_calls"],
    "| unresolved:",
    final_api_summary["unresolved_attempts"],
    "| LLM cache hits:",
    int(getattr(getattr(assistant, "llm", None), "cache_hits", 0)),
    "| total reported HTTP requests:",
    int(globals().get("api_preflight_requests", 0)) + int(final_api_summary["api_attempts"]),
)

(config.METRICS_DIR / "api_accounting_final.json").write_text(
    json.dumps(final_api_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

(config.METRICS_DIR / "project_api_report.json").write_text(
    json.dumps(project_api_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

if "phase4" in globals() and isinstance(phase4, dict):
    phase4["metrics"]["api_accounting_after_main_run"] = final_api_summary
    phase4["metrics"]["api_preflight_requests"] = int(globals().get("api_preflight_requests", 0))
    (config.METRICS_DIR / "phase4_metrics.json").write_text(
        json.dumps(phase4["metrics"], ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


In [ ]:
# Compact end-of-run report for the final audit.

retrieval_q = phase4["metrics"]["retrieval_quality"]
generation_q = phase4["metrics"]["generation"]
api_q = api_budget.summary()

_demo_ids = [
    int(x)
    for x in globals().get("demo_context", {}).get("product_ids", [])
]

_demo_categories = []
for _pid in _demo_ids:
    _row = catalog.product(_pid) if "catalog" in globals() else None
    if _row:
        _demo_categories.append(str(_row.get("category1_norm", "")))

_demo_same_category = (
    len(_demo_categories) >= 2
    and len(set(_demo_categories)) == 1
)


readiness = {
    "run_all_reached_final_cell": True,
    "artifact_dir": str(BASE),
    "embedding_device": config.EMBEDDING_DEVICE,
    "embedding_model_load_source": globals().get("_model_load_source"),
    "run_mode": RUN_MODE,
    "generation_policy": {
        "discovery": (
            "hosted_llm_with_grounded_fallback"
            if RUN_MODE in ("hosted_auto", "project", "free", "paid")
            else "deterministic_grounded"
        ),
        "product_qa": (
            "hosted_llm_with_grounded_fallback"
            if RUN_MODE in ("hosted_auto", "project", "free", "paid")
            else "deterministic_grounded"
        ),
        "comparison": "deterministic_grounded",
        "managerial": "deterministic_grounded",
    },
    "sample_size_requested": SAMPLE_SIZE,
    "n_products": int(len(products_df)),
    "n_comments": int(len(comments_df)),
    "raw_comment_rows_scanned": phase1_report.get("sampling", {}).get("raw_comment_rows_scanned"),
    "raw_product_rows_scanned": phase1_report.get("sampling", {}).get("raw_product_rows_scanned"),
    "demo_context": globals().get("demo_context", {}),
    "demo_same_category_pair": bool(_demo_same_category),
    "comparison_evidence_checks_passed": bool(
        globals().get("comparison_evidence_diagnostic", {}).get("passed", False)
    ),
    "comparison_evidence_diagnostic": globals().get(
        "comparison_evidence_diagnostic", {}
    ),
    "router_checks_passed": bool(
        globals().get("router_diagnostic")
        and all(row.get("passed", False) for row in router_diagnostic)
    ),
    "router_diagnostic": globals().get("router_diagnostic", []),
    "phase3_grouped_test_macro_f1": p3_metrics.get("test_macro_f1"),
    "phase3_leakage_ablation": p3_metrics.get("leakage_ablation", {}),
    "phase3_leakage_ablation_available": bool(
        p3_metrics.get("leakage_ablation", {}).get("available", False)
    ),
    "retrieval_component_ablation": phase4["metrics"].get(
        "retrieval_component_ablation",
        {},
    ),
    "retrieval_component_ablation_available": bool(
        phase4["metrics"].get(
            "retrieval_component_ablation",
            {},
        ).get("available", False)
    ),
    "retrieval_quality_verdict_by_ndcg": retrieval_q.get("quality_verdict_by_ndcg"),
    "recommended_retriever_for_quality": retrieval_q.get(
        "recommended_retriever_for_quality"
    ),
    "hybrid_quality_improvement_claim_supported": bool(
        retrieval_q.get("hybrid_quality_improvement_claim_supported", False)
    ),
    "hybrid_recall_at_10": retrieval_q["hybrid"]["recall@k"],
    "lexical_recall_at_10": retrieval_q["lexical_baseline"]["recall@k"],
    "hybrid_ndcg_at_10": retrieval_q["hybrid"]["ndcg@k"],
    "lexical_ndcg_at_10": retrieval_q["lexical_baseline"]["ndcg@k"],
    "hybrid_latency_speedup_vs_baseline": retrieval_q.get("hybrid_latency_speedup_vs_baseline"),
    "response_eval_n_queries": int(generation_q.get("n_queries", 0)),
    "response_eval_source": generation_q.get("evaluation_set", {}).get("source"),
    "response_eval_n_contexts": generation_q.get("evaluation_set", {}).get("n_contexts"),
    "response_eval_distinct_from_demo_category": generation_q.get("evaluation_set", {}).get(
        "distinct_from_demo_category"
    ),
    "response_eval_routing_checks_passed": generation_q.get("evaluation_set", {}).get(
        "routing_checks_passed"
    ),
    "generation_tier_counts": generation_q.get("tier_counts", {}),
    "generation_by_tier": generation_q.get("by_tier", []),
    "mean_proxy_relevance_0_5": generation_q["mean_proxy_relevance_0_5"],
    "mean_task_completion_proxy_0_5": generation_q.get("mean_task_completion_proxy_0_5"),
    "mean_proxy_grounding_0_5": generation_q["mean_proxy_grounding_0_5"],
    "mean_citation_validity": generation_q["mean_citation_validity"],
    "mean_citation_coverage": generation_q["mean_citation_coverage"],
    "mean_judge_relevance_0_5": generation_q.get("mean_judge_relevance_0_5"),
    "mean_judge_faithfulness_0_5": generation_q.get("mean_judge_faithfulness_0_5"),
    "failure_count": phase4["metrics"]["failure_count"],
    "api_preflight_requests": int(globals().get("api_preflight_requests", 0)),
    "api_chat_attempts": api_q["api_attempts"],
    "api_total_http_requests_reported": int(globals().get("api_preflight_requests", 0)) + int(api_q["api_attempts"]),
    "api_attempts": api_q["api_attempts"],
    "api_successful_calls": api_q["successful_calls"],
    "api_failed_calls": api_q["failed_calls"],
    "api_unresolved_attempts": api_q["unresolved_attempts"],
    "llm_cache_hits": int(getattr(getattr(assistant, "llm", None), "cache_hits", 0)),
    "llm_generated_rejections": int(getattr(assistant, "generated_rejections", 0)),
    "hosted_disabled_reason": getattr(getattr(assistant, "llm", None), "hosted_disabled_reason", None),
    "hosted_preferred_network_path": getattr(getattr(assistant, "llm", None), "preferred_network_path", None),
    "api_input_tokens": api_q["input_tokens"],
    "api_output_tokens": api_q["output_tokens"],
    "api_estimated_list_cost_usd": api_q["estimated_list_cost_usd"],
    "phase4_evaluation_generation_cost_usd": phase4["metrics"]
        .get("generation", {})
        .get("evaluation_generation_cost_usd"),
    "project_api_report": globals().get("project_api_report", {}),
    "project_api_evidence_passed": bool(
        globals().get("project_api_report", {}).get("api_evidence_passed", False)
    ),
    "project_api_provider": globals().get("project_api_report", {}).get("provider"),
    "project_api_model": globals().get("project_api_report", {}).get("model"),
    "project_api_key_source": globals().get("project_api_report", {}).get("key_source"),
    "project_api_price_basis": globals().get("project_api_report", {}).get("price_basis"),
    "project_credit_limit_usd": float(config.PROJECT_CREDIT_LIMIT_USD),
    "operational_safety_stop_usd": float(config.API_SAFETY_STOP_USD),
    "api_total_http_requests": globals().get("project_api_report", {}).get(
        "total_http_requests", 0
    ),
    "api_within_budget": bool(
        globals().get("project_api_report", {}).get("within_5_usd_budget", False)
    ),
    "api_within_operational_safety_stop": bool(
        globals().get("project_api_report", {}).get(
            "within_operational_safety_stop",
            False,
        )
    ),
}

if RUN_FREE_LLM_SMOKE_TEST and "ans" in globals():
    readiness["hosted_llm_answer_tier"] = ans.tier
    readiness["hosted_llm_test_query"] = q



if "main_llm_diagnostic" in globals():
    readiness["main_llm_diagnostic"] = main_llm_diagnostic



import platform
import sklearn

readiness["environment"] = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
}

readiness["submission_checks"] = {
    "run_all_complete": bool(readiness.get("run_all_reached_final_cell")),
    "router_checks": bool(readiness.get("router_checks_passed")),
    "same_category_demo_pair": bool(readiness.get("demo_same_category_pair")),
    "comparison_evidence_polarity": bool(
        readiness.get("comparison_evidence_checks_passed")
        and not readiness.get(
            "comparison_evidence_diagnostic", {}
        ).get("invalid_positive")
        and not readiness.get(
            "comparison_evidence_diagnostic", {}
        ).get("invalid_negative")
    ),
    "retrieval_result_reported_honestly": (
        readiness.get("recommended_retriever_for_quality")
        in {"hybrid", "lexical_baseline", "tie"}
    ),
    "response_eval_separate": bool(
        readiness.get("response_eval_distinct_from_demo_category")
        and int(readiness.get("response_eval_n_queries") or 0) >= 10
    ),
    "response_eval_routing": bool(
        readiness.get("response_eval_routing_checks_passed")
    ),
    "phase3_macro_f1_available": (
        readiness.get("phase3_grouped_test_macro_f1") is not None
    ),
    "phase3_leakage_ablation_reported": bool(
        readiness.get("phase3_leakage_ablation_available")
    ),
    "retrieval_component_ablation_reported": bool(
        readiness.get("retrieval_component_ablation_available")
    ),
    "grounding_at_least_4_5": (
        float(readiness.get("mean_proxy_grounding_0_5") or 0) >= 4.5
    ),
    "task_completion_at_least_4": (
        float(readiness.get("mean_task_completion_proxy_0_5") or 0) >= 4.0
    ),
    "citation_validity_complete": (
        float(readiness.get("mean_citation_validity") or 0) >= 0.999
    ),
    "api_accounting_resolved": (
        int(readiness.get("api_unresolved_attempts") or 0) == 0
    ),
    "hosted_provider_preflight_or_probe_passed": bool(
        (
            isinstance(readiness.get("main_llm_diagnostic"), dict)
            and (
                (
                    readiness["main_llm_diagnostic"].get("models_status") == 200
                    and readiness["main_llm_diagnostic"].get("model_listed") is True
                )
                or readiness["main_llm_diagnostic"].get("chat_probe_success") is True
            )
        )
    ),
    "project_api_executed_and_reported": bool(
        readiness.get("project_api_evidence_passed")
    ),
    "api_tokens_reported": (
        int(readiness.get("api_input_tokens") or 0) > 0
        and int(readiness.get("api_output_tokens") or 0) > 0
    ),
    "api_cost_reported": (
        readiness.get("api_estimated_list_cost_usd") is not None
        and float(readiness.get("api_estimated_list_cost_usd") or 0) > 0
    ),
    "phase4_cost_attribution_consistent": (
        readiness.get("phase4_evaluation_generation_cost_usd") is not None
        and float(readiness.get("phase4_evaluation_generation_cost_usd") or 0)
            <= float(readiness.get("api_estimated_list_cost_usd") or 0) + 1e-12
    ),
    "api_request_count_reported": (
        int(readiness.get("api_total_http_requests") or 0) >= 2
    ),
    "api_within_5_usd_budget": bool(readiness.get("api_within_budget")),
    "api_within_4_50_usd_safety_stop": bool(
        readiness.get("api_within_operational_safety_stop")
    ),
    "official_course_api_provider_metis": (
        readiness.get("project_api_provider") == "metis"
    ),
    "official_course_api_key_source": (
        readiness.get("project_api_key_source") == "METIS_API_KEY"
    ),
    "official_course_api_model_gpt4o_mini": (
        readiness.get("project_api_model") == "gpt-4o-mini"
    ),
}

readiness["submission_checks_passed"] = bool(
    all(readiness["submission_checks"].values())
)

print("Final audit summary:")
print(json.dumps(readiness, ensure_ascii=False, indent=2))

_submission_readiness_chars = (
    config.METRICS_DIR / "submission_readiness.json"
).write_text(
    json.dumps(readiness, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("submission readiness saved:", config.METRICS_DIR / "submission_readiness.json")


# Write a concise final report next to the notebook.  This file is generated
# from the actual final Run All, so API tokens/cost and evaluation metrics are
# never hand-entered.
_final_report_path = Path.cwd() / "FINAL_REPORT.md"

_p3_ab = readiness.get("phase3_leakage_ablation", {})
_ret_ab = readiness.get("retrieval_component_ablation", {})
_api_report = readiness.get("project_api_report", {})

_final_report = f"""# Digikala AI Shopping Assistant — Final Report

## Final run status

- `submission_checks_passed`: **{readiness.get("submission_checks_passed")}**
- Python: `{readiness.get("environment", {}).get("python")}`
- Embedding device: `{readiness.get("embedding_device")}`
- Products in Phase-2 sampled catalogue: **{readiness.get("n_products")}**
- Clean sampled reviews: **{readiness.get("n_comments")}**

## Phase 2

The system implements:
1. Persian natural-language Product Discovery.
2. Review-grounded Product QA with Review IDs.
3. Product Comparison with direct facts, positive/negative evidence and inference.
4. Category/brand Managerial Analytics.

Router checks passed: **{readiness.get("router_checks_passed")}**

Comparison evidence checks passed:
**{readiness.get("comparison_evidence_checks_passed")}**

## Phase 3 — recommendation prediction

- Primary model: text-only TF-IDF + Logistic Regression.
- Split: product-grouped train/validation/test.
- Grouped test Macro-F1: **{readiness.get("phase3_grouped_test_macro_f1")}**
- Leakage/target-adjacent metadata ablation available:
  **{readiness.get("phase3_leakage_ablation_available")}**
- Text + metadata Macro-F1:
  **{_p3_ab.get("text_plus_target_adjacent_metadata_macro_f1")}**
- Metadata lift:
  **{_p3_ab.get("target_adjacent_metadata_lift")}**
- Naive random-split product overlap:
  **{_p3_ab.get("naive_random_split_product_overlap_pct")}%**

The primary submitted classifier remains text-only. Metadata ablation is
diagnostic evidence of leakage/inflation risk.

## Phase 4 — retrieval and response evaluation

- Hybrid Recall@10: **{readiness.get("hybrid_recall_at_10")}**
- Lexical Recall@10: **{readiness.get("lexical_recall_at_10")}**
- Hybrid nDCG@10: **{readiness.get("hybrid_ndcg_at_10")}**
- Lexical nDCG@10: **{readiness.get("lexical_ndcg_at_10")}**
- Recommended retriever for measured quality:
  **{readiness.get("recommended_retriever_for_quality")}**
- Hybrid latency speedup:
  **{readiness.get("hybrid_latency_speedup_vs_baseline")}x**

Retrieval component ablation:
- Dense: **{_ret_ab.get("by_method", {}).get("dense")}**
- BM25: **{_ret_ab.get("by_method", {}).get("bm25")}**
- Hybrid: **{_ret_ab.get("by_method", {}).get("hybrid")}**
- Hybrid minus best single-method nDCG:
  **{_ret_ab.get("hybrid_minus_best_single_ndcg")}**

Response evaluation:
- number of queries: **{readiness.get("response_eval_n_queries")}**
- mean task-completion proxy: **{readiness.get("mean_task_completion_proxy_0_5")} / 5**
- mean grounding proxy: **{readiness.get("mean_proxy_grounding_0_5")} / 5**
- citation validity: **{readiness.get("mean_citation_validity")}**
- citation coverage: **{readiness.get("mean_citation_coverage")}**
- failure cases retained for analysis: **{readiness.get("failure_count")}**

The response and grounding metrics are deterministic proxies, not human
evaluation or LLM-as-a-Judge.

## Official course API — Metis

- Provider: **{readiness.get("project_api_provider")}**
- Model: **{readiness.get("project_api_model")}**
- Key source: **{readiness.get("project_api_key_source")}**
- Base URL: `https://api.metisai.ir/openai/v1`
- HTTP requests reported: **{readiness.get("api_total_http_requests")}**
- Successful hosted calls: **{readiness.get("api_successful_calls")}**
- Failed hosted attempts: **{readiness.get("api_failed_calls")}**
- Unresolved attempts: **{readiness.get("api_unresolved_attempts")}**
- Input tokens: **{readiness.get("api_input_tokens")}**
- Output tokens: **{readiness.get("api_output_tokens")}**
- Estimated full-run token cost: **${readiness.get("api_estimated_list_cost_usd")}**
- Phase-4 held-out response-evaluation generation cost:
  **${readiness.get("phase4_evaluation_generation_cost_usd")}**
- Non-rechargeable course credit: **$5.00**
- Operational safety stop used by the notebook: **$4.50**
- Under $5 credit limit: **{readiness.get("api_within_budget")}**
- Under $4.50 safety stop:
  **{readiness.get("api_within_operational_safety_stop")}**

The cost above is a transparent engineering estimate from configured token
rates; the authoritative remaining credit is the Metis account balance.

## Reproducibility / limitation

The review sample is reproducible with a fixed seed and is drawn uniformly
from the full comments CSV.  Phase 2 indexes only products represented in the
sampled reviews, not the complete million-product catalogue.

The raw CSV files are intentionally not bundled with the submission package.
"""

_final_report_path.write_text(_final_report, encoding="utf-8")
print("final report saved:", _final_report_path)


In [ ]:
# Bonus configuration + install missing optional dependencies only.
import importlib.util
import subprocess
import sys
import os
import json
import time
from pathlib import Path

RUN_CACHE_BENCHMARK = True
RUN_QUANTIZATION_BONUS = True
RUN_LORA_BONUS = True
RUN_NEW_PROBLEM_EXPERIMENT = False
AUTO_LAUNCH_STREAMLIT = True

MENTOR_APPROVED_NEW_PROBLEM = (
    os.environ.get("MENTOR_APPROVED_NEW_PROBLEM", "0").strip().lower()
    in {"1", "true", "yes"}
)
MENTOR_APPROVED_AUCTION = (
    os.environ.get("MENTOR_APPROVED_AUCTION", "0").strip().lower()
    in {"1", "true", "yes"}
)

BONUS_OUTPUT_DIR = Path.cwd() / "bonus_outputs"
BONUS_OUTPUT_DIR.mkdir(exist_ok=True)
bonus_status = {}

_optional = {
    "streamlit": "streamlit>=1.47",
    "ipywidgets": "ipywidgets>=8",
    "datasets": "datasets",
    "peft": "peft>=0.17",
    "accelerate": "accelerate",
    "optimum": "optimum[onnxruntime]",
    "onnxruntime": "onnxruntime",
}
_missing = [
    pip_name
    for module_name, pip_name in _optional.items()
    if importlib.util.find_spec(module_name) is None
]

if _missing:
    print("Installing missing bonus dependencies:", _missing)

    _constraint_path = Path.cwd() / "_bonus_pip_constraints.txt"
    _constraint_path.write_text(
        "numpy==1.26.4\n"
        "scipy<1.15\n"
        "scikit-learn<1.6\n"
        "transformers<5\n",
        encoding="utf-8",
    )
    try:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--upgrade-strategy",
                "only-if-needed",
                "-c",
                str(_constraint_path),
                *sorted(set(_missing)),
            ]
        )
        bonus_status["dependency_install"] = {"success": True, "installed": _missing}
    except Exception as exc:
        bonus_status["dependency_install"] = {
            "success": False,
            "error": repr(exc),
        }
        print("Optional dependency install failed; affected bonuses may be skipped.")
else:
    bonus_status["dependency_install"] = {"success": True, "installed": []}

print(json.dumps(bonus_status["dependency_install"], ensure_ascii=False, indent=2))
print("MENTOR_APPROVED_NEW_PROBLEM =", MENTOR_APPROVED_NEW_PROBLEM)


### 5.1 Advanced evaluation — Metis LLM-as-a-Judge

`JUDGE_MODE="project"` evaluates the same held-out response set with the
official Metis `gpt-4o-mini`.

Two 0–5 rubric scores are produced:
- **Relevance:** does the answer address the user's query?
- **Faithfulness:** is the answer supported by the evidence?

Limitations are explicit: one model judge, prompt sensitivity, possible shared
provider/model biases, and no inter-rater agreement by itself. The Human
Evaluation widget below is the independent validation step.


In [ ]:
judge_summary = {
    "judge_mode": phase4["metrics"].get("judge_mode"),
    "n_queries": phase4["metrics"]["generation"].get("n_queries"),
    "mean_judge_relevance_0_5": phase4["metrics"]["generation"].get(
        "mean_judge_relevance_0_5"
    ),
    "mean_judge_faithfulness_0_5": phase4["metrics"]["generation"].get(
        "mean_judge_faithfulness_0_5"
    ),
    "limitations": [
        "single LLM judge",
        "ordinal score is prompt-sensitive",
        "not a substitute for human validation",
        "possible shared provider/model-family bias",
    ],
}
(BONUS_OUTPUT_DIR / "llm_judge_summary.json").write_text(
    json.dumps(judge_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps(judge_summary, ensure_ascii=False, indent=2))


### 5.2 Caching optimization benchmark

Measures the existing embedding-cache design directly by comparing loading
3,000 cached vectors with recomputing the same 3,000 product embeddings on the
current hardware.


In [ ]:
def run_cache_bonus():
    if not RUN_CACHE_BENCHMARK:
        return {"available": False, "reason": "disabled"}

    try:
        import numpy as np
        import pandas as pd
        import torch
        import time
        from sentence_transformers import SentenceTransformer

        n = 3000
        p = pd.read_parquet(config.PRODUCT_INDEX_DIR / "products.parquet")
        text_col = next(
            c for c in ("product_text_norm", "title_norm", "title_fa")
            if c in p.columns
        )
        texts = p[text_col].fillna("").astype(str).head(n).tolist()

        t0 = time.perf_counter()
        cached = np.load(config.PRODUCT_INDEX_DIR / "vectors.npy", mmap_mode="r")
        _ = np.asarray(cached[:len(texts)])
        cached_s = time.perf_counter() - t0

        model = SentenceTransformer(
            config.EMBEDDING_MODEL,
            device="cuda" if torch.cuda.is_available() else "cpu",
        )
        model.encode(texts[:16], normalize_embeddings=True, show_progress_bar=False)

        t0 = time.perf_counter()
        _ = model.encode(
            texts,
            batch_size=128,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        recompute_s = time.perf_counter() - t0

        speedup = recompute_s / max(cached_s, 1e-9)
        report = {
            "available": True,
            "n_vectors": len(texts),
            "cached_load_seconds": round(cached_s, 4),
            "recompute_seconds": round(recompute_s, 4),
            "cache_speedup": round(speedup, 2),
            "bonus_claim_supported": bool(speedup > 2.0),
        }
    except Exception as exc:
        report = {
            "available": False,
            "bonus_claim_supported": False,
            "error": repr(exc),
        }

    (BONUS_OUTPUT_DIR / "cache_benchmark_metrics.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return report

cache_bonus = run_cache_bonus()
bonus_status["cache"] = cache_bonus
print(json.dumps(cache_bonus, ensure_ascii=False, indent=2))


### 5.3 INT8 quantization benchmark

Attempts dynamic ONNX INT8 quantization of the multilingual embedding model and
measures CPU latency plus embedding/retrieval-quality preservation. Quantization
is claimable only when latency improves and quality remains materially
preserved.


In [ ]:
def run_quantization_bonus():
    if not RUN_QUANTIZATION_BONUS:
        return {"available": False, "reason": "disabled"}

    try:
        import numpy as np
        import pandas as pd
        import time
        from sentence_transformers import (
            SentenceTransformer,
            export_dynamic_quantized_onnx_model,
        )

        n_texts = 600
        products_bonus = pd.read_parquet(config.PRODUCT_INDEX_DIR / "products.parquet")
        text_col = next(
            c for c in ("product_text_norm", "title_norm", "title_fa")
            if c in products_bonus.columns
        )
        texts = (
            products_bonus[text_col]
            .fillna("")
            .astype(str)
            .loc[lambda x: x.str.len().gt(3)]
            .head(n_texts)
            .tolist()
        )

        fp_model = SentenceTransformer(config.EMBEDDING_MODEL, device="cpu")
        fp_model.encode(texts[:16], batch_size=16, normalize_embeddings=True)

        t0 = time.perf_counter()
        emb_fp = fp_model.encode(
            texts, batch_size=64, normalize_embeddings=True,
            show_progress_bar=False,
        )
        fp_s = time.perf_counter() - t0

        qdir = BONUS_OUTPUT_DIR / "embedding_onnx_int8"
        qdir.mkdir(parents=True, exist_ok=True)

        onnx_model = SentenceTransformer(
            config.EMBEDDING_MODEL, backend="onnx", device="cpu"
        )
        export_dynamic_quantized_onnx_model(
            model=onnx_model,
            quantization_config="avx2",
            model_name_or_path=str(qdir),
            push_to_hub=False,
        )

        qfiles = sorted(qdir.rglob("*qint8*.onnx"))
        if not qfiles:
            raise RuntimeError("quantized ONNX file not found after export")

        rel = qfiles[0].relative_to(qdir).as_posix()
        q_model = SentenceTransformer(
            str(qdir),
            backend="onnx",
            model_kwargs={"file_name": rel},
            device="cpu",
        )
        q_model.encode(texts[:16], batch_size=16, normalize_embeddings=True)

        t0 = time.perf_counter()
        emb_q = q_model.encode(
            texts, batch_size=64, normalize_embeddings=True,
            show_progress_bar=False,
        )
        q_s = time.perf_counter() - t0

        aligned_cos = np.sum(emb_fp * emb_q, axis=1)
        nq = min(30, max(5, len(texts) // 10))
        sim_fp = emb_fp[:nq] @ emb_fp[nq:].T
        sim_q = emb_q[:nq] @ emb_q[nq:].T
        top_fp = np.argsort(-sim_fp, axis=1)[:, :10]
        top_q = np.argsort(-sim_q, axis=1)[:, :10]
        overlap = float(np.mean([
            len(set(a.tolist()) & set(b.tolist())) / 10.0
            for a, b in zip(top_fp, top_q)
        ]))

        speedup = fp_s / max(q_s, 1e-9)
        report = {
            "available": True,
            "fp32_seconds": round(fp_s, 4),
            "int8_seconds": round(q_s, 4),
            "speedup": round(speedup, 3),
            "mean_aligned_embedding_cosine": round(
                float(np.mean(aligned_cos)), 6
            ),
            "p05_aligned_embedding_cosine": round(
                float(np.quantile(aligned_cos, .05)), 6
            ),
            "mean_top10_overlap": round(overlap, 4),
        }
        report["bonus_claim_supported"] = bool(
            speedup > 1.05
            and report["mean_aligned_embedding_cosine"] >= .98
            and overlap >= .90
        )
    except Exception as exc:
        report = {
            "available": False,
            "bonus_claim_supported": False,
            "error": repr(exc),
        }

    (BONUS_OUTPUT_DIR / "quantization_metrics.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return report

quant_bonus = run_quantization_bonus()
bonus_status["quantization"] = quant_bonus
print(json.dumps(quant_bonus, ensure_ascii=False, indent=2))


### Sponsored Search Auction — mentor-requested business extension

طبق پیشنهاد منتور، یک سیستم Sponsored Search Auction برای سه Vendor اضافه شده است.

هر Vendor یک **Max CPC** برای یک Product ثبت می‌کند. سیستم برای هر Query با
ترکیب Bid، کیفیت محصول و Query relevance، `Ad Rank` را محاسبه می‌کند؛ سه
برنده در جایگاه‌های Sponsored شماره 1، 3 و 5 قرار می‌گیرند و Actual CPC با
quality-adjusted **Generalized Second Price (GSP)** محاسبه می‌شود.

تبلیغات در UI با برچسب واضح `تبلیغ · Sponsored` از نتایج Organic جدا می‌شوند.
آزمایش زیر 500 Auction تصادفی را اجرا می‌کند و invariantهای قیمت‌گذاری و
allocation را بررسی می‌کند.


In [ ]:
# Sponsored Search Auction validation + offline simulation.
def quality_adjusted_gsp(
    bids,
    qualities,
    reserve_cpc=1000.0,
):
    bids = np.asarray(bids, dtype=float)
    qualities = np.asarray(qualities, dtype=float)

    if len(bids) != 3 or len(qualities) != 3:
        raise ValueError("Exactly three vendors are required.")

    if np.any(bids < 0):
        raise ValueError("Bids cannot be negative.")

    qualities = np.clip(qualities, 0.05, 1.0)
    ad_rank = bids * qualities
    order = np.argsort(-ad_rank, kind="stable")

    rows = []
    for slot, idx in enumerate(order, start=1):
        next_rank = (
            ad_rank[order[slot]]
            if slot < len(order)
            else reserve_cpc * qualities[idx]
        )

        threshold_cpc = (
            next_rank / max(qualities[idx], 1e-9)
        ) + 1.0

        actual_cpc = min(
            bids[idx],
            max(reserve_cpc, threshold_cpc),
        )

        rows.append({
            "vendor_index": int(idx),
            "slot": int(slot),
            "max_cpc": round(float(bids[idx]), 2),
            "quality": round(float(qualities[idx]), 4),
            "ad_rank": round(float(ad_rank[idx]), 4),
            "actual_cpc": round(float(actual_cpc), 2),
        })

    return rows


def bid_only_auction(
    bids,
    qualities,
    reserve_cpc=1000.0,
):
    """Naive baseline: rank only by bid, ignoring quality/relevance."""
    bids = np.asarray(bids, dtype=float)
    qualities = np.asarray(qualities, dtype=float)

    order = np.argsort(-bids, kind="stable")
    rows = []

    for slot, idx in enumerate(order, start=1):
        next_bid = (
            bids[order[slot]]
            if slot < len(order)
            else reserve_cpc
        )
        actual_cpc = min(
            bids[idx],
            max(reserve_cpc, next_bid + 1.0),
        )
        rows.append({
            "vendor_index": int(idx),
            "slot": int(slot),
            "max_cpc": round(float(bids[idx]), 2),
            "quality": round(float(qualities[idx]), 4),
            "ad_rank": round(float(bids[idx]), 4),
            "actual_cpc": round(float(actual_cpc), 2),
        })

    return rows


def _position_ctr(slot):
    # Transparent presentation-only CTR proxy for offline simulation.
    return {1: 0.12, 2: 0.075, 3: 0.05}.get(int(slot), 0.03)


def _simulated_market_metrics(rows):
    weighted_quality = 0.0
    expected_revenue = 0.0

    for row in rows:
        ctr = _position_ctr(row["slot"]) * float(row["quality"])
        weighted_quality += ctr * float(row["quality"])
        expected_revenue += ctr * float(row["actual_cpc"])

    return {
        "quality_weighted_click_value": weighted_quality,
        "expected_revenue_per_impression_proxy": expected_revenue,
    }


def validate_auction_system(
    seed=42,
    n_invariant_trials=500,
    n_simulation_trials=2000,
):
    rng = np.random.default_rng(seed)
    violations = []

    # 1) Safety / economic invariants.
    for trial in range(n_invariant_trials):
        bids = rng.integers(
            1000,
            50001,
            size=3,
        ).astype(float)

        qualities = rng.uniform(
            0.2,
            1.0,
            size=3,
        )

        rows = quality_adjusted_gsp(
            bids,
            qualities,
            reserve_cpc=1000.0,
        )

        ranks = [r["ad_rank"] for r in rows]
        slots = [r["slot"] for r in rows]
        vendor_ids = [r["vendor_index"] for r in rows]

        checks = {
            "three_unique_winners": len(set(vendor_ids)) == 3,
            "slots_are_1_2_3": slots == [1, 2, 3],
            "rank_is_descending": all(
                ranks[i] >= ranks[i + 1]
                for i in range(len(ranks) - 1)
            ),
            "actual_cpc_never_exceeds_bid": all(
                r["actual_cpc"] <= r["max_cpc"] + 1e-9
                for r in rows
            ),
            "nonnegative_cpc": all(
                r["actual_cpc"] >= 0
                for r in rows
            ),
        }

        if not all(checks.values()):
            violations.append({
                "trial": trial,
                "checks": checks,
            })

    # 2) Offline simulation against a naive highest-bid baseline.
    qa_quality = []
    base_quality = []
    qa_revenue = []
    base_revenue = []

    for _ in range(n_simulation_trials):
        bids = rng.integers(
            1000,
            50001,
            size=3,
        ).astype(float)

        # "quality" represents the combined product-quality/query-relevance
        # factor used by the live dashboard. This is an offline proxy simulation,
        # not observed production CTR.
        qualities = rng.beta(
            2.5,
            1.8,
            size=3,
        )
        qualities = np.clip(qualities, 0.05, 1.0)

        qa_rows = quality_adjusted_gsp(
            bids,
            qualities,
            reserve_cpc=1000.0,
        )
        base_rows = bid_only_auction(
            bids,
            qualities,
            reserve_cpc=1000.0,
        )

        qa_m = _simulated_market_metrics(qa_rows)
        base_m = _simulated_market_metrics(base_rows)

        qa_quality.append(
            qa_m["quality_weighted_click_value"]
        )
        base_quality.append(
            base_m["quality_weighted_click_value"]
        )
        qa_revenue.append(
            qa_m["expected_revenue_per_impression_proxy"]
        )
        base_revenue.append(
            base_m["expected_revenue_per_impression_proxy"]
        )

    mean_qa_quality = float(np.mean(qa_quality))
    mean_base_quality = float(np.mean(base_quality))
    mean_qa_revenue = float(np.mean(qa_revenue))
    mean_base_revenue = float(np.mean(base_revenue))

    quality_lift = (
        mean_qa_quality / max(mean_base_quality, 1e-12) - 1.0
    )
    revenue_lift = (
        mean_qa_revenue / max(mean_base_revenue, 1e-12) - 1.0
    )

    demo = quality_adjusted_gsp(
        bids=[8000, 6200, 5400],
        qualities=[0.72, 0.93, 0.81],
        reserve_cpc=1000.0,
    )

    report = {
        "available": True,
        "mentor_requested": False,
        "mentor_approved_new_problem": bool(MENTOR_APPROVED_AUCTION),
        "mentor_approval_status": (
            "approved" if MENTOR_APPROVED_AUCTION else "pending_confirmation"
        ),
        "problem": "three_vendor_sponsored_search_auction",
        "auction_type": "quality-adjusted generalized second-price",
        "vendors": 3,
        "sponsored_search_positions": [1, 3, 5],
        "reserve_cpc_toman": 1000.0,
        "invariant_trials": int(n_invariant_trials),
        "violations": int(len(violations)),
        "invariant_pass_rate": round(
            1.0 - len(violations) / n_invariant_trials,
            4,
        ),
        "offline_simulation": {
            "trials": int(n_simulation_trials),
            "baseline": "highest-bid ranking + second-price CPC",
            "proposed": "quality-adjusted GSP",
            "mean_quality_weighted_click_value_baseline": round(
                mean_base_quality, 6
            ),
            "mean_quality_weighted_click_value_proposed": round(
                mean_qa_quality, 6
            ),
            "quality_proxy_lift_pct": round(100 * quality_lift, 2),
            "mean_expected_revenue_proxy_baseline": round(
                mean_base_revenue, 4
            ),
            "mean_expected_revenue_proxy_proposed": round(
                mean_qa_revenue, 4
            ),
            "revenue_proxy_lift_pct": round(100 * revenue_lift, 2),
            "important_limitation": (
                "CTR/revenue values are transparent offline simulation proxies, "
                "not observed production advertising outcomes."
            ),
        },
        "demo_allocation": demo,
        "tested_invariants": [
            "unique allocation",
            "slot ordering",
            "descending ad rank",
            "actual CPC <= max CPC",
            "nonnegative CPC",
        ],
    }

    # Claim support requires perfect invariant validation plus a positive
    # quality-proxy lift over the naive bid-only baseline.
    report["technical_result_supported"] = bool(
        len(violations) == 0 and quality_lift > 0
    )
    report["bonus_claim_supported"] = bool(
        MENTOR_APPROVED_AUCTION
        and report["technical_result_supported"]
    )

    (BONUS_OUTPUT_DIR / "auction_system_metrics.json").write_text(
        json.dumps(
            report,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    return report


auction_bonus = validate_auction_system()
bonus_status["auction"] = auction_bonus

print(
    json.dumps(
        auction_bonus,
        ensure_ascii=False,
        indent=2,
    )
)


### 5.5 LoRA fine-tuning experiment

The submitted classifier remains the reproducible text-only
TF-IDF + Logistic Regression model.

The bonus experiment fine-tunes a Persian transformer with LoRA on a
product-grouped split and compares Macro-F1 against the sklearn baseline on the
**same held-out test subset**.

Default: `HooshvareLab/bert-fa-base-uncased`; fallback:
`distilbert-base-multilingual-cased`.

The Fine-tuning bonus is marked supported only if measured Macro-F1 improves.


In [ ]:
def run_lora_bonus():
    if not RUN_LORA_BONUS:
        return {"available": False, "reason": "disabled"}

    try:
        import numpy as np
        import pandas as pd
        import torch
        import joblib
        import time
        from datasets import Dataset
        from peft import LoraConfig, TaskType, get_peft_model
        from sklearn.metrics import f1_score
        from sklearn.model_selection import GroupShuffleSplit
        from sklearn.preprocessing import LabelEncoder
        from transformers import (
            AutoModelForSequenceClassification,
            AutoTokenizer,
            DataCollatorWithPadding,
            Trainer,
            TrainingArguments,
        )

        seed = 42
        max_per_class = 18000
        max_train_rows = 24000
        max_eval_rows = 5000

        df = comments_df[
            comments_df["recommendation_status"].isin(
                ["recommended", "not_recommended", "no_idea"]
            )
            & comments_df["product_id"].notna()
            & comments_df["comment_text_norm"].fillna("").str.len().gt(3)
        ][
            ["product_id", "comment_text_norm", "recommendation_status"]
        ].drop_duplicates("comment_text_norm").copy()

        parts = []
        for _, g in df.groupby("recommendation_status", sort=False):
            parts.append(g.sample(min(len(g), max_per_class), random_state=seed))
        df = pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

        le = LabelEncoder()
        df["label"] = le.fit_transform(df["recommendation_status"])

        split_test = GroupShuffleSplit(n_splits=1, test_size=.30, random_state=seed)
        tv_idx, test_idx = next(split_test.split(df, groups=df["product_id"]))
        tv = df.iloc[tv_idx].reset_index(drop=True)
        test = df.iloc[test_idx].reset_index(drop=True)

        split_val = GroupShuffleSplit(
            n_splits=1, test_size=.25, random_state=seed + 1
        )
        tr_idx, va_idx = next(split_val.split(tv, groups=tv["product_id"]))
        train = tv.iloc[tr_idx].reset_index(drop=True)
        val = tv.iloc[va_idx].reset_index(drop=True)

        assert not (set(train.product_id) & set(val.product_id))
        assert not (set(train.product_id) & set(test.product_id))
        assert not (set(val.product_id) & set(test.product_id))

        if len(train) > max_train_rows:
            chunks = []
            per = max(1, max_train_rows // len(le.classes_))
            for _, g in train.groupby("label", sort=False):
                chunks.append(g.sample(min(len(g), per), random_state=seed))
            train = pd.concat(chunks).sample(
                frac=1, random_state=seed
            ).reset_index(drop=True)

        if len(val) > max_eval_rows:
            val = val.sample(max_eval_rows, random_state=seed).reset_index(drop=True)
        if len(test) > max_eval_rows:
            test = test.sample(
                max_eval_rows, random_state=seed + 2
            ).reset_index(drop=True)

        baseline_bundle = joblib.load(
            config.MODELS_DIR / "recommendation_model.pkl"
        )
        baseline_pred = baseline_bundle["pipeline"].predict(
            test["comment_text_norm"]
        )
        baseline_f1 = f1_score(
            test["label"], baseline_pred, average="macro", zero_division=0
        )

        model_name = None
        tokenizer = None
        base_model = None
        last_error = None

        for candidate in [
            "HooshvareLab/bert-fa-base-uncased",
            "distilbert-base-multilingual-cased",
        ]:
            try:
                tokenizer = AutoTokenizer.from_pretrained(candidate)
                base_model = AutoModelForSequenceClassification.from_pretrained(
                    candidate,
                    num_labels=len(le.classes_),
                )
                model_name = candidate
                break
            except Exception as exc:
                last_error = exc

        if base_model is None:
            raise RuntimeError(
                f"Could not load a LoRA base model: {last_error!r}"
            )

        def tokenize(batch):
            return tokenizer(
                batch["text"], truncation=True, max_length=160
            )

        def make_ds(frame):
            return Dataset.from_dict({
                "text": frame["comment_text_norm"].tolist(),
                "labels": frame["label"].astype(int).tolist(),
            }).map(tokenize, batched=True, remove_columns=["text"])

        train_ds, val_ds, test_ds = map(make_ds, [train, val, test])

        targets = (
            ["q_lin", "v_lin"]
            if "distilbert" in model_name.lower()
            else ["query", "value"]
        )
        model = get_peft_model(
            base_model,
            LoraConfig(
                task_type=TaskType.SEQ_CLS,
                r=8,
                lora_alpha=16,
                lora_dropout=.08,
                target_modules=targets,
                bias="none",
            ),
        )

        trainable = sum(
            p.numel() for p in model.parameters() if p.requires_grad
        )
        total = sum(p.numel() for p in model.parameters())

        def compute_metrics(pred):
            logits, labels = pred
            yhat = np.argmax(logits, axis=-1)
            return {
                "macro_f1": f1_score(
                    labels, yhat, average="macro", zero_division=0
                )
            }

        lora_dir = BONUS_OUTPUT_DIR / "lora_adapter"
        lora_dir.mkdir(parents=True, exist_ok=True)

        args = TrainingArguments(
            output_dir=str(lora_dir / "trainer"),
            learning_rate=2e-4,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            gradient_accumulation_steps=2,
            num_train_epochs=2.0,
            weight_decay=.01,
            warmup_ratio=.05,
            eval_strategy="epoch",
            save_strategy="epoch",
            logging_steps=50,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=True,
            fp16=torch.cuda.is_available(),
            report_to=[],
            seed=seed,
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            tokenizer=tokenizer,
            data_collator=DataCollatorWithPadding(tokenizer),
            compute_metrics=compute_metrics,
        )

        t0 = time.perf_counter()
        trainer.train()
        training_s = time.perf_counter() - t0
        metrics = trainer.evaluate(test_ds, metric_key_prefix="test")
        lora_f1 = float(metrics["test_macro_f1"])

        trainer.model.save_pretrained(lora_dir)
        tokenizer.save_pretrained(lora_dir)

        report = {
            "available": True,
            "base_model": model_name,
            "same_test_subset_baseline_macro_f1": round(
                float(baseline_f1), 4
            ),
            "lora_macro_f1": round(lora_f1, 4),
            "lift_vs_same_subset_baseline": round(
                lora_f1 - float(baseline_f1), 4
            ),
            "train_rows": int(len(train)),
            "val_rows": int(len(val)),
            "test_rows": int(len(test)),
            "trainable_parameters": int(trainable),
            "total_parameters": int(total),
            "trainable_parameter_pct": round(
                100 * trainable / total, 4
            ),
            "training_seconds": round(training_s, 2),
            "device": "cuda" if torch.cuda.is_available() else "cpu",
        }
        report["bonus_claim_supported"] = bool(
            lora_f1 > baseline_f1 + 1e-6
        )
    except Exception as exc:
        report = {
            "available": False,
            "bonus_claim_supported": False,
            "error": repr(exc),
        }

    (BONUS_OUTPUT_DIR / "lora_metrics.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return report

lora_bonus = run_lora_bonus()
bonus_status["lora"] = lora_bonus
print(json.dumps(lora_bonus, ensure_ascii=False, indent=2))


### 5.6 Human validation — inside the notebook

The widget below uses the actual 10 held-out answers from Phase 4. Rate each
answer directly here; no external spreadsheet is required.

For a stronger claim, have a second person independently rate the same samples.
One rater alone does not establish inter-rater agreement.

The widget saves `human_eval_labels.csv` and updates
`bonus_outputs/human_eval_validation.json`.


In [ ]:
def build_human_eval_widget():
    try:
        import ipywidgets as widgets
        from IPython.display import display, clear_output
        import pandas as pd
        import numpy as np
        from scipy.stats import spearmanr

        pq = phase4["per_query"].reset_index(drop=True).copy()
        labels_path = Path.cwd() / "human_eval_labels.csv"

        idx_w = widgets.IntSlider(
            value=0, min=0, max=max(0, len(pq)-1),
            description="Sample", continuous_update=False
        )
        rel_w = widgets.IntSlider(value=4, min=1, max=5, description="Relevance")
        grd_w = widgets.IntSlider(value=4, min=1, max=5, description="Grounding")
        cmp_w = widgets.IntSlider(value=4, min=1, max=5, description="Complete")
        note_w = widgets.Textarea(
            description="Note", layout=widgets.Layout(width="80%")
        )
        save_w = widgets.Button(
            description="Save rating", button_style="success"
        )
        out_w = widgets.Output()

        def load_labels():
            if labels_path.exists():
                return pd.read_csv(labels_path)
            return pd.DataFrame(columns=[
                "sample_index", "query", "human_relevance_1_5",
                "human_grounding_1_5", "human_task_completion_1_5",
                "human_note",
            ])

        def corr_report(labels):
            merged = labels.merge(
                pq.reset_index().rename(columns={"index": "sample_index"}),
                on="sample_index", how="inner"
            )

            def corr(hcol, acol):
                if acol not in merged or len(merged) < 3:
                    return None
                x = pd.to_numeric(merged[hcol], errors="coerce")
                y = pd.to_numeric(merged[acol], errors="coerce")
                m = x.notna() & y.notna()
                if m.sum() < 3:
                    return None
                rho, p = spearmanr(x[m], y[m])
                return {
                    "n": int(m.sum()),
                    "rho": None if np.isnan(rho) else round(float(rho), 4),
                    "p": None if np.isnan(p) else round(float(p), 6),
                    "mae": round(float(np.mean(np.abs(x[m]-y[m]))), 4),
                }

            return {
                "n_human_labels": int(len(labels)),
                "relevance_vs_proxy": corr(
                    "human_relevance_1_5", "proxy_relevance_0_5"
                ),
                "grounding_vs_proxy": corr(
                    "human_grounding_1_5", "proxy_grounding_0_5"
                ),
                "completion_vs_proxy": corr(
                    "human_task_completion_1_5",
                    "task_completion_proxy_0_5",
                ),
                "relevance_vs_llm_judge": corr(
                    "human_relevance_1_5", "judge_relevance_0_5"
                ),
                "grounding_vs_llm_judge": corr(
                    "human_grounding_1_5",
                    "judge_faithfulness_0_5",
                ),
                "limitations": (
                    "One-rater labels do not establish inter-rater agreement."
                ),
            }

        def render(*_):
            with out_w:
                clear_output(wait=True)
                row = pq.iloc[int(idx_w.value)]
                print("QUERY:")
                print(row["query"])
                print("\\nANSWER:")
                print(row.get("answer_text", ""))
                print("\\nEVIDENCE:")
                ev = str(row.get("evidence_text", ""))
                print(ev[:3500] if ev else "(no serialized evidence)")
                labels = load_labels()
                print(f"\\nProgress: {len(labels)}/{len(pq)} labeled")
                if len(labels):
                    print(json.dumps(
                        corr_report(labels),
                        ensure_ascii=False,
                        indent=2,
                    ))

        def save_rating(_):
            labels = load_labels()
            i = int(idx_w.value)
            row = pq.iloc[i]
            rec = pd.DataFrame([{
                "sample_index": i,
                "query": row["query"],
                "human_relevance_1_5": int(rel_w.value),
                "human_grounding_1_5": int(grd_w.value),
                "human_task_completion_1_5": int(cmp_w.value),
                "human_note": note_w.value,
            }])
            if len(labels):
                labels = labels[labels["sample_index"] != i]
            labels = pd.concat([labels, rec], ignore_index=True)
            labels.to_csv(labels_path, index=False, encoding="utf-8-sig")

            report = corr_report(labels)
            (BONUS_OUTPUT_DIR / "human_eval_validation.json").write_text(
                json.dumps(report, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )

            note_w.value = ""
            if i < len(pq)-1:
                idx_w.value = i + 1
            render()

        idx_w.observe(render, names="value")
        save_w.on_click(save_rating)

        display(widgets.VBox([
            widgets.HTML(
                "<b>Human evaluation:</b> rate the held-out answer below."
            ),
            idx_w,
            widgets.HBox([rel_w, grd_w, cmp_w]),
            note_w,
            save_w,
            out_w,
        ]))
        render()

        return {
            "available": True,
            "n_queries": len(pq),
            "labels_path": str(labels_path),
        }
    except Exception as exc:
        print("Human evaluation widget unavailable:", repr(exc))
        return {"available": False, "error": repr(exc)}

human_widget_status = build_human_eval_widget()
bonus_status["human_widget"] = human_widget_status


In [ ]:
def summarize_human_validation():
    labels_path = Path.cwd() / "human_eval_labels.csv"
    if not labels_path.exists():
        return {"available": False, "reason": "human_eval_labels.csv not found"}

    labels = pd.read_csv(labels_path)
    auto = phase4["per_query"].reset_index(drop=True).reset_index().rename(columns={"index": "sample_index"})
    merged = labels.merge(auto, on=["sample_index", "query"], how="inner", suffixes=("_human", "_auto"))
    if len(merged) == 0:
        return {"available": False, "reason": "No labels match current sample_index + query pairs."}

    def corr(human_col, auto_col):
        if auto_col not in merged.columns:
            return None
        x = pd.to_numeric(merged[human_col], errors="coerce")
        y = pd.to_numeric(merged[auto_col], errors="coerce")
        m = x.notna() & y.notna()
        if m.sum() < 3:
            return None
        from scipy.stats import spearmanr
        rho, p = spearmanr(x[m], y[m])
        return {
            "n": int(m.sum()),
            "rho": None if np.isnan(rho) else round(float(rho), 4),
            "p_value": None if np.isnan(p) else round(float(p), 6),
            "mae": round(float(np.mean(np.abs(x[m] - y[m]))), 4),
        }

    report = {
        "available": True,
        "n_human_labels": int(len(labels)),
        "n_labels_matching_current_run": int(len(merged)),
        "mean_human_relevance_1_5": round(float(pd.to_numeric(merged["human_relevance_1_5"], errors="coerce").mean()), 3),
        "mean_human_grounding_1_5": round(float(pd.to_numeric(merged["human_grounding_1_5"], errors="coerce").mean()), 3),
        "mean_human_task_completion_1_5": round(float(pd.to_numeric(merged["human_task_completion_1_5"], errors="coerce").mean()), 3),
        "relevance_vs_proxy": corr("human_relevance_1_5", "proxy_relevance_0_5"),
        "grounding_vs_proxy": corr("human_grounding_1_5", "proxy_grounding_0_5"),
        "task_completion_vs_proxy": corr("human_task_completion_1_5", "task_completion_proxy_0_5"),
        "relevance_vs_llm_judge": corr("human_relevance_1_5", "judge_relevance_0_5"),
        "grounding_vs_llm_judge": corr("human_grounding_1_5", "judge_faithfulness_0_5"),
        "limitations": "Single-rater labels do not establish inter-rater agreement.",
    }
    (BONUS_OUTPUT_DIR / "human_eval_validation.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return report


human_validation_summary = summarize_human_validation()
bonus_status["human_validation"] = human_validation_summary
print(json.dumps(human_validation_summary, ensure_ascii=False, indent=2))


### 5.7 Streamlit presentation UI — auto-generated and auto-launched

The notebook writes `streamlit_app.py` from the embedded source and attempts to
launch it automatically at the end of Run All.

Features:
- Dark / Light toggle;
- Try-it product search;
- Review QA with optional Metis answer;
- comparison;
- managerial dashboard;
- model/evaluation dashboard;
- Human Evaluation tab.

The UI is a presentation/product layer. Scientific metrics remain those from
the notebook.


In [ ]:
# Generate and launch the embedded Streamlit app.
import socket
import subprocess
import sys
import time
import webbrowser
from pathlib import Path

STREAMLIT_APP_PATH = Path.cwd() / "streamlit_app.py"
STREAMLIT_APP_SOURCE = 'from __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport os\nimport re\nimport time\nfrom collections import Counter\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport plotly.express as px\nimport scipy.sparse as sp\nimport streamlit as st\nimport streamlit.components.v1 as components\n\nAPP_TITLE = "Digikala AI Shopping Assistant"\nDEFAULT_METIS_BASE_URL = "https://api.metisai.ir/openai/v1"\nDEFAULT_METIS_MODEL = "gpt-4o-mini"\nTOKEN_RE = re.compile(r"[\u0600-ۿ0-9A-Za-z]+")\nNEGATIVE_CUES = (\n    "بد", "ضعیف", "مشکل", "ایراد", "خراب", "شکسته", "پاره", "ناراضی",\n    "بی کیفیت", "بی\u200cکیفیت", "افتضاح", "گران", "کوچیک", "کوچک", "سخت",\n)\nSTOPWORDS = {\n    "و", "در", "به", "از", "که", "این", "با", "را", "برای", "رو", "هم",\n    "یک", "ها", "است", "بود", "شد", "شود", "من", "خیلی", "ولی", "اما",\n}\n\nst.set_page_config(\n    page_title=APP_TITLE,\n    page_icon="🛍️",\n    layout="wide",\n    initial_sidebar_state="collapsed",\n)\n\n\ndef fa_norm(text) -> str:\n    if text is None:\n        return ""\n    text = str(text).replace("ي", "ی").replace("ك", "ک")\n    text = text.replace("\\u200c", " ")\n    return re.sub(r"\\s+", " ", text).strip().lower()\n\n\ndef tokens(text):\n    return [t for t in TOKEN_RE.findall(fa_norm(text)) if t not in STOPWORDS]\n\n\ndef inject_theme(dark: bool):\n    if dark:\n        p = {\n            "bg": "#071426", "panel": "#0B1F3A", "panel2": "#102A4C",\n            "text": "#FFFFFF", "muted": "#D7E5F5", "border": "#6FB8FF",\n            "sky": "#68BEFF", "sky_soft": "#16385A", "navy": "#0B1F3A",\n            "orange": "#FF9A32", "orange_soft": "#4B2C12",\n            "success": "#48D597", "danger": "#FF6B6B", "input_bg": "#102A4C",\n            "shadow": "0 12px 34px rgba(0,0,0,.30)", "tab_bg": "#13365E",\n            "hero_toggle": "rgba(255,255,255,.12)", "surface": "rgba(255,255,255,.08)",\n        }\n    else:\n        p = {\n            "bg": "#EEF6FF", "panel": "#FFFFFF", "panel2": "#E4F1FF",\n            "text": "#101828", "muted": "#475467", "border": "#A9CCE8",\n            "sky": "#3FA9F5", "sky_soft": "#EAF6FF", "navy": "#0B1F3A",\n            "orange": "#FF8A00", "orange_soft": "#FFF0DF",\n            "success": "#138A60", "danger": "#D64550", "input_bg": "#FDFEFF",\n            "shadow": "0 10px 28px rgba(20,54,92,.10)", "tab_bg": "#F7FBFF",\n            "hero_toggle": "rgba(255,255,255,.20)", "surface": "#F8FBFF",\n        }\n\n    st.markdown(\n        f"""\n        <style>\n        html, body, [class*="css"] {{font-family: Tahoma, "Segoe UI", sans-serif;}}\n        .stApp {{background:{p[\'bg\']}; color:{p[\'text\']}; direction:rtl;}}\n        section.main > div.block-container {{padding-top:.65rem; max-width: 96rem;}}\n\n        .st-key-hero_panel {{\n            background:linear-gradient(120deg,{p[\'navy\']} 0%,#174778 56%,{p[\'sky\']} 100%);\n            border:1px solid {p[\'border\']}; border-radius:24px; padding:22px 24px 18px;\n            box-shadow:{p[\'shadow\']}; margin-bottom:14px; position:sticky; top:8px; z-index:1000;\n        }}\n        .st-key-hero_panel h1,.st-key-hero_panel h2,.st-key-hero_panel h3,\n        .st-key-hero_panel p,.st-key-hero_panel label,.st-key-hero_panel span {{color:#FFF!important;}}\n        .st-key-hero_panel [data-testid="stToggle"] {{\n            background:{p[\'hero_toggle\']}; border:1px solid rgba(255,255,255,.38);\n            border-radius:16px; padding:10px 12px; box-shadow: inset 0 0 0 1px rgba(255,255,255,.05);\n        }}\n\n        [data-testid="stMetric"] {{background:{p[\'panel\']};border:1px solid {p[\'border\']};padding:14px 16px;border-radius:18px;box-shadow:{p[\'shadow\']};}}\n        [data-testid="stMetricLabel"], [data-testid="stMetricLabel"] *, [data-testid="stMetricValue"], [data-testid="stMetricValue"] *, [data-testid="stMetricDelta"], [data-testid="stMetricDelta"] * {{color:{p[\'text\']}!important;}}\n        .card {{background:{p[\'panel\']};border:1px solid {p[\'border\']};border-radius:18px;padding:16px 18px;box-shadow:{p[\'shadow\']};margin:8px 0 12px;color:{p[\'text\']};}}\n        .card-title {{font-size:1.04rem;font-weight:800;color:{p[\'text\']};}}\n        .pill {{display:inline-block;padding:5px 10px;border-radius:999px;margin:4px 2px;background:{p[\'sky_soft\']};border:1px solid {p[\'border\']};color:{p[\'text\']};font-size:.78rem;}}\n        .sponsored-pill {{display:inline-block;padding:5px 10px;border-radius:999px;margin:4px 2px;background:{p[\'orange\']};color:#FFF;font-size:.78rem;font-weight:800;}}\n        .auction-card {{background:{p[\'panel\']};border:1px solid {p[\'border\']};border-right:6px solid {p[\'orange\']};border-radius:18px;padding:16px 18px;box-shadow:{p[\'shadow\']};margin:8px 0 12px;}}\n        .help-box {{background:{p[\'panel2\']};border:1px solid {p[\'border\']};border-right:5px solid {p[\'sky\']};border-radius:16px;padding:14px 16px;color:{p[\'text\']};margin:8px 0 14px;}}\n        .score-guide {{background:{p[\'orange_soft\']};border:1px solid {p[\'border\']};border-right:5px solid {p[\'orange\']};border-radius:16px;padding:13px 16px;color:{p[\'text\']};margin:8px 0 14px;}}\n        .soft-panel {{background:{p[\'surface\']}; border:1px solid {p[\'border\']}; border-radius:16px;padding:14px 16px; margin:8px 0 12px; box-shadow:{p[\'shadow\']};}}\n        .muted {{color:{p[\'muted\']};}} .ok {{color:{p[\'success\']};font-weight:800;}} .bad {{color:{p[\'danger\']};font-weight:800;}}\n        div[data-testid="stForm"],div[data-testid="stExpander"],div[data-testid="stAlert"] {{background:{p[\'panel\']};border:1px solid {p[\'border\']};border-radius:16px;}}\n        [data-baseweb="input"] > div,[data-baseweb="select"] > div,[data-testid="stTextArea"] textarea,[data-testid="stNumberInput"] input,[data-testid="stTextInput"] input,[data-testid="stMultiSelect"] > div > div {{background:{p[\'input_bg\']}!important;color:{p[\'text\']}!important;border:1px solid {p[\'border\']}!important;border-radius:14px!important;-webkit-text-fill-color:{p[\'text\']}!important;}}\n        div[data-testid="stNumberInput"] button, div[data-testid="stTextInput"] button, div[data-testid="stMultiSelect"] button {{background:{p[\'panel2\']}!important; color:{p[\'navy\']}!important; border:1px solid {p[\'border\']}!important;border-radius:10px!important;}}\n        div[data-testid="stNumberInput"] button:hover, div[data-testid="stTextInput"] button:hover, div[data-testid="stMultiSelect"] button:hover {{background:{p[\'sky_soft\']}!important;}}\n        [data-baseweb="input"] input,[data-testid="stTextArea"] textarea,[data-testid="stJson"] *, [data-testid="stDataFrame"] * {{color:{p[\'text\']}!important;-webkit-text-fill-color:{p[\'text\']}!important;}}\n        [data-testid="stWidgetLabel"] p,[data-testid="stMarkdownContainer"] p,[data-testid="stMarkdownContainer"] li,label,span, small {{color:{p[\'text\']};}}\n        .stButton>button,.stDownloadButton>button,div[data-testid="stFormSubmitButton"] button {{background:{p[\'navy\']};color:#FFF!important;border:1px solid {(\'#FFFFFF\' if dark else p[\'sky\'])};border-radius:13px;font-weight:750; min-height:2.6rem;}}\n        .stButton>button:hover,.stDownloadButton>button:hover,div[data-testid="stFormSubmitButton"] button:hover {{background:{p[\'sky\']};color:{(\'#071426\' if dark else \'#FFFFFF\')}!important;border-color:{(\'#FFFFFF\' if dark else p[\'sky\'])};}}\n        .stTabs [data-baseweb="tab-list"] {{gap:8px; flex-wrap:nowrap; width:100%; position:sticky; top:154px; z-index:998;background:{p[\'bg\']}; padding:6px 0 10px 0; border-bottom:1px solid {p[\'border\']};}}\n        .stTabs [data-baseweb="tab"] {{background:{(p[\'tab_bg\'] if not dark else \'#12345A\')}; color:{(\'#EAF6FF\' if dark else p[\'text\'])}!important;border:1px solid {(\'#FFFFFF\' if dark else p[\'border\'])}; border-radius:12px 12px 0 0;padding:10px 14px; flex:1 1 0%; justify-content:center; text-align:center;}}\n        .stTabs [data-baseweb="tab"]:hover {{background:{p[\'sky_soft\']};}}\n        .stTabs [data-baseweb="tab"][aria-selected="true"] {{background:{p[\'panel\']}; color:{p[\'orange\']}!important; border-bottom:3px solid {p[\'orange\']};box-shadow:{p[\'shadow\']};}}\n        table.nice-table {{width:100%; border-collapse:collapse; font-size:.95rem;}}\n        table.nice-table th, table.nice-table td {{border:1px solid {p[\'border\']}; padding:10px 12px; text-align:center; color:{p[\'text\']};background:{p[\'panel\']}; vertical-align:middle;}}\n        table.nice-table th {{background:{p[\'panel2\']}; font-weight:800;}}\n        .table-wrap {{background:{p[\'panel\']}; border:1px solid {p[\'border\']}; border-radius:18px;overflow-x:auto; padding:8px; box-shadow:{p[\'shadow\']}; margin:8px 0 12px;}}\n        .finder-row {{background:{p[\'panel\']}; border:1px solid {p[\'border\']}; border-radius:18px;padding:14px 16px; margin:8px 0; box-shadow:{p[\'shadow\']};}}\n        code {{color:{p[\'orange\']};}} hr {{border-color:{p[\'border\']};}}\n        #backToTop {{position:fixed; left:24px; bottom:28px; width:50px; height:50px; border:none;border-radius:999px; background:{p[\'navy\']}; color:#FFF; font-size:22px; cursor:pointer;box-shadow:{p[\'shadow\']}; opacity:0; pointer-events:none; transition:all .25s ease; z-index:1200;}}\n        #backToTop.show {{opacity:1; pointer-events:auto;}}\n        </style>\n        """,\n        unsafe_allow_html=True,\n    )\n\n\ndef find_artifact_run() -> Path | None:\n    candidates = []\n    for base in (Path.cwd(), Path.cwd().parent):\n        art = base / "nb_artifacts"\n        if not art.exists():\n            continue\n        latest = art / "latest_run.txt"\n        if latest.exists():\n            try:\n                raw = latest.read_text(encoding="utf-8").strip()\n                p = Path(raw)\n                if not p.is_absolute():\n                    p = art / raw\n                if p.exists():\n                    return p\n            except Exception:\n                pass\n        candidates.extend(p for p in art.glob("run_*") if p.is_dir())\n    return max(candidates, key=lambda p: p.stat().st_mtime) if candidates else None\n\n\nRUN_DIR = find_artifact_run()\n\n\n@st.cache_data(show_spinner=False)\ndef read_json(path: str):\n    p = Path(path)\n    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}\n\n\n@st.cache_data(show_spinner="در حال بارگذاری داده\u200cها...")\ndef load_products(path: str):\n    return pd.read_parquet(path)\n\n\n@st.cache_data(show_spinner="در حال بارگذاری نظرات...")\ndef load_comments(path: str):\n    return pd.read_parquet(path)\n\n\n@st.cache_resource(show_spinner="در حال بارگذاری BM25...")\ndef load_bm25(index_dir: str):\n    base = Path(index_dir)\n    counts = sp.load_npz(base / "bm25_counts.npz").tocsr()\n    doc_len = np.load(base / "bm25_doc_len.npy")\n    idf = np.load(base / "bm25_idf.npy")\n    vocab = json.loads((base / "bm25_vocab.json").read_text(encoding="utf-8"))\n    return counts, doc_len, idf, vocab\n\n\n@st.cache_data(show_spinner=False)\ndef build_product_stats(comments_path: str):\n    comments = load_comments(comments_path).copy()\n    comments["product_id"] = pd.to_numeric(comments["product_id"], errors="coerce")\n    comments = comments[comments["product_id"].notna()].copy()\n    comments["product_id"] = comments["product_id"].astype(int)\n    if "recommendation_status" in comments.columns:\n        valid = comments["recommendation_status"].isin(["recommended", "not_recommended", "no_idea"])\n        rec = comments.loc[valid, ["product_id", "recommendation_status"]].copy()\n        rec["recommended_flag"] = (rec["recommendation_status"] == "recommended").astype(float)\n        rec_stats = rec.groupby("product_id").agg(\n            labeled_reviews=("recommended_flag", "size"),\n            recommendation_rate=("recommended_flag", "mean"),\n        )\n    else:\n        rec_stats = pd.DataFrame()\n    if "rate_clean" in comments.columns:\n        rate = comments.groupby("product_id")["rate_clean"].agg(["mean", "count"]).rename(\n            columns={"mean": "review_rate_mean", "count": "review_count"}\n        )\n    else:\n        rate = comments.groupby("product_id").size().to_frame("review_count")\n    out = rate.join(rec_stats, how="left").reset_index()\n    if "recommendation_rate" in out:\n        out["recommendation_rate"] *= 100\n    return out\n\n\ndef bm25_scores(query, counts, doc_len, idf, vocab, k1=1.5, b=0.75):\n    q_tokens = [t for t in tokens(query) if t in vocab]\n    if not q_tokens:\n        return np.zeros(counts.shape[0], dtype=np.float32)\n    avgdl = max(float(np.mean(doc_len)), 1e-9)\n    scores = np.zeros(counts.shape[0], dtype=np.float32)\n    for tok in q_tokens:\n        j = int(vocab[tok])\n        tf = counts[:, j].toarray().ravel().astype(np.float32)\n        denom = tf + k1 * (1.0 - b + b * doc_len / avgdl)\n        scores += idf[j] * ((tf * (k1 + 1.0)) / np.maximum(denom, 1e-9))\n    return scores\n\n\ndef get_col(df, *names):\n    for name in names:\n        if name in df.columns:\n            return name\n    return None\n\n\ndef fmt_money(x):\n    if x is None or pd.isna(x):\n        return "نامشخص"\n    return f"{float(x):,.0f} تومان"\n\n\ndef product_card(row, stats=None):\n    title = row.get("title_fa") or row.get("title_norm") or row.get("product_text_norm") or "محصول"\n    pid = int(row["product_id"])\n    price = row.get("price_clean")\n    brand = row.get("brand") or row.get("brand_norm") or "—"\n    cat = row.get("category1") or row.get("category1_norm") or "—"\n    rate = row.get("product_rate_clean")\n    rec = stats.get("recommendation_rate") if stats else None\n    nrev = stats.get("review_count") if stats else None\n    pills = [f"ID: {pid}", f"برند: {brand}", f"دسته: {cat}", f"قیمت: {fmt_money(price)}"]\n    if pd.notna(rate): pills.append(f"امتیاز محصول: {float(rate):.2f}")\n    if rec is not None and pd.notna(rec): pills.append(f"توصیه کاربران: {float(rec):.1f}%")\n    if nrev is not None and pd.notna(nrev): pills.append(f"Review نمونه: {int(nrev)}")\n    pills_html = "".join(f\'<span class="pill">{x}</span>\' for x in pills)\n    st.markdown(\n        f\'<div class="card"><div class="card-title">{title}</div><div style="margin-top:8px">{pills_html}</div></div>\',\n        unsafe_allow_html=True,\n    )\n\n\ndef review_text(row):\n    for col in ("comment_text_norm", "body_norm", "body", "title_norm"):\n        val = row.get(col)\n        if isinstance(val, str) and val.strip():\n            return val.strip()\n    return ""\n\n\ndef retrieve_reviews(comments, product_id, query, k=6):\n    subset = comments[pd.to_numeric(comments["product_id"], errors="coerce") == int(product_id)].copy()\n    if subset.empty:\n        return subset\n    q = set(tokens(query))\n    def score(row):\n        text = review_text(row)\n        overlap = len(q & set(tokens(text)))\n        neg_bonus = 0.0\n        if any(c in fa_norm(query) for c in ("مشکل", "ایراد", "ضعف", "بد", "ناراضی")):\n            neg_bonus = 2.0 if any(c in fa_norm(text) for c in NEGATIVE_CUES) else 0.0\n        return overlap + neg_bonus\n    subset["_score"] = subset.apply(score, axis=1)\n    sort_cols = ["_score"] + (["likes"] if "likes" in subset.columns else [])\n    return subset.sort_values(sort_cols, ascending=False).head(k)\n\n\ndef metis_answer(prompt, max_tokens=350):\n    key = os.getenv("METIS_API_KEY")\n    if not key:\n        return None, None, "METIS_API_KEY در محیط سیستم دیده نشد."\n    try:\n        from openai import OpenAI\n        client = OpenAI(api_key=key, base_url=os.getenv("METIS_BASE_URL", DEFAULT_METIS_BASE_URL))\n        t0 = time.perf_counter()\n        resp = client.chat.completions.create(\n            model=os.getenv("METIS_MODEL", DEFAULT_METIS_MODEL),\n            messages=[\n                {"role": "system", "content": "تو دستیار خرید دیجی\u200cکالا هستی. فقط از شواهد داده\u200cشده استفاده کن. اگر شواهد کافی نیست صریح بگو. اعداد و IDها را تغییر نده. پاسخ فارسی و کوتاه باشد."},\n                {"role": "user", "content": prompt},\n            ],\n            temperature=0.1,\n            max_tokens=max_tokens,\n        )\n        latency = time.perf_counter() - t0\n        usage = resp.usage\n        return resp.choices[0].message.content, {\n            "input_tokens": getattr(usage, "prompt_tokens", 0),\n            "output_tokens": getattr(usage, "completion_tokens", 0),\n            "total_tokens": getattr(usage, "total_tokens", 0),\n            "latency_s": round(latency, 3),\n        }, None\n    except Exception as exc:\n        return None, None, str(exc)\n\n\ndef auction_quality(row, stat, query_relevance=0.0):\n    rate = pd.to_numeric(pd.Series([row.get("product_rate_clean")]), errors="coerce").iloc[0]\n    rate_norm = float(np.clip(rate / 5.0, 0, 1)) if pd.notna(rate) else 0.5\n    rec = stat.get("recommendation_rate")\n    rec_norm = float(np.clip(float(rec) / 100.0, 0, 1)) if rec is not None and pd.notna(rec) else 0.5\n    rel_norm = float(np.clip(query_relevance, 0, 1))\n    return float(np.clip(0.40 * rate_norm + 0.35 * rec_norm + 0.25 * rel_norm, 0.05, 1.0))\n\n\ndef run_three_vendor_auction(products, stats_idx, campaigns, scores=None, eligible_mask=None, reserve_cpc=1000.0):\n    pid_to_pos = {int(pid): int(pos) for pos, pid in enumerate(products["product_id"].astype(int))}\n    max_query_score = 0.0\n    if scores is not None and len(scores):\n        positive = np.asarray(scores)[np.asarray(scores) > 0]\n        if positive.size:\n            max_query_score = float(np.max(positive))\n    ranked = []\n    for campaign in campaigns[:3]:\n        if not campaign.get("active", True):\n            continue\n        pid = int(campaign["product_id"])\n        pos = pid_to_pos.get(pid)\n        if pos is None or (eligible_mask is not None and not bool(eligible_mask[pos])):\n            continue\n        row = products.iloc[pos]\n        stat = stats_idx.loc[pid].to_dict() if pid in stats_idx.index else {}\n        raw_rel = float(scores[pos]) if scores is not None else 0.0\n        rel = raw_rel / max_query_score if max_query_score > 0 else 0.5\n        quality = auction_quality(row, stat, rel)\n        bid = max(0.0, float(campaign["max_cpc"]))\n        ranked.append({**campaign, "product_id": pid, "product_pos": pos, "quality": quality,\n                       "query_relevance": float(np.clip(rel, 0, 1)), "ad_rank": bid * quality})\n    ranked = sorted(ranked, key=lambda x: x["ad_rank"], reverse=True)\n    for i, item in enumerate(ranked):\n        next_rank = ranked[i+1]["ad_rank"] if i+1 < len(ranked) else reserve_cpc * item["quality"]\n        threshold = next_rank / max(item["quality"], 1e-9) + 1.0\n        item["slot"] = i + 1\n        item["placement_position"] = [1, 3, 5][min(i, 2)]\n        item["actual_cpc"] = min(float(item["max_cpc"]), max(reserve_cpc, threshold))\n    return ranked[:3]\n\n\ndef sponsored_product_card(row, stat, auction_row):\n    title = row.get("title_fa") or row.get("title_norm") or row.get("product_text_norm") or "محصول"\n    pid = int(row["product_id"])\n    brand = row.get("brand") or row.get("brand_norm") or "—"\n    st.markdown(\n        f\'\'\'<div class="auction-card"><div><span class="sponsored-pill">تبلیغ · Sponsored</span>\n        <span class="pill">{auction_row.get("vendor_name", "Vendor")}</span></div>\n        <div class="card-title" style="margin-top:8px">{title}</div>\n        <div style="margin-top:8px"><span class="pill">Product ID: {pid}</span>\n        <span class="pill">برند: {brand}</span><span class="pill">قیمت: {fmt_money(row.get("price_clean"))}</span></div></div>\'\'\',\n        unsafe_allow_html=True,\n    )\n    with st.expander("جزئیات Auction برای Demo", expanded=False):\n        c1, c2, c3, c4 = st.columns(4)\n        c1.metric("Max CPC", f\'{auction_row["max_cpc"]:,.0f} تومان\')\n        c2.metric("Actual CPC", f\'{auction_row["actual_cpc"]:,.0f} تومان\')\n        c3.metric("Quality", f\'{auction_row["quality"]:.3f}\')\n        c4.metric("Ad Rank", f\'{auction_row["ad_rank"]:,.1f}\')\n\n\ndef render_table(df: pd.DataFrame, max_rows: int | None = None):\n    if df is None or len(df) == 0:\n        st.info("داده\u200cای برای نمایش وجود ندارد.")\n        return\n    show = df.head(max_rows).copy() if max_rows else df.copy()\n    show = show.fillna("—")\n    st.markdown(\n        f\'<div class="table-wrap">{show.to_html(index=False, escape=False, classes="nice-table")}</div>\',\n        unsafe_allow_html=True,\n    )\n\n\ndef product_label(product_id: int) -> str:\n    try:\n        row = products.loc[products["product_id"].astype(int) == int(product_id)].iloc[0]\n    except Exception:\n        return str(product_id)\n    title = str(row.get(title_col, "محصول"))[:55]\n    cat = str(row.get(cat_col, "—")) if cat_col else "—"\n    return f"{int(product_id)} | {title} | {cat}"\n\n\ndef search_catalog(df: pd.DataFrame, query: str, limit: int = 10) -> pd.DataFrame:\n    if not query or not query.strip():\n        return pd.DataFrame()\n    qn = fa_norm(query)\n    q_tokens = tokens(qn) or [qn]\n    base = pd.DataFrame({\n        "product_id": df["product_id"].astype(int),\n        "title": df[title_col].fillna("").astype(str) if title_col else "",\n        "brand": df[brand_col].fillna("").astype(str) if brand_col else "",\n        "category": df[cat_col].fillna("").astype(str) if cat_col else "",\n        "price": pd.to_numeric(df[price_col], errors="coerce") if price_col else np.nan,\n        "rating": pd.to_numeric(df[rating_col], errors="coerce") if rating_col else np.nan,\n    })\n    combo = (base["title"] + " " + base["brand"] + " " + base["category"]).map(fa_norm)\n    score = np.zeros(len(base), dtype=int)\n    for tok in q_tokens:\n        tok = tok.strip()\n        if not tok:\n            continue\n        score += combo.str.contains(re.escape(tok), regex=True, na=False).astype(int)\n    out = base.loc[score > 0].copy()\n    if out.empty:\n        return out\n    out["match_score"] = score[score > 0]\n    out = out.sort_values(["match_score", "rating"], ascending=[False, False])\n    cols = ["product_id", "title", "brand", "category", "price", "rating", "match_score"]\n    return out[cols].head(limit)\n\n\ndef install_back_to_top():\n    # JavaScript runs in a tiny component iframe and controls the parent Streamlit page.\n    components.html(\n        """\n        <script>\n        (() => {\n          const win = window.parent;\n          const doc = win.document;\n          let btn = doc.getElementById(\'backToTop\');\n          if (!btn) {\n            btn = doc.createElement(\'button\');\n            btn.id = \'backToTop\';\n            btn.type = \'button\';\n            btn.title = \'رفتن به بالای صفحه\';\n            btn.innerHTML = \'↑\';\n            doc.body.appendChild(btn);\n          }\n          const candidates = () => [\n            win,\n            doc.scrollingElement,\n            doc.documentElement,\n            doc.body,\n            doc.querySelector(\'[data-testid="stAppViewContainer"]\'),\n            doc.querySelector(\'section.main\')\n          ].filter(Boolean);\n          const scrollY = () => Math.max(...candidates().map(x => {\n            try { return Number(x.scrollY ?? x.scrollTop ?? 0); } catch(e) { return 0; }\n          }));\n          const sync = () => {\n            if (scrollY() > 260) btn.classList.add(\'show\');\n            else btn.classList.remove(\'show\');\n          };\n          btn.onclick = () => {\n            candidates().forEach(x => {\n              try {\n                if (typeof x.scrollTo === \'function\') x.scrollTo({top: 0, behavior: \'smooth\'});\n                else x.scrollTop = 0;\n              } catch(e) {}\n            });\n          };\n          win.addEventListener(\'scroll\', sync, true);\n          doc.addEventListener(\'scroll\', sync, true);\n          sync();\n        })();\n        </script>\n        """,\n        height=0,\n        width=0,\n    )\n\n\ndef clipboard_button(text: str, key: str):\n    safe_text = json.dumps(str(text), ensure_ascii=False)\n    safe_key = re.sub(r"[^A-Za-z0-9_-]", "_", str(key))\n    components.html(\n        f"""\n        <button id="copy_{safe_key}" style="width:100%;height:36px;border-radius:10px;border:1px solid #5CA9E6;background:#0B1F3A;color:white;font-weight:700;cursor:pointer;">📋 کپی ID</button>\n        <script>\n        (() => {{\n          const b = document.getElementById(\'copy_{safe_key}\');\n          b.onclick = async () => {{\n            try {{\n              await navigator.clipboard.writeText({safe_text});\n              const old = b.innerText;\n              b.innerText = \'✅ کپی شد\';\n              setTimeout(() => b.innerText = old, 1200);\n            }} catch(e) {{\n              b.innerText = \'ID: \' + {safe_text};\n            }}\n          }};\n        }})();\n        </script>\n        """,\n        height=42,\n    )\n\n\nif "dark_mode" not in st.session_state:\n    st.session_state.dark_mode = False\nif "auction_campaigns" not in st.session_state:\n    st.session_state.auction_campaigns = []\nif "auction_enabled" not in st.session_state:\n    st.session_state.auction_enabled = True\n\ninject_theme(st.session_state.dark_mode)\ninstall_back_to_top()\n\nwith st.container(key="hero_panel"):\n    h1, h2 = st.columns([5.4, 1.1])\n    with h1:\n        st.markdown("## 🛍️ دستیار هوشمند خرید دیجی\u200cکالا")\n        st.markdown("جست\u200cوجو، Review QA، مقایسه، تحلیل مدیریتی، تبلیغات Auction و ارزیابی روی داده واقعی پروژه")\n    with h2:\n        new_dark = st.toggle("🌙 حالت تاریک", value=st.session_state.dark_mode, key="hero_theme_toggle")\n        if new_dark != st.session_state.dark_mode:\n            st.session_state.dark_mode = new_dark\n            st.rerun()\n\nif not RUN_DIR:\n    st.error("Artifactهای Run All پیدا نشد. ابتدا Notebook را کامل اجرا کنید.")\n    st.stop()\n\nproducts = load_products(str(RUN_DIR / "index" / "products.parquet"))\ncomments = load_comments(str(RUN_DIR / "comments_clean.parquet"))\nstats = build_product_stats(str(RUN_DIR / "comments_clean.parquet"))\nstats_idx = stats.set_index("product_id")\n\ntitle_col = get_col(products, "title_fa", "title_norm", "product_text_norm")\nbrand_col = get_col(products, "brand", "brand_norm")\ncat_col = get_col(products, "category1", "category1_norm")\nprice_col = get_col(products, "price_clean")\nrating_col = get_col(products, "product_rate_clean")\n\ntabs = st.tabs([\n    "✨ Try it", "⚖️ مقایسه", "📊 مدیر فروشگاه", "🧪 مدل و ارزیابی",\n    "📣 تبلیغات / Auction", "🧑\u200d⚖️ ارزیابی انسانی", "ℹ️ درباره",\n])\n\nwith tabs[0]:\n    st.subheader("✨ Try it — جست\u200cوجو و پرسش واقعی")\n    mode = st.radio("نوع تست", ["جست\u200cوجوی محصول", "پرسش درباره یک محصول"], horizontal=True)\n    if mode == "جست\u200cوجوی محصول":\n        c1, c2, c3 = st.columns([2.2, 1, 1])\n        query = c1.text_input("نیاز خود را فارسی بنویسید", value="یک اسباب بازی اقتصادی و باکیفیت پیشنهاد بده")\n        top_k = c2.number_input("تعداد نتیجه", min_value=3, max_value=12, value=6, step=1)\n        use_metis = c3.toggle("جمع\u200cبندی Metis", value=False)\n        fc1, fc2, fc3 = st.columns(3)\n        categories = ["همه"] + (sorted(products[cat_col].dropna().astype(str).unique().tolist())[:500] if cat_col else [])\n        selected_cat = fc1.selectbox("دسته", categories)\n        max_price = 0\n        if price_col and products[price_col].notna().any():\n            p95 = float(products[price_col].dropna().quantile(.95))\n            max_price = fc2.number_input("حداکثر قیمت (۰ = بدون محدودیت)", min_value=0,\n                                         max_value=max(int(p95*2), 1), value=0, step=10000)\n        min_rate = fc3.number_input("حداقل امتیاز", min_value=0.0, max_value=5.0, value=0.0, step=0.5)\n        if st.button("🔎 جست\u200cوجو", type="primary", use_container_width=True):\n            counts, dl, idf, vocab = load_bm25(str(RUN_DIR / "index"))\n            scores = bm25_scores(query, counts, dl, idf, vocab)\n            mask = np.ones(len(products), dtype=bool)\n            if selected_cat != "همه" and cat_col:\n                mask &= products[cat_col].astype(str).eq(selected_cat).to_numpy()\n            if max_price and price_col:\n                mask &= pd.to_numeric(products[price_col], errors="coerce").fillna(np.inf).to_numpy() <= max_price\n            if rating_col and min_rate > 0:\n                mask &= pd.to_numeric(products[rating_col], errors="coerce").fillna(0).to_numpy() >= min_rate\n            valid = np.flatnonzero(mask)\n            if not valid.size:\n                st.warning("با این فیلترها محصولی پیدا نشد.")\n            else:\n                organic_order = valid[np.argsort(-scores[valid], kind="stable")]\n                auction_rows = []\n                if st.session_state.auction_enabled and st.session_state.auction_campaigns:\n                    auction_rows = run_three_vendor_auction(products, stats_idx, st.session_state.auction_campaigns,\n                                                            scores=scores, eligible_mask=mask)\n                sponsored_ids = {int(x["product_id"]) for x in auction_rows}\n                organic_order = [int(pos) for pos in organic_order\n                                 if int(products.iloc[pos]["product_id"]) not in sponsored_ids]\n                sponsor_by_position = {x["placement_position"]: x for x in auction_rows}\n                displayed, organic_cursor = [], 0\n                for display_pos in range(1, int(top_k)+1):\n                    sponsor = sponsor_by_position.get(display_pos)\n                    if sponsor is not None:\n                        prow = products.iloc[sponsor["product_pos"]]\n                        pid = int(prow["product_id"])\n                        stat = stats_idx.loc[pid].to_dict() if pid in stats_idx.index else {}\n                        sponsored_product_card(prow, stat, sponsor)\n                        displayed.append({"kind": "sponsored", "row": prow, "stat": stat})\n                    elif organic_cursor < len(organic_order):\n                        pos = organic_order[organic_cursor]; organic_cursor += 1\n                        prow = products.iloc[pos]; pid = int(prow["product_id"])\n                        stat = stats_idx.loc[pid].to_dict() if pid in stats_idx.index else {}\n                        product_card(prow, stat)\n                        displayed.append({"kind": "organic", "row": prow, "stat": stat})\n                st.session_state["last_search_ids"] = [int(x["row"]["product_id"]) for x in displayed]\n                if use_metis and displayed:\n                    lines = []\n                    for item in displayed:\n                        row, stat = item["row"], item["stat"]\n                        kind = "SPONSORED" if item["kind"] == "sponsored" else "ORGANIC"\n                        lines.append(f\'- [{kind}] محصول {int(row["product_id"])}: {row.get(title_col, "")}; \'\n                                     f\'قیمت={row.get(price_col) if price_col else None}; \'\n                                     f\'نرخ توصیه نمونه={stat.get("recommendation_rate")}\')\n                    answer, usage, error = metis_answer(\n                        "نیاز کاربر:\\n" + query + "\\n\\nگزینه\u200cهای نمایش\u200cداده\u200cشده:\\n" + "\\n".join(lines)\n                        + "\\n\\nتبلیغ بودن را صریح ذکر کن و Sponsored بودن را هرگز دلیل برتری محصول ندان.")\n                    if error: st.error(error)\n                    else:\n                        st.markdown("### 🤖 جمع\u200cبندی Metis"); st.write(answer); st.caption(f"Usage: {usage}")\n    else:\n        last_ids = st.session_state.get("last_search_ids", [])\n        default_pid = int(last_ids[0]) if last_ids else int(products["product_id"].iloc[0])\n        c1, c2 = st.columns([1, 2])\n        pid = c1.number_input("Product ID", min_value=1, value=default_pid, step=1)\n        question = c2.text_input("سؤال از نظرات", value="مهم\u200cترین نقاط قوت و ضعف این محصول چیست؟")\n        use_metis = st.toggle("پاسخ نهایی با Metis", value=True, key="qa_metis")\n        if st.button("💬 پاسخ بده", type="primary"):\n            hits = retrieve_reviews(comments, pid, question, k=6)\n            if hits.empty:\n                st.warning("برای این محصول در Sample فعلی Review پیدا نشد.")\n            else:\n                lines = []\n                for _, row in hits.iterrows():\n                    cid, text = row.get("comment_id"), review_text(row)\n                    st.markdown(f\'<div class="card"><b>Review {cid}</b><br>{text}</div>\', unsafe_allow_html=True)\n                    lines.append(f"[Review {cid}] {text}")\n                if use_metis:\n                    answer, usage, error = metis_answer(\n                        f"سؤال: {question}\\nProduct ID: {pid}\\n\\nشواهد:\\n" + "\\n".join(lines)\n                        + "\\n\\nپاسخ باید Review IDهای پشتیبان را ذکر کند.")\n                    if error: st.error(error)\n                    else:\n                        st.markdown("### 🤖 پاسخ Grounded"); st.write(answer); st.caption(f"Usage: {usage}")\n\nwith tabs[1]:\n    st.subheader("⚖️ مقایسه چند محصول")\n    st.markdown(\'<div class="help-box">ابتدا در کاتالوگ جست\u200cوجو کنید تا Product IDهای مرتبط را ببینید، سپس آن\u200cها را به فهرست مقایسه اضافه کنید. امکان مقایسه هم\u200cزمان ۲ تا ۴ محصول فعال است.</div>\', unsafe_allow_html=True)\n    if "compare_ids" not in st.session_state:\n        st.session_state.compare_ids = []\n    search_ids = st.session_state.get("last_search_ids", [])\n    if not st.session_state.compare_ids:\n        st.session_state.compare_ids = list(dict.fromkeys(search_ids[:2] or products["product_id"].astype(int).head(2).tolist()))\n\n    finder_query = st.text_input("جست\u200cوجو در کاتالوگ برای یافتن Product ID", value="شامپو", key="compare_finder")\n    matches = search_catalog(products, finder_query, limit=8)\n    if matches.empty:\n        st.info("برای این جست\u200cوجو موردی پیدا نشد یا هنوز چیزی تایپ نشده است.")\n    else:\n        st.markdown("#### نتایج پیشنهادی")\n        for _, mrow in matches.iterrows():\n            pid = int(mrow["product_id"])\n            c1, c2, c3, c4 = st.columns([5.8, 1.2, 1.4, 1.4])\n            with c1:\n                st.markdown(\n                    f\'<div class="finder-row"><div class="card-title">{mrow["title"]}</div>\'\n                    f\'<div style="margin-top:8px"><span class="pill">Product ID: {pid}</span>\'\n                    f\'<span class="pill">دسته: {mrow["category"] or "—"}</span>\'\n                    f\'<span class="pill">برند: {mrow["brand"] or "—"}</span>\'\n                    f\'<span class="pill">قیمت: {fmt_money(mrow["price"] if not pd.isna(mrow["price"]) else None)}</span></div></div>\',\n                    unsafe_allow_html=True,\n                )\n            with c2:\n                st.code(str(pid), language=None)\n            with c3:\n                clipboard_button(str(pid), key=f"copy_{pid}")\n            with c4:\n                if st.button("افزودن", key=f"cmpadd_{pid}", use_container_width=True):\n                    if pid not in st.session_state.compare_ids:\n                        st.session_state.compare_ids.append(pid)\n                        st.rerun()\n\n    manual_ids = st.text_input("در صورت نیاز Product IDها را دستی با کاما وارد کنید", value="", key="manual_compare_ids")\n    mc1, mc2 = st.columns([1.5, 1])\n    if mc1.button("افزودن IDهای دستی", use_container_width=True):\n        parsed = []\n        for part in re.split(r"[,،\\s]+", manual_ids.strip()):\n            if part.isdigit():\n                parsed.append(int(part))\n        for pid in parsed:\n            if pid not in st.session_state.compare_ids:\n                st.session_state.compare_ids.append(pid)\n        st.rerun()\n    if mc2.button("پاک\u200cکردن فهرست مقایسه", use_container_width=True):\n        st.session_state.compare_ids = []\n        st.rerun()\n\n    compare_options = st.session_state.compare_ids or products["product_id"].astype(int).head(2).tolist()\n    selected_ids = st.multiselect(\n        "محصول\u200cهای انتخاب\u200cشده برای مقایسه",\n        options=compare_options,\n        default=compare_options[: min(4, len(compare_options))],\n        format_func=product_label,\n        key="compare_selected_ids",\n    )\n\n    if len(selected_ids) < 2:\n        st.warning("حداقل دو محصول انتخاب کنید.")\n    else:\n        if len(selected_ids) > 4:\n            st.info("برای خوانایی Dashboard فقط چهار محصول اول نمایش داده می\u200cشوند.")\n            selected_ids = selected_ids[:4]\n        selected = products[products["product_id"].astype(int).isin(selected_ids)].copy()\n        selected = selected.set_index(selected["product_id"].astype(int)).loc[selected_ids].reset_index(drop=True)\n        compare_table = []\n        for _, row in selected.iterrows():\n            pid = int(row["product_id"])\n            stat = stats_idx.loc[pid].to_dict() if pid in stats_idx.index else {}\n            compare_table.append({\n                "Product ID": pid,\n                "عنوان": row.get(title_col, "—"),\n                "دسته": row.get(cat_col, "—") if cat_col else "—",\n                "برند": row.get(brand_col, "—") if brand_col else "—",\n                "قیمت": fmt_money(row.get(price_col)) if price_col else "—",\n                "امتیاز محصول": row.get(rating_col, "—") if rating_col else "—",\n                "نرخ توصیه": f"{stat.get(\'recommendation_rate\', np.nan):.1f}%" if not pd.isna(stat.get(\'recommendation_rate\', np.nan)) else "—",\n                "تعداد نظر": int(stat.get("review_count", 0) or 0),\n            })\n        render_table(pd.DataFrame(compare_table))\n\n        cols = st.columns(len(selected_ids))\n        for col, (_, row) in zip(cols, selected.iterrows()):\n            with col:\n                pid = int(row["product_id"])\n                stat = stats_idx.loc[pid].to_dict() if pid in stats_idx.index else {}\n                product_card(row, stat)\n                pos = retrieve_reviews(comments, pid, "خوب عالی راضی کیفیت مناسب", 3)\n                neg = retrieve_reviews(comments, pid, "مشکل ایراد بد ضعیف خراب", 3)\n                st.markdown("**نمونه تجربه\u200cهای مثبت**")\n                if pos.empty:\n                    st.caption("نمونه مثبت کافی پیدا نشد.")\n                for _, rr in pos.iterrows():\n                    st.write(f\'• [{rr.get("comment_id")}] {review_text(rr)[:200]}\')\n                st.markdown("**نمونه نقاط ضعف / شکایت**")\n                neg = neg[neg.apply(lambda r: any(c in fa_norm(review_text(r)) for c in NEGATIVE_CUES), axis=1)]\n                if neg.empty:\n                    st.caption("شکایت صریح کافی در Sample پیدا نشد.")\n                for _, rr in neg.iterrows():\n                    st.write(f\'• [{rr.get("comment_id")}] {review_text(rr)[:200]}\')\n\nwith tabs[2]:\n    st.subheader("📊 داشبورد مدیر فروشگاه")\n    if not cat_col:\n        st.warning("ستون دسته\u200cبندی در artifact پیدا نشد.")\n    else:\n        cats = sorted(products[cat_col].dropna().astype(str).unique())\n        with st.expander("فیلترها و تنظیمات تحلیل دسته", expanded=True):\n            fc1, fc2 = st.columns([1.2, 1.8])\n            cat_hint = fc1.text_input("فیلتر نام دسته", value="", key="manager_cat_filter")\n            filtered_cats = [c for c in cats if cat_hint.strip() in c] if cat_hint.strip() else cats\n            selected_cat = fc2.selectbox("دسته مورد تحلیل", filtered_cats, key="manager_cat")\n        cat_products = products[products[cat_col].astype(str).eq(selected_cat)].copy()\n        cat_ids = set(cat_products["product_id"].astype(int))\n        cat_comments = comments[pd.to_numeric(comments["product_id"], errors="coerce").isin(cat_ids)].copy()\n        m1, m2, m3, m4 = st.columns(4)\n        m1.metric("محصول", f"{len(cat_products):,}")\n        m2.metric("Review نمونه", f"{len(cat_comments):,}")\n        if "recommendation_status" in cat_comments:\n            valid = cat_comments["recommendation_status"].isin(["recommended", "not_recommended", "no_idea"])\n            recommendation = ((cat_comments.loc[valid, "recommendation_status"] == "recommended").mean() * 100 if valid.any() else np.nan)\n        else:\n            recommendation = np.nan\n        m3.metric("نرخ توصیه", "—" if pd.isna(recommendation) else f"{recommendation:.1f}%")\n        avg_rate = pd.to_numeric(cat_comments.get("rate_clean"), errors="coerce").mean() if "rate_clean" in cat_comments else np.nan\n        m4.metric("میانگین امتیاز Review", "—" if pd.isna(avg_rate) else f"{avg_rate:.2f}")\n\n        st.markdown("#### شکایت\u200cهای پرتکرار")\n        counter = Counter()\n        for rtext in cat_comments.apply(review_text, axis=1):\n            n = fa_norm(rtext)\n            for cue in NEGATIVE_CUES:\n                if cue in n:\n                    counter[cue] += 1\n        complaints = pd.DataFrame(counter.most_common(12), columns=["عبارت", "تعداد"])\n        if not complaints.empty:\n            st.plotly_chart(px.bar(complaints, x="عبارت", y="تعداد", title="عبارت\u200cهای شکایت پرتکرار"), use_container_width=True)\n\n        risky = stats[stats["product_id"].isin(cat_ids)].copy()\n        if "recommendation_rate" in risky:\n            risky = (\n                risky[risky["labeled_reviews"].fillna(0) >= 3]\n                .sort_values(["recommendation_rate", "review_count"], ascending=[True, False])\n                .head(15)\n            )\n            risky = risky.merge(cat_products[["product_id", title_col]], on="product_id", how="left")\n            st.markdown("#### محصولات پرریسک")\n            risky_show = risky[["product_id", title_col, "review_count", "labeled_reviews", "recommendation_rate"]].copy()\n            risky_show.columns = ["Product ID", "عنوان", "تعداد نظر", "نظرات برچسب\u200cدار", "نرخ توصیه"]\n            risky_show["نرخ توصیه"] = risky_show["نرخ توصیه"].map(lambda x: f"{x:.1f}%" if pd.notna(x) else "—")\n            render_table(risky_show)\n\nwith tabs[3]:\n    st.subheader("🧪 کیفیت مدل و ارزیابی")\n    ready = read_json(str(RUN_DIR / "submission_readiness.json"))\n    c1, c2, c3, c4 = st.columns(4)\n    c1.metric("Macro-F1", ready.get("phase3_grouped_test_macro_f1", "—"))\n    c2.metric("Grounding / 5", ready.get("mean_proxy_grounding_0_5", "—"))\n    c3.metric("Task completion / 5", ready.get("mean_task_completion_proxy_0_5", "—"))\n    c4.metric("API cost est.", f\'${ready.get("api_estimated_list_cost_usd", 0):.6f}\')\n    retrieval = ready.get("retrieval_component_ablation", {}).get("by_method", {})\n    if retrieval:\n        rdf = pd.DataFrame([{"روش": k, **v} for k, v in retrieval.items()])\n        st.plotly_chart(\n            px.bar(rdf, x="روش", y=["recall@k", "mrr", "ndcg@k"], barmode="group", title="Dense vs BM25 vs Hybrid"),\n            use_container_width=True,\n        )\n    ab = ready.get("phase3_leakage_ablation", {})\n    if ab.get("available"):\n        adf = pd.DataFrame([\n            {"مدل": "Text only", "Macro-F1": ab.get("text_only_grouped_test_macro_f1")},\n            {"مدل": "Text + target-adjacent metadata", "Macro-F1": ab.get("text_plus_target_adjacent_metadata_macro_f1")},\n        ])\n        st.plotly_chart(px.bar(adf, x="مدل", y="Macro-F1", title="Leakage-risk ablation"), use_container_width=True)\n    api = ready.get("project_api_report", {})\n    st.markdown("#### API")\n    api_table = pd.DataFrame([\n        {"شاخص": "Provider", "مقدار": api.get("provider", "—")},\n        {"شاخص": "Model", "مقدار": api.get("model", "—")},\n        {"شاخص": "HTTP requests", "مقدار": api.get("total_http_requests", "—")},\n        {"شاخص": "Successful requests", "مقدار": api.get("successful_chat_requests", "—")},\n        {"شاخص": "Input tokens", "مقدار": api.get("input_tokens", "—")},\n        {"شاخص": "Output tokens", "مقدار": api.get("output_tokens", "—")},\n        {"شاخص": "Estimated cost (USD)", "مقدار": api.get("estimated_list_cost_usd", "—")},\n        {"شاخص": "Within $5 budget", "مقدار": "بله" if api.get("within_5_usd_budget") else "خیر"},\n    ])\n    render_table(api_table)\n    st.markdown(\'<div class="help-box">در این بخش به\u200cجای JSON خام، خلاصه خوانا و سازگار با تم روشن/تیره نمایش داده می\u200cشود تا اعداد و برچسب\u200cها در هر دو مود واضح بمانند.</div>\', unsafe_allow_html=True)\n\nwith tabs[4]:\n    st.subheader("📣 تبلیغات Sponsored / Auction")\n    st.markdown(\'\'\'<div class="help-box">هر Vendor یک Product ID و <b>Max CPC</b> ثبت می\u200cکند.\n    سیستم با Bid، کیفیت محصول و Query relevance، Ad Rank را می\u200cسازد. سه برنده در جایگاه\u200cهای\n    Sponsored شماره 1، 3 و 5 قرار می\u200cگیرند. Actual CPC با quality-adjusted GSP محاسبه می\u200cشود\n    و از Max CPC همان Vendor بیشتر نمی\u200cشود.</div>\'\'\',unsafe_allow_html=True)\n    default_ids=products["product_id"].astype(int).head(3).tolist()\n    saved=st.session_state.auction_campaigns\n    vendor_cols=st.columns(3); campaigns=[]\n    for idx,col in enumerate(vendor_cols):\n        existing=saved[idx] if idx<len(saved) else {}\n        with col:\n            st.markdown(f"#### Vendor {idx+1}")\n            name=st.text_input("نام Vendor",value=existing.get("vendor_name",f"Vendor {idx+1}"),key=f"vname_{idx}")\n            pid=st.number_input("Product ID تبلیغ",min_value=1,\n                                value=int(existing.get("product_id",default_ids[min(idx,len(default_ids)-1)])),\n                                step=1,key=f"vpid_{idx}")\n            bid=st.number_input("Max CPC (تومان)",min_value=0,max_value=1000000,\n                                value=int(existing.get("max_cpc",[8000,6500,5000][idx])),step=500,key=f"vbid_{idx}")\n            active=st.toggle("فعال",value=existing.get("active",True),key=f"vactive_{idx}")\n            campaigns.append({"vendor_name":name,"product_id":int(pid),"max_cpc":float(bid),"active":bool(active)})\n    a1,a2=st.columns(2)\n    if a1.button("💾 ذخیره تنظیمات Auction",type="primary",use_container_width=True):\n        st.session_state.auction_campaigns=campaigns; st.success("تنظیمات سه Vendor ذخیره شد.")\n    st.session_state.auction_enabled=a2.toggle("نمایش تبلیغات در Try it",value=st.session_state.auction_enabled,\n                                               key="auction_enable_toggle")\n    preview_query=st.text_input("Query آزمایشی",value="یک محصول باکیفیت و محبوب معرفی کن",key="auction_preview_query")\n    if st.button("⚡ اجرای Auction آزمایشی",use_container_width=True):\n        counts,dl,idf,vocab=load_bm25(str(RUN_DIR / "index"))\n        preview_scores=bm25_scores(preview_query,counts,dl,idf,vocab)\n        preview=run_three_vendor_auction(products,stats_idx,campaigns,scores=preview_scores,\n                                         eligible_mask=np.ones(len(products),dtype=bool))\n        if not preview: st.warning("کمپین معتبر/فعال پیدا نشد.")\n        else:\n            table = pd.DataFrame([\n                {"Slot": x["slot"], "Vendor": x["vendor_name"], "Product ID": x["product_id"],\n                 "Max CPC": round(x["max_cpc"], 0), "Quality": round(x["quality"], 3),\n                 "Query relevance": round(x["query_relevance"], 3), "Ad Rank": round(x["ad_rank"], 2),\n                 "Actual CPC": round(x["actual_cpc"], 0), "Placement": x["placement_position"]}\n                for x in preview\n            ])\n            render_table(table)\n            st.caption("Actual CPC هر برنده از Max CPC خودش بیشتر نمی\u200cشود.")\n\nwith tabs[5]:\n    st.subheader("🧑\u200d⚖️ Human Evaluation")\n    st.markdown(\'\'\'<div class="help-box">اینجا به <b>پاسخ سیستم</b> امتیاز می\u200cدهید، نه به Query.\n    Query، Answer و Evidence را بخوانید و سپس سه نمره 1 تا 5 ثبت کنید.</div>\'\'\',unsafe_allow_html=True)\n    per_query_path=RUN_DIR / "phase4_per_query.csv"\n    if not per_query_path.exists(): st.warning("phase4_per_query.csv پیدا نشد.")\n    else:\n        eval_df=pd.read_csv(per_query_path)\n        answer_col=next((c for c in ["answer_text","answer","response","text"] if c in eval_df.columns),None)\n        evidence_col=next((c for c in ["evidence_text","evidence","sources_text","context"] if c in eval_df.columns),None)\n        i=st.number_input("شماره نمونه",min_value=0,max_value=max(0,len(eval_df)-1),value=0,step=1)\n        row=eval_df.iloc[int(i)]\n        label_path=Path.cwd()/"human_eval_labels.csv"\n        labels=pd.read_csv(label_path) if label_path.exists() else pd.DataFrame()\n        current=None\n        if not labels.empty and "sample_index" in labels.columns:\n            matched=labels[labels["sample_index"]==int(i)]\n            if "run_id" in matched.columns: matched=matched[matched["run_id"].astype(str)==RUN_DIR.name]\n            if len(matched): current=matched.iloc[-1]\n        st.markdown("### Query"); st.markdown(f\'<div class="card">{str(row.get("query",""))}</div>\',unsafe_allow_html=True)\n        st.markdown("### Answer")\n        if answer_col: st.markdown(f\'<div class="card">{str(row.get(answer_col,""))}</div>\',unsafe_allow_html=True)\n        else: st.warning("ستون Answer پیدا نشد.")\n        st.markdown("### Evidence")\n        if evidence_col:\n            st.text_area("Evidence",value=str(row.get(evidence_col,"")),height=220,disabled=True,\n                         key=f"evidence_{int(i)}",label_visibility="collapsed")\n        else: st.info("ستون Evidence پیدا نشد.")\n        st.markdown(\'\'\'<div class="score-guide"><b>راهنمای نمره:</b>\n        1 = خیلی ضعیف · 2 = ضعیف · 3 = متوسط · 4 = خوب · 5 = خیلی خوب</div>\'\'\',unsafe_allow_html=True)\n        with st.form("human_eval_form"):\n            c1,c2,c3=st.columns(3)\n            rel=c1.number_input("Relevance / usefulness",1,5,int(current["human_relevance_1_5"]) if current is not None else 4,1)\n            grd=c2.number_input("Grounding / evidence support",1,5,int(current["human_grounding_1_5"]) if current is not None else 4,1)\n            comp=c3.number_input("Task completion",1,5,int(current["human_task_completion_1_5"]) if current is not None else 4,1)\n            note=st.text_area("یادداشت ارزیاب",value=(str(current["human_note"]) if current is not None and "human_note" in current and not pd.isna(current["human_note"]) else ""),height=110)\n            submitted=st.form_submit_button("💾 ذخیره امتیاز",use_container_width=True)\n        if submitted:\n            answer_text=str(row.get(answer_col,"")) if answer_col else ""\n            answer_hash=hashlib.sha256(answer_text.encode("utf-8")).hexdigest()[:16]\n            rec=pd.DataFrame([{"sample_index":int(i),"query":row.get("query",""),"answer_hash":answer_hash,\n                               "run_id":RUN_DIR.name,"human_relevance_1_5":int(rel),"human_grounding_1_5":int(grd),\n                               "human_task_completion_1_5":int(comp),"human_note":note,"timestamp":pd.Timestamp.now().isoformat()}])\n            if label_path.exists():\n                old=pd.read_csv(label_path)\n                if "sample_index" in old.columns and "run_id" in old.columns:\n                    old=old[~((old["sample_index"]==int(i))&(old["run_id"].astype(str)==RUN_DIR.name))]\n                rec=pd.concat([old,rec],ignore_index=True)\n            rec.to_csv(label_path,index=False,encoding="utf-8-sig"); st.success("امتیاز این پاسخ ذخیره شد.")\n        if label_path.exists():\n            labels=pd.read_csv(label_path)\n            current_run=labels[labels["run_id"].astype(str)==RUN_DIR.name] if "run_id" in labels.columns else labels.iloc[0:0]\n            st.progress(min(1.0,len(current_run)/max(len(eval_df),1)))\n            st.caption(f"{len(current_run)} از {len(eval_df)} پاسخ این Run امتیازدهی شده")\n            st.download_button("دانلود human_eval_labels.csv",data=label_path.read_bytes(),file_name="human_eval_labels.csv",mime="text/csv")\n\nwith tabs[6]:\n    st.subheader("ℹ️ درباره پروژه")\n    st.markdown(\'\'\'این Dashboard لایه Demo و محصول پروژه است و Artifactهای آخرین Run را می\u200cخواند.\n\n**قابلیت\u200cها:** Product Search، Review QA، Comparison، Manager Analytics، Model Evaluation،\nSponsored Search Auction برای سه Vendor، Human Evaluation و Dark/Light mode.\n\nتبلیغات با برچسب **Sponsored / تبلیغ** از نتایج Organic جدا می\u200cشوند و Sponsored بودن هرگز\nبه\u200cعنوان نشانه بهتر بودن محصول به مدل داده نمی\u200cشود.\'\'\')\n'
STREAMLIT_APP_PATH.write_text(STREAMLIT_APP_SOURCE, encoding="utf-8")
print("Streamlit app generated:", STREAMLIT_APP_PATH)

streamlit_launch = {"launched": False}

if AUTO_LAUNCH_STREAMLIT:
    try:
        def free_port(start=8501, end=8520):
            for port in range(start, end+1):
                sock = socket.socket()
                try:
                    sock.bind(("127.0.0.1", port))
                    return port
                except OSError:
                    pass
                finally:
                    sock.close()
            raise RuntimeError("No free Streamlit port found")

        port = free_port()
        log_path = Path.cwd() / "streamlit_dashboard.log"
        log_handle = open(log_path, "w", encoding="utf-8")

        proc = subprocess.Popen(
            [
                sys.executable, "-m", "streamlit", "run",
                str(STREAMLIT_APP_PATH),
                "--server.port", str(port),
                "--server.headless", "true",
                "--browser.gatherUsageStats", "false",
            ],
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            cwd=str(Path.cwd()),
        )

        url = f"http://127.0.0.1:{port}"
        time.sleep(3)
        streamlit_launch = {
            "launched": proc.poll() is None,
            "pid": proc.pid,
            "url": url,
            "log": str(log_path),
        }
        print(json.dumps(streamlit_launch, ensure_ascii=False, indent=2))

        if streamlit_launch["launched"]:
            try:
                webbrowser.open(url)
            except Exception:
                pass

            try:
                from IPython.display import display, HTML, IFrame
                display(HTML(
                    f'<a href="{url}" target="_blank">Open Streamlit dashboard</a>'
                ))
                display(IFrame(url, width="100%", height=760))
            except Exception:
                pass

    except Exception as exc:
        streamlit_launch = {
            "launched": False,
            "error": repr(exc),
        }
        print("Streamlit auto-launch failed:", repr(exc))

bonus_status["streamlit"] = streamlit_launch


### 5.8 Bonus audit

The final bonus audit distinguishes:
- implemented;
- executed;
- measured improvement;
- claimable.

This prevents unsuccessful optional experiments from being presented as
successful improvements.


In [ ]:
# Final bonus audit: implemented != measured != claimable.
def _claim(x):
    return bool(isinstance(x, dict) and x.get("bonus_claim_supported", False))

retrieval_ablation = readiness.get("retrieval_component_ablation", {}) if "readiness" in globals() else {}
hybrid_lift = retrieval_ablation.get("hybrid_minus_best_single_method_ndcg")
if hybrid_lift is None:
    hybrid_lift = retrieval_ablation.get("hybrid_minus_best_single_ndcg")

bonus_audit = {
    "llm_as_judge": {
        "implemented": True,
        "executed": judge_summary.get("mean_judge_relevance_0_5") is not None,
        "claimable": judge_summary.get("mean_judge_relevance_0_5") is not None,
        "note": "Advanced evaluation method; interpret with the documented single-judge limitations.",
    },
    "human_validation": {
        "implemented": True,
        "executed": bool(human_validation_summary.get("available")),
        "n_matching": int(human_validation_summary.get("n_labels_matching_current_run", 0) or 0),
        "claimable": bool(
            human_validation_summary.get("available")
            and int(human_validation_summary.get("n_labels_matching_current_run", 0) or 0) >= 3
        ),
        "note": "Single-rater validation; no inter-rater agreement claim.",
    },
    "cache_optimization": {
        "implemented": True,
        "executed": bool(cache_bonus.get("available")),
        "measured_speedup": cache_bonus.get("cache_speedup"),
        "claimable": _claim(cache_bonus),
    },
    "quantization": {
        "implemented": True,
        "executed": bool(quant_bonus.get("available")),
        "measured_speedup": quant_bonus.get("speedup"),
        "claimable": _claim(quant_bonus),
        "note": quant_bonus.get("reason") or quant_bonus.get("error"),
    },
    "lora_finetuning": {
        "implemented": True,
        "executed": bool(lora_bonus.get("available")),
        "claimable": _claim(lora_bonus),
        "note": lora_bonus.get("reason") or lora_bonus.get("error"),
    },
    "sponsored_search_auction": {
        "implemented": True,
        "executed": bool(auction_bonus.get("available")),
        "technical_result_supported": bool(auction_bonus.get("technical_result_supported", False)),
        "mentor_approved": bool(auction_bonus.get("mentor_approved_new_problem", False)),
        "claimable": _claim(auction_bonus),
        "note": "Do not claim the mentor-approved new-problem bonus until explicit mentor confirmation.",
    },
    "streamlit_dashboard": {
        "implemented": True,
        "executed": bool(streamlit_launch.get("launched")),
        "claimable": bool(streamlit_launch.get("launched")),
        "url": streamlit_launch.get("url"),
    },
    "hybrid_retrieval_quality_bonus": {
        "implemented": True,
        "executed": True,
        "measured_ndcg_lift": hybrid_lift,
        "claimable": bool(hybrid_lift is not None and float(hybrid_lift) > 0),
        "note": "Current benchmark must show a positive quality lift; latency alone is not relabeled as quality improvement.",
    },
}

BONUS_OUTPUT_DIR.mkdir(exist_ok=True)
(BONUS_OUTPUT_DIR / "bonus_audit.json").write_text(
    json.dumps(bonus_audit, ensure_ascii=False, indent=2), encoding="utf-8"
)

lines = [
    "# Bonus Audit",
    "",
    "Only measured and supported items are claimable. Optional failures do not invalidate the mandatory project.",
    "",
]
for name, info in bonus_audit.items():
    lines.append(f"## {name}")
    for k, v in info.items():
        lines.append(f"- {k}: {v}")
    lines.append("")
bonus_report_text = "\n".join(lines)
(BONUS_OUTPUT_DIR / "BONUS_REPORT.md").write_text(bonus_report_text, encoding="utf-8")

# Append/update a clearly separated bonus section in FINAL_REPORT.md.
final_report_path = Path.cwd() / "FINAL_REPORT.md"
if final_report_path.exists():
    base_report = final_report_path.read_text(encoding="utf-8")
    marker = "\n\n# Bonus Audit\n"
    if marker in base_report:
        base_report = base_report.split(marker, 1)[0].rstrip()
    final_report_path.write_text(base_report + "\n\n" + bonus_report_text + "\n", encoding="utf-8")

print(json.dumps(bonus_audit, ensure_ascii=False, indent=2))


## 6 · Final submission notes

### Official hosted API configuration

The final run uses the course-provided **Metis** OpenAI-compatible endpoint:

- Base URL: `https://api.metisai.ir/openai/v1`
- Environment variable: `METIS_API_KEY`
- Model: `gpt-4o-mini`
- Course credit: **$5, non-rechargeable**
- Notebook operational safety stop: **$4.50**
- Maximum hosted attempts in one Run All: **40** (main generation + LLM-as-a-Judge; the dollar safety stop remains authoritative)

The secret key is never written into notebook outputs or generated files.

### Evaluation interpretation

The final notebook reports multiple evaluation layers instead of mixing them:

- deterministic response/grounding proxies;
- retrieval metrics and component ablations;
- Metis LLM-as-a-Judge relevance/faithfulness scores, with explicit limitations;
- independent human validation when `human_eval_labels.csv` matches the held-out queries;
- latency and API request/token/cost accounting;
- retained failure cases.

The human file is a **single-rater** validation and therefore is not presented as inter-rater agreement.

### Bonus-claim policy

Optional experiments are only claimed when the executed measurements support them. In particular:

- Hybrid retrieval is **not** called a quality improvement when nDCG is below the best single retriever.
- Quantization and LoRA are claimed only if their own output flags report a measured supported gain.
- Sponsored Search Auction can run as an experimental extension, but the **mentor-approved new-problem bonus remains pending** unless `MENTOR_APPROVED_AUCTION=1` is set after explicit mentor confirmation.
- Streamlit is a presentation/product layer and does not replace notebook scientific evaluation.

### Generated final files

After **Restart Kernel → Run All**, the notebook writes the core evidence under `nb_artifacts/run_*` plus:

- `submission_readiness.json`
- `project_api_report.json`
- `phase4_metrics.json`
- `bonus_outputs/bonus_audit.json`
- `bonus_outputs/BONUS_REPORT.md`
- `bonus_outputs/human_eval_validation.json` when human labels match
- `FINAL_REPORT.md` next to the notebook
- `streamlit_app.py` and `streamlit_dashboard.log` for the presentation UI

For the intended final run, every mandatory `submission_check` must be `true` and `submission_checks_passed` must be `true`. Optional bonus failures do **not** invalidate the mandatory project.
